# CÓDIGO PARA EXPANSÃO DE ÁREA GEOGRÁFICA DOS MAPAS

In [1]:
import pandas as pd
import re

def corrigir_encoding_mapas(texto):
    """
    Corrige problemas de encoding específicos da planilha de mapas.
    """
    if pd.isna(texto) or str(texto).strip().upper() == 'NULL' or str(texto).strip() == '':
        return texto
    
    texto_str = str(texto)
    
    # Mapeamento de correções (pode ser ampliado se necessário)
    correcoes = {
        'SÃ£o': 'São',
        'JosÃ©': 'José',
        'TamandarÃ©': 'Tamandaré',
        'AraucÃ¡ria': 'Araucária',
        'BocaiÃºva': 'Bocaiúva',
        'ItaperuÃ§u': 'Itaperuçu',
        'MaringÃ¡': 'Maringá',
        'BelÃ©m': 'Belém',
        'GoiÃ¢nia': 'Goiânia',
        'tamandare': 'Tamandaré',
        'sul': 'Sul',
        'Piaraquara': 'Piraquara',
        'Piraqura': 'Piraquara',
        'ItaperuÃ§Ãº': 'Itaperuçu',
        'Almitante': 'Almirante',
        'ParanÃ¡': 'Paraná',
    }
    
    for errado, correto in correcoes.items():
        texto_str = texto_str.replace(errado, correto)
    
    return texto_str

def processar_area_geografica_mapas(arquivo_excel, coluna='area_geografica_mapas'):
    """
    Processa a planilha de área geográfica para mapas.
    
    Args:
        arquivo_excel: Caminho do arquivo Excel
        coluna: Nome da coluna a ser processada
    
    Returns:
        DataFrame com os dados processados
    """
    
    # Carregar o arquivo
    df = pd.read_excel(arquivo_excel)
    
    # Lista para armazenar resultados
    dados_processados = []
    
    for idx, valor in enumerate(df[coluna], start=1):
        linha_excel = idx + 1  # +1 porque Excel começa em 1 e tem cabeçalho
        
        # CASO 1: Valor NULL ou vazio
        if pd.isna(valor) or str(valor).strip().upper() == 'NULL' or str(valor).strip() == '':
            dados_processados.append({
                'linha_original': linha_excel,
                'municipio': '',
                'codigo': '',
                'valor_original': '',
                'tipo': 'vazio',
                'status': 'NULL'
            })
            continue
        
        # Corrigir problemas de encoding
        valor_corrigido = corrigir_encoding_mapas(valor)
        valor_str = str(valor_corrigido).strip()
        
        # CASO ESPECIAL: "cedoc" (se aparecer em mapas, manter; caso contrário, remover)
        if valor_str.lower() == 'cedoc':
            dados_processados.append({
                'linha_original': linha_excel,
                'municipio': 'CEDOC',
                'codigo': '',
                'valor_original': valor_str,
                'tipo': 'especial',
                'status': 'cedoc'
            })
            continue
        
        # CASO ESPECIAL: "Porto Alegre" (se aparecer em mapas, manter; caso contrário, remover)
        if valor_str == 'Porto Alegre':
            dados_processados.append({
                'linha_original': linha_excel,
                'municipio': 'Porto Alegre',
                'codigo': '',
                'valor_original': valor_str,
                'tipo': 'único',
                'status': 'sem código'
            })
            continue
        
        # CASO ESPECIAL: Estado "Paraná - 41" (ou outros estados)
        # Códigos de estado geralmente têm 2 dígitos. Vamos tratá‑los como "estado" em vez de município.
        padrao_estado = re.search(r'^([A-Za-zÀ-ÿ\s]+) - (\d{2})$', valor_str)
        if padrao_estado:
            estado = padrao_estado.group(1).strip()
            codigo_estado = padrao_estado.group(2).strip()
            dados_processados.append({
                'linha_original': linha_excel,
                'municipio': estado,          # Aqui seria o nome do estado, mas mantemos como "municipio" por simplicidade
                'codigo': codigo_estado,
                'valor_original': valor_str,
                'tipo': 'estado',
                'status': 'código de estado'
            })
            continue
        
        # CASO ESPECIAL: Município com código grudado no final (ex: "SÃ£o JosÃ© dos Pinhais - 2550622206")
        padrao_codigo_grudado = re.search(r'(.+?) - (\d{5})(\d{5})$', valor_str)
        if padrao_codigo_grudado:
            municipio = padrao_codigo_grudado.group(1).strip()
            codigo1 = padrao_codigo_grudado.group(2).strip()
            codigo2 = padrao_codigo_grudado.group(3).strip()
            
            # Primeiro município
            dados_processados.append({
                'linha_original': linha_excel,
                'municipio': municipio,
                'codigo': codigo1,
                'valor_original': valor_str,
                'tipo': 'múltiplo',
                'status': 'ok'
            })
            
            # Segundo município (presumindo que é Rio Branco do Sul baseado no código 22206 – ajuste se necessário)
            dados_processados.append({
                'linha_original': linha_excel,
                'municipio': 'Rio Branco do Sul',
                'codigo': codigo2,
                'valor_original': valor_str,
                'tipo': 'múltiplo',
                'status': 'código grudado'
            })
            continue
        
        # Regex principal para capturar padrões
        # Detecta: "Município - Código" (com ou sem espaço depois do hífen)
        padrao_principal = re.compile(r'''
            ([A-Za-zÀ-ÿ\s\.\-]+?)      # Nome do município (pode ter hífen interno)
            \s*                        # Espaços opcionais
            (?:-\s*)                   # Hífen com espaços opcionais
            (\d{4,6})                  # Código (4-6 dígitos)
            (?=\s|$|[A-Z]|,|;)         # Lookahead: espaço, fim, letra maiúscula, vírgula ou ponto-vírgula
        ''', re.VERBOSE | re.IGNORECASE)
        
        # Encontrar todos os matches
        matches = list(padrao_principal.finditer(valor_str))
        
        # CASO 2: Padrão sem espaço entre municípios (mais comum)
        if not matches:
            padrao_colado = re.compile(r'''
                ([A-Za-zÀ-ÿ\s\.\-]+?)  # Nome do município
                \s*-\s*                # Hífen com espaços
                (\d{4,6})              # Código
                (?=[A-ZÀ-ÿ])           # Lookahead: próxima letra (início do próximo município)
            ''', re.VERBOSE | re.IGNORECASE)
            
            matches = list(padrao_colado.finditer(valor_str))
        
        # CASO 3: Padrão com múltiplos municípios separados por espaço
        if not matches:
            padrao_sequencia = re.compile(r'''
                ([A-Za-zÀ-ÿ\s\.\-]+?)  # Nome do município
                \s*-\s*                # Hífen com espaços
                (\d{4,6})              # Código
                (?:\s+|$)              # Espaço ou fim
            ''', re.VERBOSE | re.IGNORECASE)
            
            matches = list(padrao_sequencia.finditer(valor_str))
        
        # CASO 4: Padrão com código de outro estado (ex: São Paulo - 50308)
        if not matches:
            padrao_outros_estados = re.compile(r'''
                ([A-Za-zÀ-ÿ\s\.\-]+?)  # Nome do município
                \s*-\s*                # Hífen com espaços
                (\d{5,6})              # Código (5-6 dígitos para outros estados)
            ''', re.VERBOSE | re.IGNORECASE)
            
            matches = list(padrao_outros_estados.finditer(valor_str))
        
        # CASO 5: Nenhum padrão encontrado
        if not matches:
            # Tentar identificar se é apenas um nome de município
            if re.match(r'^[A-Za-zÀ-ÿ\s\-]+$', valor_str):
                dados_processados.append({
                    'linha_original': linha_excel,
                    'municipio': valor_str,
                    'codigo': '',
                    'valor_original': valor_str,
                    'tipo': 'único',
                    'status': 'sem código'
                })
            else:
                dados_processados.append({
                    'linha_original': linha_excel,
                    'municipio': valor_str,
                    'codigo': '',
                    'valor_original': valor_str,
                    'tipo': 'único',
                    'status': 'formato irregular'
                })
            continue
        
        # Processar os matches encontrados
        for match in matches:
            municipio = match.group(1).strip()
            codigo = match.group(2).strip()
            
            # Limpar espaços extras e caracteres estranhos no nome do município
            municipio = re.sub(r'\s+', ' ', municipio)  # Espaços múltiplos para simples
            municipio = municipio.strip()
            
            # Remover hífens soltos no início
            municipio = re.sub(r'^\s*-\s*', '', municipio)
            
            # Capitalização básica (primeira letra de cada palavra maiúscula)
            palavras = municipio.split()
            municipio = ' '.join([p.capitalize() if len(p) > 2 else p for p in palavras])
            
            # Normalização especial para municípios conhecidos
            normalizacoes = {
                'Almirante Tamandare': 'Almirante Tamandaré',
                'Almirante tamandare': 'Almirante Tamandaré',
                'Sao Jose': 'São José',
                'Sao Jose dos Pinhais': 'São José dos Pinhais',
                'Itaperucu': 'Itaperuçu',
                'Piaraquara': 'Piraquara',
                'Piraqura': 'Piraquara',
                'Araucaria': 'Araucária',
            }
            
            if municipio in normalizacoes:
                municipio = normalizacoes[municipio]
            
            # Normalizar código
            if codigo.isdigit():
                if len(codigo) < 5:
                    codigo = codigo.zfill(5)
                    status = 'normalizado'
                elif len(codigo) == 6:
                    # Códigos de 6 dígitos podem ser de outros estados
                    status = 'código 6 dígitos (outro estado?)'
                else:
                    status = 'ok'
            else:
                status = 'código inválido'
            
            # Determinar tipo
            tipo = 'múltiplo' if len(matches) > 1 else 'único'
            
            dados_processados.append({
                'linha_original': linha_excel,
                'municipio': municipio,
                'codigo': codigo,
                'valor_original': valor_str,
                'tipo': tipo,
                'status': status
            })
    
    # Criar DataFrame
    df_processado = pd.DataFrame(dados_processados)
    
    return df_processado

def exportar_relatorio_mapas(df_processado, nome_arquivo='AREA_GEOGRAFICA_MAPAS_PROCESSADA.xlsx'):
    """
    Exporta os resultados para Excel com relatório completo.
    """
    with pd.ExcelWriter(nome_arquivo, engine='openpyxl') as writer:
        # 1. Dados completos
        df_processado.to_excel(writer, sheet_name='Dados Completos', index=False)
        
        # 2. Estatísticas gerais
        stats = {
            'Total linhas originais': df_processado['linha_original'].nunique(),
            'Total registros processados': len(df_processado),
            'Registros vazios/NULL': len(df_processado[df_processado['tipo'] == 'vazio']),
            'Registros únicos': len(df_processado[df_processado['tipo'] == 'único']),
            'Registros múltiplos': len(df_processado[df_processado['tipo'] == 'múltiplo']),
            'Registros especiais': len(df_processado[df_processado['tipo'] == 'especial']),
            'Registros de estado': len(df_processado[df_processado['tipo'] == 'estado']),
            'Municípios únicos identificados': df_processado['municipio'].nunique() - 1,  # -1 para vazio
            'Registros sem código': len(df_processado[df_processado['status'] == 'sem código']),
            'Registros com formato irregular': len(df_processado[df_processado['status'] == 'formato irregular']),
        }
        
        stats_df = pd.DataFrame(list(stats.items()), columns=['Métrica', 'Valor'])
        stats_df.to_excel(writer, sheet_name='Estatísticas', index=False)
        
        # 3. Lista de municípios únicos
        municipios_unicos = df_processado[['municipio', 'codigo']].drop_duplicates()
        municipios_unicos = municipios_unicos[municipios_unicos['municipio'] != '']
        municipios_unicos = municipios_unicos.sort_values('municipio')
        municipios_unicos.to_excel(writer, sheet_name='Municípios Únicos', index=False)
        
        # 4. Distribuição por município (top 20)
        top_municipios = df_processado['municipio'].value_counts().head(20)
        top_municipios_df = pd.DataFrame({
            'Município': top_municipios.index,
            'Quantidade': top_municipios.values
        })
        top_municipios_df.to_excel(writer, sheet_name='Top Municípios', index=False)
        
        # 5. Distribuição por status
        status_counts = df_processado['status'].value_counts()
        status_df = pd.DataFrame({
            'Status': status_counts.index,
            'Quantidade': status_counts.values
        })
        status_df.to_excel(writer, sheet_name='Distribuição Status', index=False)
        
        # 6. Análise de padrões complexos
        padroes_complexos = df_processado[df_processado['tipo'] == 'múltiplo']
        if not padroes_complexos.empty:
            combinacoes = padroes_complexos.groupby('valor_original').agg({
                'municipio': lambda x: ', '.join(x.unique()),
                'linha_original': 'first'
            }).reset_index()
            
            combinacoes['qtd_municipios'] = combinacoes['municipio'].apply(lambda x: len(x.split(', ')))
            combinacoes = combinacoes.sort_values('qtd_municipios', ascending=False)
            
            combinacoes[['valor_original', 'municipio', 'qtd_municipios']].to_excel(
                writer, sheet_name='Combinações Complexas', index=False
            )
        
        # 7. Casos problemáticos
        problemas = df_processado[~df_processado['status'].isin(['ok', 'normalizado'])]
        if not problemas.empty:
            problemas.to_excel(writer, sheet_name='Casos Problemáticos', index=False)
        
        # 8. Municípios de outros estados
        outros_estados = df_processado[df_processado['status'].str.contains('outro estado', na=False)]
        if not outros_estados.empty:
            outros_estados[['municipio', 'codigo', 'valor_original']].drop_duplicates().to_excel(
                writer, sheet_name='Outros Estados', index=False
            )
    
    print(f"✅ Arquivo exportado: {nome_arquivo}")
    return True

def analise_detalhada_mapas(df_processado):
    """
    Realiza análise detalhada dos dados processados.
    """
    print("\n" + "="*60)
    print("ANÁLISE DETALHADA DA PLANILHA DE MAPAS")
    print("="*60)
    
    # Estatísticas básicas
    total_linhas = df_processado['linha_original'].nunique()
    total_registros = len(df_processado)
    vazios = len(df_processado[df_processado['tipo'] == 'vazio'])
    unicos = len(df_processado[df_processado['tipo'] == 'único'])
    multiplos = len(df_processado[df_processado['tipo'] == 'múltiplo'])
    especiais = len(df_processado[df_processado['tipo'] == 'especial'])
    estados = len(df_processado[df_processado['tipo'] == 'estado'])
    municipios_count = df_processado['municipio'].nunique() - 1  # -1 para vazio
    
    print(f"\n📊 ESTATÍSTICAS GERAIS:")
    print(f"  • Linhas originais: {total_linhas}")
    print(f"  • Registros processados: {total_registros}")
    print(f"  • Registros vazios/NULL: {vazios} ({vazios/total_linhas*100:.1f}%)")
    print(f"  • Registros únicos: {unicos}")
    print(f"  • Registros múltiplos: {multiplos}")
    print(f"  • Registros especiais: {especiais}")
    print(f"  • Registros de estado: {estados}")
    print(f"  • Municípios únicos identificados: {municipios_count}")
    
    # Top 10 municípios
    print(f"\n🏙️ TOP 10 MUNICÍPIOS MAIS FREQUENTES:")
    # Ignorar vazio e estados para o top de municípios
    top_df = df_processado[~df_processado['municipio'].isin(['', 'Paraná'])]  # ajuste conforme necessário
    top_10 = top_df['municipio'].value_counts().head(10)
    for i, (municipio, count) in enumerate(top_10.items(), 1):
        print(f"  {i:2d}. {municipio:<30} ({count} ocorrências)")
    
    # Análise de status
    print(f"\n🔍 DISTRIBUIÇÃO POR STATUS:")
    status_counts = df_processado['status'].value_counts().head(10)
    for status, count in status_counts.items():
        print(f"  • {status:<30}: {count}")
    
    # Casos complexos
    casos_complexos = df_processado[df_processado['tipo'] == 'múltiplo']
    if not casos_complexos.empty:
        print(f"\n🧩 COMBINAÇÕES MAIS COMPLEXAS:")
        grupos = casos_complexos.groupby('valor_original').agg({
            'municipio': lambda x: list(x.unique()),
            'linha_original': 'first'
        })
        
        grupos['qtd'] = grupos['municipio'].apply(len)
        top_complexos = grupos.sort_values('qtd', ascending=False).head(5)
        
        for idx, (valor_original, dados) in enumerate(top_complexos.iterrows(), 1):
            municipios = ', '.join(dados['municipio'][:5])
            if len(dados['municipio']) > 5:
                municipios += f"... (+{len(dados['municipio']) - 5} mais)"
            print(f"  {idx}. {municipios}")
            print(f"     Linha: {dados['linha_original']}, Total: {dados['qtd']} municípios")
    
    # Municípios de outros estados
    outros_estados = df_processado[df_processado['codigo'].astype(str).str.len() == 6]
    if not outros_estados.empty:
        print(f"\n🌎 MUNICÍPIOS DE OUTROS ESTADOS:")
        municipios_outros = outros_estados[['municipio', 'codigo']].drop_duplicates()
        for _, row in municipios_outros.iterrows():
            print(f"  • {row['municipio']:<25} - Código: {row['codigo']}")

# USO PRINCIPAL
if __name__ == "__main__":
    print("🔧 PROCESSANDO PLANILHA DE MAPAS...")
    
    try:
        # Processar a planilha
        df_processado = processar_area_geografica_mapas(
            'AREA_GEOGRAFICA_MAPAS.xlsx', 
            coluna='area_geografica_mapas'
        )
        
        # Análise detalhada
        analise_detalhada_mapas(df_processado)
        
        # Exportar resultados
        exportado = exportar_relatorio_mapas(df_processado)
        
        if exportado:
            print("\n" + "="*60)
            print("✅ PROCESSAMENTO CONCLUÍDO COM SUCESSO!")
            print("="*60)
            print("\n📁 O arquivo 'AREA_GEOGRAFICA_MAPAS_PROCESSADA.xlsx' foi gerado com:")
            print("   1. Dados Completos")
            print("   2. Estatísticas")
            print("   3. Municípios Únicos")
            print("   4. Top Municípios")
            print("   5. Distribuição por Status")
            print("   6. Combinações Complexas")
            print("   7. Casos Problemáticos (se houver)")
            print("   8. Outros Estados (se houver)")
            
    except FileNotFoundError:
        print("❌ ERRO: Arquivo 'AREA_GEOGRAFICA_MAPAS.xlsx' não encontrado.")
        print("   Certifique-se de que o arquivo está na mesma pasta do script.")
    except Exception as e:
        print(f"❌ ERRO: {e}")
        print("   Verifique se o arquivo tem a estrutura correta.")

🔧 PROCESSANDO PLANILHA DE MAPAS...

ANÁLISE DETALHADA DA PLANILHA DE MAPAS

📊 ESTATÍSTICAS GERAIS:
  • Linhas originais: 4470
  • Registros processados: 12860
  • Registros vazios/NULL: 123 (2.8%)
  • Registros únicos: 2259
  • Registros múltiplos: 10471
  • Registros especiais: 0
  • Registros de estado: 7
  • Municípios únicos identificados: 50

🏙️ TOP 10 MUNICÍPIOS MAIS FREQUENTES:
   1. São José Dos Pinhais           (1180 ocorrências)
   2. Curitiba                       (1068 ocorrências)
   3. Campo Largo                    (1030 ocorrências)
   4. Almirante Tamandaré            (800 ocorrências)
   5. Piraquara                      (781 ocorrências)
   6. Araucária                      (756 ocorrências)
   7. Colombo                        (725 ocorrências)
   8. Bocaiúva do Sul                (688 ocorrências)
   9. Campina Grande do Sul          (682 ocorrências)
  10. Rio Branco do Sul              (665 ocorrências)

🔍 DISTRIBUIÇÃO POR STATUS:
  • ok                         

# CÓDIGO PARA CORREÇÃO DE CARATERES CORROMPIDOS EM TITULO_MAPAS

In [2]:
# CORREÇÃO DE CARACTERES - TITULO_MAPAS.xlsx
# Versão adaptada do código TITULO_LIVROS

import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ========== 1. CONFIGURAÇÃO ==========
print("🔧 CONFIGURANDO AMBIENTE PARA TITULO_MAPAS.xlsx...")
input_file = "TITULO_MAPAS.xlsx"
output_file = "TITULO_MAPAS_CORRIGIDO.xlsx"

# ========== 2. CARREGAR O ARQUIVO ==========
print(f"\n📂 CARREGANDO ARQUIVO: {input_file}")
try:
    df = pd.read_excel(input_file)
    print(f"✅ Arquivo carregado com sucesso!")
    print(f"📊 Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas")
    print(f"📋 Primeiras 3 linhas antes da correção:")
    for i in range(min(3, len(df))):
        print(f"  {i+1}. {df.iloc[i, 0] if df.shape[1] > 0 else ''}")
except Exception as e:
    print(f"❌ Erro ao carregar: {e}")
    # Tentar com engine específica
    try:
        df = pd.read_excel(input_file, engine='openpyxl')
        print("✅ Carregado com engine 'openpyxl'")
    except Exception as e2:
        print(f"❌ Erro com openpyxl: {e2}")
        print("⚠️  Tentando carregar apenas a primeira coluna...")
        df = pd.read_excel(input_file, usecols=[0])

# ========== 3. FUNÇÃO DE CORREÇÃO ESPECÍFICA PARA TITULO_MAPAS ==========
def corrigir_caracteres_titulos_mapas(texto):
    """
    Função otimizada para correção dos caracteres corrompidos no TITULO_MAPAS
    Baseado nos padrões identificados na planilha
    """
    if pd.isna(texto):
        return ""
    
    texto = str(texto)
    
    # PRIMEIRO: tentar correção via encoding (latin-1 para UTF-8)
    try:
        # Tentativa principal: o texto parece estar em latin-1 mas foi lido como UTF-8
        texto = texto.encode('latin-1', errors='ignore').decode('utf-8', errors='ignore')
    except:
        try:
            # Tentativa alternativa: às vezes é o contrário
            texto = texto.encode('utf-8', errors='ignore').decode('latin-1', errors='ignore')
        except:
            pass
    
    # SEGUNDO: substituições específicas baseadas nos dados do TITULO_MAPAS
    substituicoes = {
        # Caracteres acentuados básicos
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú', 'Ã¼': 'ü',
        'Ã£': 'ã', 'Ãµ': 'õ', 'Ã±': 'ñ', 'Ã§': 'ç', 'Ãª': 'ê', 'Ã´': 'ô',
        'Ã ': 'à', 'Ã\xa0': 'à', 'Ã¢': 'â', 'Ã¨': 'è', 'Ã«': 'ë', 'Ã¯': 'ï',
        
        # Combinações comuns
        'Ã§Ã£': 'ção',
        'Ã§Ãµ': 'ções',
        'Ã³es': 'ões',
        'Ãµes': 'ões',
        'Ã¡rio': 'ário',
        'Ã¡ria': 'ária',
        'Ã³rio': 'ório',
        'Ã³ria': 'ória',
        'Ã©s': 'és',          # para "francês", "português"
        'Ã´nio': 'ônio',       # para "antônio"
        'Ã¢nia': 'ânia',       # para "companhia"
        'Ã¡gua': 'água',
        'Ã¡reas': 'áreas',
        'Ã©difica': 'édifica',
        'Ã³leo': 'óleo',
        'Ãºnico': 'único',
        'Ãºnica': 'única',
        
        # Palavras frequentes nos títulos dos mapas (baseado na amostra)
        'aerofotogramÃ©trico': 'aerofotogramétrico',
        'aerofotogramÃ©trica': 'aerofotogramétrica',
        'topogrÃ¡fica': 'topográfica',
        'topogrÃ¡fico': 'topográfico',
        'planialtimÃ©trico': 'planialtimétrico',
        'planialtimÃ©trica': 'planialtimétrica',
        'planimÃ©trico': 'planimétrico',
        'planimÃ©trica': 'planimétrica',
        'hipsomÃ©trico': 'hipsométrico',
        'hipsomÃ©trica': 'hipsométrica',
        'hipsometria': 'hipsometria',  # já está correto? cuidado
        'hidrogrÃ¡fica': 'hidrográfica',
        'hidrogrÃ¡fico': 'hidrográfico',
        'geolÃ³gico': 'geológico',
        'geolÃ³gica': 'geológica',
        'geomorfolÃ³gica': 'geomorfológica',
        'geomorfolÃ³gico': 'geomorfológico',
        'cartogrÃ¡fica': 'cartográfica',
        'cartogrÃ¡fico': 'cartográfico',
        'cadastral': 'cadastral',  # pode estar ok
        'esgotamento': 'esgotamento',
        'saneamento': 'saneamento',
        'abastecimento': 'abastecimento',
        'ocupaÃ§Ã£o': 'ocupação',
        'ocupaÃ§Ãµes': 'ocupações',
        'legislaÃ§Ã£o': 'legislação',
        'estruturaÃ§Ã£o': 'estruturação',
        'implantaÃ§Ã£o': 'implantação',
        'delimitaÃ§Ã£o': 'delimitação',
        'desapropriaÃ§Ã£o': 'desapropriação',
        'desapropriaÃ§Ãµes': 'desapropriações',
        'localizaÃ§Ã£o': 'localização',
        'conservaÃ§Ã£o': 'conservação',
        'preservaÃ§Ã£o': 'preservação',
        'valorizaÃ§Ã£o': 'valorização',
        'caracterizaÃ§Ã£o': 'caracterização',
        'macrocaracterizaÃ§Ã£o': 'macrocaracterização',
        'compartimentaÃ§Ã£o': 'compartimentação',
        'erodibilidade': 'erodibilidade',
        'suscetibilidade': 'suscetibilidade',
        'susceptibilidade': 'susceptibilidade',
        'erodibilidade': 'erodibilidade',
        'geotÃ©cnico': 'geotécnico',
        'geotÃ©cnica': 'geotécnica',
        'litologia': 'litologia',
        'pedologia': 'pedologia',
        'vegetaÃ§Ã£o': 'vegetação',
        'reflorestamento': 'reflorestamento',
        'declividade': 'declividade',
        'declividades': 'declividades',
        'drenagem': 'drenagem',
        'hidrografia': 'hidrografia',
        'hidrologia': 'hidrologia',
        'topografia': 'topografia',
        'geomorfologia': 'geomorfologia',
        'hipsometria': 'hipsometria',
        'hipsomÃ©trico': 'hipsométrico',
        'clinografia': 'clinografia',
        'bacias': 'bacias',
        'sub-bacias': 'sub-bacias',
        'mananciais': 'mananciais',
        'reservatÃ³rio': 'reservatório',
        'reservatÃ³rios': 'reservatórios',
        'captaÃ§Ãµes': 'captações',
        'cursos': 'cursos',
        'd\'Ã¡gua': 'd\'água',
        'perÃ­metro': 'perímetro',
        'perÃ­metros': 'perímetros',
        'zoneamento': 'zoneamento',
        'zoneamentos': 'zoneamentos',
        'macrozoneamento': 'macrozoneamento',
        'microzoneamento': 'microzoneamento',
        'uso': 'uso',
        'ocupaÃ§Ã£o': 'ocupação',
        'solo': 'solo',
        'aptidÃ£o': 'aptidão',
        'aptidÃµes': 'aptidões',
        'adequabilidade': 'adequabilidade',
        'recomendaÃ§Ã£o': 'recomendação',
        'recomendaÃ§Ãµes': 'recomendações',
        'padrÃµes': 'padrões',
        'proposta': 'proposta',
        'propostas': 'propostas',
        'situaÃ§Ã£o': 'situação',
        'situaÃ§Ãµes': 'situações',
        'diagnÃ³stico': 'diagnóstico',
        'diagnÃ³sticos': 'diagnósticos',
        'prognÃ³stico': 'prognóstico',
        'prognÃ³sticos': 'prognósticos',
        'diretriz': 'diretriz',
        'diretrizes': 'diretrizes',
        'estratÃ©gia': 'estratégia',
        'estratÃ©gias': 'estratégias',
        'planejamento': 'planejamento',
        'gestÃ£o': 'gestão',
        'desenvolvimento': 'desenvolvimento',
        'crescimento': 'crescimento',
        'expansÃ£o': 'expansão',
        'evoluÃ§Ã£o': 'evolução',
        'ocupaÃ§Ã£o': 'ocupação',
        'ocupaÃ§Ãµes': 'ocupações',
        'urbanizaÃ§Ã£o': 'urbanização',
        'rural': 'rural',
        'urbano': 'urbano',
        'urbanos': 'urbanos',
        'municipal': 'municipal',
        'municipais': 'municipais',
        'municÃ­pio': 'município',
        'municÃ­pios': 'municípios',
        'distrito': 'distrito',
        'distritos': 'distritos',
        'sede': 'sede',
        'sedes': 'sedes',
        'nÃºcleo': 'núcleo',
        'nÃºcleos': 'núcleos',
        'perÃ­metro': 'perímetro',
        'perÃ­metros': 'perímetros',
        'limite': 'limite',
        'limites': 'limites',
        'divisa': 'divisa',
        'divisas': 'divisas',
        'fronteira': 'fronteira',
        'fronteiras': 'fronteiras',
        'regional': 'regional',
        'regionais': 'regionais',
        'metropolitano': 'metropolitano',
        'metropolitana': 'metropolitana',
        'metropolitanos': 'metropolitanos',
        'metropolitanas': 'metropolitanas',
        'RMC': 'RMC',  # sigla
        'COMEC': 'COMEC',
        'URBS': 'URBS',
        'SANEPAR': 'SANEPAR',
        'IAP': 'IAP',
        'IBGE': 'IBGE',
        'PDM': 'PDM',  # Plano de Desenvolvimento Municipal
        'PEU': 'PEU',  # Plano de Estruturação Urbana
        'UTP': 'UTP',  # Unidade Territorial de Planejamento
        'APA': 'APA',  # Área de Proteção Ambiental
        'APP': 'APP',  # Área de Preservação Permanente
        'ZEIS': 'ZEIS',
        'ZUPI': 'ZUPI',
        'COHAB': 'COHAB',
        'INOCOOP': 'INOCOOP',
        'PROSAM': 'PROSAM',
        'CIAC': 'CIAC',
        'CEF': 'CEF',
        'TELEPAR': 'TELEPAR',
        'KARST': 'Karst',  # ou 'Carste'?
        'Karst': 'Karst',
        'Carste': 'Carste',
        'AUDI': 'AUDI',
        'WOLKS': 'Wolks',
        
        # Nomes geográficos e municípios
        'ParanÃ¡': 'Paraná',
        'Curitiba': 'Curitiba',
        'SÃ£o': 'São',
        'JosÃ©': 'José',
        'dos': 'dos',
        'Pinhais': 'Pinhais',
        'IguaÃ§u': 'Iguaçu',
        'IguaÃ§Ãº': 'Iguaçu',
        'PassaÃºna': 'Passaúna',
        'IraÃ­': 'Iraí',
        'Piraquara': 'Piraquara',
        'Barigui': 'Barigui',
        'Colombo': 'Colombo',
        'TamandarÃ©': 'Tamandaré',
        'Campo': 'Campo',
        'Magro': 'Magro',
        'Fazenda': 'Fazenda',
        'Rio': 'Rio',
        'Grande': 'Grande',
        'ItaperuÃ§u': 'Itaperuçu',
        'Quatro': 'Quatro',
        'Barras': 'Barras',
        'Almirante': 'Almirante',
        'Campina': 'Campina',
        'AraucÃ¡ria': 'Araucária',
        'Balsa': 'Balsa',
        'Nova': 'Nova',
        'Contenda': 'Contenda',
        'Mandirituba': 'Mandirituba',
        'BocaiÃºva': 'Bocaiúva',
        'Guarituba': 'Guarituba',
        'Itaqui': 'Itaqui',
        'Guadalupe': 'Guadalupe',
        'MaracanÃ£': 'Maracanã',
        'Renault': 'Renault',
        'Alphaville': 'Alphaville',
        'Graciosa': 'Graciosa',
        'Guajuvira': 'Guajuvira',
        'Bugre': 'Bugre',
        'Recife': 'Recife',
        'Porto': 'Porto',
        'Alegre': 'Alegre',
        'Salvador': 'Salvador',
        'Londrina': 'Londrina',
        'MaringÃ¡': 'Maringá',
        'FlorianÃ³polis': 'Florianópolis',
        'Monterrey': 'Monterrey',
        'BolÃ­via': 'Bolívia',
        'Brasil': 'Brasil',
        'Abrantes': 'Abrantes',
        'Araraquara': 'Araraquara',
        'Areia': 'Areia',
        'Branca': 'Branca',
        'Assis': 'Assis',
        'Barra': 'Barra',
        'ChapÃ©u': 'Chapéu',
        'Turvo': 'Turvo',
        'BraÃ§o': 'Braço',
        'Cachoeira': 'Cachoeira',
        'Lamenha': 'Lamenha',
        'TanguÃ¡': 'Tanguá',
        'Campina': 'Campina',
        'Elias': 'Elias',
        'Cerro': 'Cerro',
        'Azul': 'Azul',
        'ColÃ´nia': 'Colônia',
        'Santos': 'Santos',
        'Andrade': 'Andrade',
        'IapÃ³': 'Iapó',
        'Castro': 'Castro',
        'Catanduvas': 'Catanduvas',
        'CuriÃºva': 'Curiúva',
        'Doutor': 'Doutor',
        'Ulisses': 'Ulisses',
        'Eldorado': 'Eldorado',
        'Garuva': 'Garuva',
        'GuaraqueÃ§aba': 'Guaraqueçaba',
        'Guaratuba': 'Guaratuba',
        'Ibaiti': 'Ibaiti',
        'Ilha': 'Ilha',
        'Mel': 'Mel',
        'Iporanga': 'Iporanga',
        'Itaiacoca': 'Itaiacoca',
        'ItararÃ©': 'Itararé',
        'Jaboti': 'Jaboti',
        'JaguariaÃ­va': 'Jaguariaíva',
        'Japira': 'Japira',
        'Joaquim': 'Joaquim',
        'Murtinho': 'Murtinho',
        'Lapa': 'Lapa',
        'Matinhos': 'Matinhos',
        'Morretes': 'Morretes',
        'Mundo': 'Mundo',
        'Novo': 'Novo',
        'Ouro': 'Ouro',
        'Verde': 'Verde',
        'Palmeira': 'Palmeira',
        'ParanaguÃ¡': 'Paranaguá',
        'Pedra': 'Pedra',
        'Branca': 'Branca',
        'PiÃªn': 'Piên',
        'PinhalÃ£o': 'Pinhalão',
        'PiraÃ­': 'Piraí',
        'Ponta': 'Ponta',
        'Grossa': 'Grossa',
        'Pontal': 'Pontal',
        'Quero-quero': 'Quero-quero',
        'Quitandinha': 'Quitandinha',
        'Represa': 'Represa',
        'Capivari': 'Capivari',
        'Ribeira': 'Ribeira',
        'RibeirÃ£o': 'Ribeirão',
        'Rocha': 'Rocha',
        'SengÃ©s': 'Sengés',
        'Serra': 'Serra',
        'Virgem': 'Virgem',
        'Maria': 'Maria',
        'Serrinha': 'Serrinha',
        'SocavÃ£o': 'Socavão',
        'TelÃªmaco': 'Telêmaco',
        'Borba': 'Borba',
        'Tibagi': 'Tibagi',
        'Tijucas': 'Tijucas',
        'Tomazina': 'Tomazina',
        'Tunas': 'Tunas',
        'VarzeÃ£o': 'Varzão',
        'Ventania': 'Ventania',
        'Vila': 'Vila',
        'Branca': 'Branca',
        'Wenceslau': 'Wenceslau',
        'Braz': 'Braz',
        
        # Termos técnicos específicos
        'loteamento': 'loteamento',
        'loteamentos': 'loteamentos',
        'arruamento': 'arruamento',
        'pavimentaÃ§Ã£o': 'pavimentação',
        'saneamento': 'saneamento',
        'esgotamento': 'esgotamento',
        'abastecimento': 'abastecimento',
        'energia': 'energia',
        'elÃ©trica': 'elétrica',
        'elÃ©trico': 'elétrico',
        'telecomunicaÃ§Ãµes': 'telecomunicações',
        'comunicaÃ§Ãµes': 'comunicações',
        'transporte': 'transporte',
        'transportes': 'transportes',
        'coletivo': 'coletivo',
        'pÃºblico': 'público',
        'pÃºblica': 'pública',
        'pÃºblicos': 'públicos',
        'pÃºblicas': 'públicas',
        'viÃ¡rio': 'viário',
        'viÃ¡ria': 'viária',
        'viÃ¡rios': 'viários',
        'viÃ¡rias': 'viárias',
        'rodoviÃ¡rio': 'rodoviário',
        'rodoviÃ¡ria': 'rodoviária',
        'ferroviÃ¡rio': 'ferroviário',
        'ferroviÃ¡ria': 'ferroviária',
        'aeroviÃ¡rio': 'aeroviário',
        'aeroviÃ¡ria': 'aeroviária',
        'hidroviÃ¡rio': 'hidroviário',
        'hidroviÃ¡ria': 'hidroviária',
        'terminal': 'terminal',
        'terminais': 'terminais',
        'estaÃ§Ã£o': 'estação',
        'estaÃ§Ãµes': 'estações',
        'parque': 'parque',
        'parques': 'parques',
        'reserva': 'reserva',
        'reservas': 'reservas',
        'unidade': 'unidade',
        'unidades': 'unidades',
        'conservaÃ§Ã£o': 'conservação',
        'preservaÃ§Ã£o': 'preservação',
        'proteÃ§Ã£o': 'proteção',
        'recuperaÃ§Ã£o': 'recuperação',
        'manejo': 'manejo',
        'florestal': 'florestal',
        'florestais': 'florestais',
        'reflorestamento': 'reflorestamento',
        'silvicultura': 'silvicultura',
        'agropecuÃ¡rio': 'agropecuário',
        'agropecuÃ¡ria': 'agropecuária',
        'agricultura': 'agricultura',
        'pecuÃ¡ria': 'pecuária',
        'irrigaÃ§Ã£o': 'irrigação',
        'drenagem': 'drenagem',
        'erosÃ£o': 'erosão',
        'inundaÃ§Ã£o': 'inundação',
        'inundaÃ§Ãµes': 'inundações',
        'enchente': 'enchente',
        'enchentes': 'enchentes',
        'cheia': 'cheia',
        'cheias': 'cheias',
        'risco': 'risco',
        'riscos': 'riscos',
        'vulnerabilidade': 'vulnerabilidade',
        'susceptibilidade': 'susceptibilidade',
        'suscetibilidade': 'suscetibilidade',
        'erodibilidade': 'erodibilidade',
        'escorregamento': 'escorregamento',
        'escorregamentos': 'escorregamentos',
        'deslizamento': 'deslizamento',
        'deslizamentos': 'deslizamentos',
        'solapamento': 'solapamento',
        'solapamentos': 'solapamentos',
        'assoreamento': 'assoreamento',
        'assoreamentos': 'assoreamentos',
        'poluiÃ§Ã£o': 'poluição',
        'poluente': 'poluente',
        'poluentes': 'poluentes',
        'contaminaÃ§Ã£o': 'contaminação',
        'contaminante': 'contaminante',
        'contaminantes': 'contaminantes',
        'resÃ­duo': 'resíduo',
        'resÃ­duos': 'resíduos',
        'lixo': 'lixo',
        'efluente': 'efluente',
        'efluentes': 'efluentes',
        'esgoto': 'esgoto',
        'esgotos': 'esgotos',
        'sÃ³lido': 'sólido',
        'sÃ³lidos': 'sólidos',
        'lÃ­quido': 'líquido',
        'lÃ­quidos': 'líquidos',
        'gasoso': 'gasoso',
        'gasosos': 'gasosos',
        'atmosfÃ©rica': 'atmosférica',
        'atmosfÃ©rico': 'atmosférico',
        'climatologia': 'climatologia',
        'clima': 'clima',
        'climÃ¡tico': 'climático',
        'climÃ¡tica': 'climática',
        'meteorologia': 'meteorologia',
        'meteorolÃ³gico': 'meteorológico',
        'meteorolÃ³gica': 'meteorológica',
        'pluviometria': 'pluviometria',
        'pluviomÃ©trico': 'pluviométrico',
        'pluviomÃ©trica': 'pluviométrica',
        'temperatura': 'temperatura',
        'temperaturas': 'temperaturas',
        'umidade': 'umidade',
        'ventos': 'ventos',
        'direÃ§Ã£o': 'direção',
        'direÃ§Ãµes': 'direções',
        'velocidade': 'velocidade',
        'velocidades': 'velocidades',
        'intensidade': 'intensidade',
        'intensidades': 'intensidades',
        'freqÃ¼Ãªncia': 'frequência',
        'freqÃ¼Ãªncias': 'frequências',
        'duraÃ§Ã£o': 'duração',
        'duraÃ§Ãµes': 'duraçõess',
        
        # Correções de palavras compostas
        'meio-ambiente': 'meio ambiente',
        'meio ambiente': 'meio ambiente',
        'recursos hÃ­dricos': 'recursos hídricos',
        'recursos hÃ­dricas': 'recursos hídricos',
        'sistema viÃ¡rio': 'sistema viário',
        'sistemas viÃ¡rios': 'sistemas viários',
        'uso do solo': 'uso do solo',
        'ocupaÃ§Ã£o do solo': 'ocupação do solo',
        'cobertura vegetal': 'cobertura vegetal',
        'vegetaÃ§Ã£o': 'vegetação',
        'formaÃ§Ã£o vegetal': 'formação vegetal',
        'formaÃ§Ãµes vegetais': 'formações vegetais',
        'espÃ©cie': 'espécie',
        'espÃ©cies': 'espécies',
        'fauna': 'fauna',
        'flora': 'flora',
        
        # Unidades de medida e abreviações
        'kmÂ²': 'km²',
        'mÂ²': 'm²',
        'ha': 'ha',
        'm': 'm',
        'cm': 'cm',
        'mm': 'mm',
        'kg': 'kg',
        't': 't',
        'L': 'L',
        'mL': 'mL',
        
        # Caracteres especiais remanescentes
        'Âº': 'º',
        'Âª': 'ª',
        'Â°': '°',
        'â‚¬': '€',
        'â„¢': '™',
        'â€': '”',
        'â€œ': '“',
        'â€˜': '‘',
        'â€™': '’',
        'â€¢': '•',
        'â€“': '–',
        'â€”': '—',
        'â€¡': '‡',
        'â€¹': '‹',
        'â€º': '›',
        'â€¦': '…',
        'âˆ': '√',
        'â‰': '≠',
        'â‰¤': '≤',
        'â‰¥': '≥',
        'Ã—': '×',
        'Ã·': '÷',
        'Ã¸': 'ø',
        'Ã¦': 'æ',
        'Ã°': 'ð',
        'Ã¾': 'þ',
        'Ã¿': 'ÿ',
        'Å’': 'Œ',
        'Å“': 'œ',
        'Å ': 'Š',
        'Å¡': 'š',
        'Å½': 'Ž',
        'Å¾': 'ž',
        'Å¸': 'Ÿ',
        'Å': 'Ł',
        'Å‚': 'ł',
        'Å': 'Ő',
        'Å‘': 'ő',
        'Å’': 'Œ',
        'Å“': 'œ',
        'Å®': 'Ů',
        'Å¯': 'ů',
        'Å°': 'Ű',
        'Å±': 'ű',
        'Å²': 'Ų',
        'Å³': 'ų',
        
        # Limpeza de tags HTML ou formatação
        '<A>': 'A',
        '<O>': 'O',
        '<The>': 'The',
        'NULL': '',
        '#NOME?': '',
        '#REF!': '',
        '#DIV/0!': '',
        '#VALOR!': '',
        '#NUM!': '',
        '#N/A': '',
    }
    
    for errado, correto in substituicoes.items():
        texto = texto.replace(errado, correto)
    
    # TERCEIRO: tratamento específico para números e códigos
    import re
    
    # Corrigir padrões como "1:50.000" (separador de milhar com ponto)
    texto = re.sub(r'(\d):(\d{3})', r'\1:\2', texto)  # mantém os dois pontos, mas sem separador
    
    # Corrigir espaços extras
    texto = re.sub(r'\s+', ' ', texto)
    
    # Remover caracteres não imprimíveis
    texto = ''.join(char for char in texto if ord(char) >= 32 or char in '\n\t\r')
    
    texto = texto.strip()
    
    return texto

# ========== 4. APLICAR CORREÇÃO ==========
print("\n🔄 APLICANDO CORREÇÃO DE CARACTERES...")

# Verificar estrutura do DataFrame
print(f"Colunas disponíveis: {list(df.columns)}")
print(f"Nome da primeira coluna: '{df.columns[0]}'")

# A coluna parece não ter nome, então vamos renomear para facilitar
if df.columns[0] == 'A' or df.columns[0] == 0 or df.columns[0] == '0':
    df = df.rename(columns={df.columns[0]: 'titulo_mapa'})
    coluna_alvo = 'titulo_mapa'
else:
    coluna_alvo = df.columns[0]

print(f"📝 Coluna alvo para correção: '{coluna_alvo}'")

# Criar cópia para comparação
df['titulo_original'] = df[coluna_alvo].astype(str).copy()

# Aplicar correção
antes = df[coluna_alvo].astype(str).iloc[0]
df[coluna_alvo] = df[coluna_alvo].astype(str).apply(corrigir_caracteres_titulos_mapas)
depois = df[coluna_alvo].iloc[0]

print(f"✅ Correção aplicada em {len(df)} registros")
print(f"\n📋 Exemplo de correção:")
print(f"   ANTES: {antes[:80]}...")
print(f"   DEPOIS: {depois[:80]}...")

# ========== 5. COMPARAR ANTES/DEPOIS ==========
print("\n🔍 COMPARAÇÃO DETALHADA ANTES/DEPOIS:")

# Criar DataFrame de comparação com 15 exemplos
comparacao = pd.DataFrame({
    'ORIGINAL': df['titulo_original'].head(15),
    'CORRIGIDO': df[coluna_alvo].head(15)
})

display(HTML(comparacao.to_html()))

# Mostrar exemplos específicos baseados nos dados que vi
print("\n📋 EXEMPLOS ESPECÍFICOS DE CORREÇÃO (baseados na amostra):")

exemplos_corrigidos = [
    ("Levantamento aerofotogramÃ©trico da RegiÃ£o Metropolitana de Curitiba", 
     "Levantamento aerofotogramétrico da Região Metropolitana de Curitiba"),
    
    ("AtualizaÃ§Ã£o topogrÃ¡fica", 
     "Atualização topográfica"),
    
    ("Carta de hipsometria", 
     "Carta de hipsometria"),
    
    ("Arruamento e ocupaÃ§Ãµes", 
     "Arruamento e ocupações"),
    
    ("Ãrea da PetrobrÃ¡s", 
     "Área da Petrobrás"),
    
    ("Levantamento planialtimÃ©trico", 
     "Levantamento planialtimétrico"),
    
    ("Mapa rodoviÃ¡rio", 
     "Mapa rodoviário"),
    
    ("Plano de estruturaÃ§Ã£o urbana", 
     "Plano de estruturação urbana"),
    
    ("Zoneamento espacial da legislaÃ§Ã£o municipal", 
     "Zoneamento espacial da legislação municipal"),
    
    ("VegetaÃ§Ã£o", 
     "Vegetação"),
    
    ("Hidrografia", 
     "Hidrografia"),
    
    ("Vias rurais", 
     "Vias rurais"),
    
    ("PerÃ­metro urbano no zoneamento", 
     "Perímetro urbano no zoneamento"),
    
    ("AtualizaÃ§Ã£o da base: loteamentos (parte 1)", 
     "Atualização da base: loteamentos (parte 1)"),
    
    ("Levantamento aerofotogramÃ©trico de Piraquara", 
     "Levantamento aerofotogramétrico de Piraquara"),
    
    ("Ãreas invadidas", 
     "Áreas invadidas"),
    
    ("Morfologia", 
     "Morfologia"),
    
    ("PadrÃµes e recomendaÃ§Ãµes do uso do solo", 
     "Padrões e recomendações do uso do solo"),
    
    ("Uso do solo: cobertura vegetal", 
     "Uso do solo: cobertura vegetal"),
    
    ("Unidades de conservaÃ§Ã£o", 
     "Unidades de conservação"),
    
    ("Bacias hidrogrÃ¡ficas", 
     "Bacias hidrográficas"),
    
    ("Levantamento planimÃ©trico para implantaÃ§Ã£o do Parque HistÃ³rico do Mate", 
     "Levantamento planimétrico para implantação do Parque Histórico do Mate"),
    
    ("Plano diretor de manejo florestal: situaÃ§Ã£o", 
     "Plano diretor de manejo florestal: situação"),
    
    ("IndicaÃ§Ã£o dos vÃ©rtices de referÃªncia geodÃ©sica", 
     "Indicação dos vértices de referência geodésica"),
    
    ("Ãreas para reassentamento do PROSAM: declividade", 
     "Áreas para reassentamento do PROSAM: declividade"),
    
    ("Parque metropolitado do IguaÃ§u", 
     "Parque metropolitano do Iguaçu"),  # Note: "metropolitado" provavelmente é erro, mas corrigimos o acento
    
    ("Distrito industrial de SÃ£o JosÃ© dos Pinhais", 
     "Distrito industrial de São José dos Pinhais"),
    
    ("UTP Campo Magro: energia elÃ©trica - consumidores residenciais", 
     "UTP Campo Magro: energia elétrica - consumidores residenciais"),
    
    ("Mapa do MunicÃ­pio de Campo Magro", 
     "Mapa do Município de Campo Magro"),
    
    ("Planta de referÃªncia cadastral", 
     "Planta de referência cadastral"),
    
    ("Mapa de emancipaÃ§Ã£o", 
     "Mapa de emancipação"),
    
    ("Energia elÃ©trica", 
     "Energia elétrica"),
    
    ("BocaiÃºva do Sul", 
     "Bocaiúva do Sul"),
    
    ("Campina do Elias", 
     "Campina do Elias"),
    
    ("Cerro Azul", 
     "Cerro Azul"),
    
    ("Contenda", 
     "Contenda"),
    
    ("Mundo Novo", 
     "Mundo Novo"),
    
    ("Ribeira", 
     "Ribeira"),
    
    ("Rio Branco do Sul", 
     "Rio Branco do Sul"),
    
    ("SÃ£o JosÃ© dos Pinhais", 
     "São José dos Pinhais"),
    
    ("SocavÃ£o", 
     "Socavão"),
    
    ("Tunas do ParanÃ¡", 
     "Tunas do Paraná"),
    
    ("Taxa de ocupaÃ§Ã£o por quadra", 
     "Taxa de ocupação por quadra"),
    
    ("Vila Branca", 
     "Vila Branca"),
    
    ("ColÃ´nia Santos Andrade", 
     "Colônia Santos Andrade"),
    
    ("Rio Preto do Sul", 
     "Rio Preto do Sul"),
    
    ("ApiaÃ­", 
     "Apiaí"),
    
    ("Barra do ChapÃ©u", 
     "Barra do Chapéu"),
    
    ("Barra do Turvo", 
     "Barra do Turvo"),
    
    ("BraÃ§o", 
     "Braço"),
    
    ("Campo Alegre", 
     "Campo Alegre"),
    
    ("Ibaiti", 
     "Ibaiti"),
    
    ("Iporanga", 
     "Iporanga"),
    
    ("Lapa", 
     "Lapa"),
    
    ("Mandirituba", 
     "Mandirituba"),
    
    ("MarquÃªs de Abrantes", 
     "Marquês de Abrantes"),
    
    ("Morretes", 
     "Morretes"),
    
    ("Novo Mundo", 
     "Novo Mundo"),
    
    ("Ouro Verde", 
     "Ouro Verde"),
    
    ("Pedra Branca do Araraquara", 
     "Pedra Branca do Araraquara"),
    
    ("Folha geolÃ³gica de Itaiacoca", 
     "Folha geológica de Itaiacoca"),
    
    ("Porto Amazonas", 
     "Porto Amazonas"),
    
    ("Quero-quero", 
     "Quero-quero"),
    
    ("Represa do Capivari", 
     "Represa do Capivari"),
    
    ("Serra da Virgem Maria", 
     "Serra da Virgem Maria"),
    
    ("Tijucas do Sul", 
     "Tijucas do Sul"),
    
    ("CuriÃºva", 
     "Curiúva"),
    
    ("JaguariaÃ­va", 
     "Jaguariaíva"),
    
    ("Levantamento geolÃ³gico de Barra de Ararapira", 
     "Levantamento geológico de Barra de Ararapira"),
    
    ("Levantamento aerofotogramÃ©trico de Curitiba", 
     "Levantamento aerofotogramétrico de Curitiba"),
    
    ("Levantamento geolÃ³gico de GuaraqueÃ§aba", 
     "Levantamento geológico de Guaraqueçaba"),
    
    ("Levantamento geolÃ³gico de Guaratuba", 
     "Levantamento geológico de Guaratuba"),
    
    ("Levantamento geolÃ³gico de Ilha do Mel", 
     "Levantamento geológico de Ilha do Mel"),
    
    ("Levantamento geolÃ³gico de Joaquim Murtinho", 
     "Levantamento geológico de Joaquim Murtinho"),
    
    ("Folha geolÃ³gica de PiraÃ­ do Sul", 
     "Folha geológica de Piraí do Sul"),
    
    ("Levantamento geolÃ³gico de PiÃªn", 
     "Levantamento geológico de Piên"),
    
    ("Levantamento geolÃ³gico de Rio Pardinho", 
     "Levantamento geológico de Rio Pardinho"),
    
    ("Levantamento geolÃ³gico de Serra da Igreja", 
     "Levantamento geológico de Serra da Igreja"),
    
    ("Folha geolÃ³gica de AbapÃ£", 
     "Folha geológica de Abapã"),
    
    ("Folha geolÃ³gica de AdrianÃ³polis", 
     "Folha geológica de Adrianópolis"),
    
    ("Folha geolÃ³gica de AraucÃ¡ria", 
     "Folha geológica de Araucária"),
    
    ("Folha geolÃ³gica de Barra do Pardo", 
     "Folha geológica de Barra do Pardo"),
    
    ("Folha geolÃ³gica de Campo Largo", 
     "Folha geológica de Campo Largo"),
    
    ("Folha geolÃ³gica de Castro", 
     "Folha geológica de Castro"),
    
    ("Folha geolÃ³gica de ColÃ´nia IapÃ³", 
     "Folha geológica de Colônia Iapó"),
    
    ("Folha geolÃ³gica de Contenda", 
     "Folha geológica de Contenda"),
    
    ("Folha geolÃ³gica de Curitiba", 
     "Folha geológica de Curitiba"),
    
    ("Folha geolÃ³gica de Piraquara", 
     "Folha geológica de Piraquara"),
    
    ("Folha geolÃ³gica de Porto Amazonas", 
     "Folha geológica de Porto Amazonas"),
    
    ("Folha geolÃ³gica de RibeirÃ£o do Rocha", 
     "Folha geológica de Ribeirão do Rocha"),
    
    ("Folha geolÃ³gica de SÃ£o JosÃ© dos Pinhais", 
     "Folha geológica de São José dos Pinhais"),
    
    ("Folha geolÃ³gica de Tibagi", 
     "Folha geológica de Tibagi"),
    
    ("Folha geolÃ³gica de VarzeÃ£o", 
     "Folha geológica de Varzão"),
    
    ("Planta de referÃªncia cadastral: Contenda - sede", 
     "Planta de referência cadastral: Contenda - sede"),
    
    ("Planta de referÃªncia cadastral: Contenda - distrito de Serrinha", 
     "Planta de referência cadastral: Contenda - distrito de Serrinha"),
    
    ("Planta de referÃªncia cadastral: Contenda - distrito de Catanduvas do Sul", 
     "Planta de referência cadastral: Contenda - distrito de Catanduvas do Sul"),
    
    ("Coleta de lixo", 
     "Coleta de lixo"),
    
    ("Planta preliminar de arruamento", 
     "Planta preliminar de arruamento"),
    
    ("Agricultura mecanizada de ciclo curto e pecuÃ¡ria com pastagens plantadas", 
     "Agricultura mecanizada de ciclo curto e pecuária com pastagens plantadas"),
    
    ("Unidades de ambiente silvi-agropecuÃ¡rio: aptidÃ£o da estrutura edÃ¡fica e geomÃ³rfica", 
     "Unidades de ambiente silvi-agropecuário: aptidão da estrutura edáfica e geomórfica"),
    
    ("Uso agroecolÃ³gico adequado", 
     "Uso agroecológico adequado"),
    
    ("Susceptibilidade das unidades de ambiente silvi-agropecuÃ¡rio Ã  erosÃ£o", 
     "Susceptibilidade das unidades de ambiente silvi-agropecuário à erosão"),
    
    ("Estimativas de horas de frio", 
     "Estimativas de horas de frio"),
    
    ("Relevo", 
     "Relevo"),
    
    ("Geologia", 
     "Geologia"),
    
    ("Carta planialtimÃ©trica", 
     "Carta planialtimétrica"),
    
    ("Base planimÃ©trica", 
     "Base planimétrica"),
    
    ("LocalizaÃ§Ã£o de indÃºstrias", 
     "Localização de indústrias"),
    
    ("Declividade", 
     "Declividade"),
    
    ("VegetaÃ§Ã£o e hidrografia", 
     "Vegetação e hidrografia"),
    
    ("Sistema de abastecimento de Ã¡gua", 
     "Sistema de abastecimento de água"),
    
    ("Esgoto", 
     "Esgoto"),
    
    ("Sistema de energia elÃ©trica", 
     "Sistema de energia elétrica"),
    
    ("Sistema de telecomunicaÃ§Ã£o", 
     "Sistema de telecomunicação"),
    
    ("Sistema de coleta de lixo", 
     "Sistema de coleta de lixo"),
    
    ("EducaÃ§Ã£o", 
     "Educação"),
    
    ("SaÃºde e assistÃªncia social", 
     "Saúde e assistência social"),
    
    ("Lazer e assistÃªncia comunitÃ¡ria", 
     "Lazer e assistência comunitária"),
    
    ("Cultura e esporte", 
     "Cultura e esporte"),
    
    ("Sistema viÃ¡rio", 
     "Sistema viário"),
    
    ("PavimentaÃ§Ã£o", 
     "Pavimentação"),
    
    ("Zoneamento urbano", 
     "Zoneamento urbano"),
    
    ("Terrenos especiais", 
     "Terrenos especiais"),
    
    ("EvoluÃ§Ã£o urbana", 
     "Evolução urbana"),
    
    ("Quadro urbano", 
     "Quadro urbano"),
    
    ("InvasÃµes", 
     "Invasões"),
    
    ("Equipamento de saÃºde/educaÃ§Ã£o", 
     "Equipamento de saúde/educação"),
    
    ("Loteamento", 
     "Loteamento"),
    
    ("Faixa nÃ£o - edificÃ¡vel", 
     "Faixa não - edificável"),
    
    ("Circuito TamandarÃ© de turismo rural", 
     "Circuito Tamandaré de turismo rural"),
    
    ("Limpeza pÃºblica", 
     "Limpeza pública"),
    
    ("TelecomunicaÃ§Ãµes", 
     "Telecomunicações"),
    
    ("Saneamento bÃ¡sico", 
     "Saneamento básico"),
    
    ("Abastecimento de Ã¡gua, esgoto e energia elÃ©trica", 
     "Abastecimento de água, esgoto e energia elétrica"),
    
    ("Pontos de Ã´nibus", 
     "Pontos de ônibus"),
    
    ("Planta de localidades de Cachoeira, Lamenha e TanguÃ¡", 
     "Planta de localidades de Cachoeira, Lamenha e Tanguá"),
    
    ("RestituiÃ§Ã£o", 
     "Restituição"),
    
    ("Mapa base", 
     "Mapa base"),
    
    ("Rio IguaÃ§u", 
     "Rio Iguaçu"),
    
    ("Guajuvira: padrÃµes e recomendaÃ§Ãµes de uso do solo", 
     "Guajuvira: padrões e recomendações de uso do solo"),
    
    ("Guajuvira: solos - sÃ­ntese", 
     "Guajuvira: solos - síntese"),
    
    ("Guajuvira: cobertura vegetal e uso atual do solo", 
     "Guajuvira: cobertura vegetal e uso atual do solo"),
    
    ("Guajuvira: aptidÃµes", 
     "Guajuvira: aptidões"),
    
    ("Guajuvira: geologia", 
     "Guajuvira: geologia"),
    
    ("Guajuvira: declividades - sÃ­ntese", 
     "Guajuvira: declividades - síntese"),
    
    ("Guajuvira: drenagem", 
     "Guajuvira: drenagem"),
    
    ("Guajuvira: hidrografia", 
     "Guajuvira: hidrografia"),
    
    ("Guajuvira: sistema viÃ¡rio bÃ¡sico", 
     "Guajuvira: sistema viário básico"),
    
    ("Guajuvira: escolas", 
     "Guajuvira: escolas"),
    
    ("Guajuvira: hipsometria", 
     "Guajuvira: hipsometria"),
    
    ("Densidade/abastecimento de Ã¡gua", 
     "Densidade/abastecimento de água"),
    
    ("Sistema de transporte coletivo", 
     "Sistema de transporte coletivo"),
    
    ("Mapa dos loteamentos e limite do quadro urbano", 
     "Mapa dos loteamentos e limite do quadro urbano"),
    
    ("Bacias (parte 1)", 
     "Bacias (parte 1)"),
    
    ("EvoluÃ§Ã£o urbana de AraucÃ¡ria", 
     "Evolução urbana de Araucária"),
    
    ("Centro industrial: uso do solo", 
     "Centro industrial: uso do solo"),
    
    ("OcupaÃ§Ã£o urbana", 
     "Ocupação urbana"),
    
    ("BUGRE: solos", 
     "BUGRE: solos"),
    
    ("BUGRE: cobertura vegetal e uso atual do solo", 
     "BUGRE: cobertura vegetal e uso atual do solo"),
    
    ("BUGRE: uso rural do solo", 
     "BUGRE: uso rural do solo"),
    
    ("Topografia", 
     "Topografia"),
    
    ("Divisa municipal da apa da escarpa devoniana", 
     "Divisa municipal da APA da escarpa devoniana"),
    
    ("Sistema viÃ¡rio bÃ¡sico/zoneamento do uso do solo", 
     "Sistema viário básico/zoneamento do uso do solo"),
    
    ("Planta do perÃ­metro urbano e zoneamento da cidade de Balsa Nova", 
     "Planta do perímetro urbano e zoneamento da cidade de Balsa Nova"),
    
    ("BUGRE: planialtimetria", 
     "BUGRE: planialtimetria"),
    
    ("BUGRE: drenagem", 
     "BUGRE: drenagem"),
    
    ("BUGRE: declividades", 
     "BUGRE: declividades"),
    
    ("BUGRE: hipsometria", 
     "BUGRE: hipsometria"),
    
    ("BUGRE: equipamentos", 
     "BUGRE: equipamentos"),
    
    ("Planta do municÃ­pio de AraucÃ¡ria", 
     "Planta do município de Araucária"),
    
    ("Escolas", 
     "Escolas"),
    
    ("CirculaÃ§Ã£o de Ã´nibus urbano", 
     "Circulação de ônibus urbano"),
    
    ("Povoados e escolas", 
     "Povoados e escolas"),
    
    ("Faixas de proteÃ§Ã£o de fundos de vale", 
     "Faixas de proteção de fundos de vale"),
    
    ("Hipsometria", 
     "Hipsometria"),
    
    ("RecomendaÃ§Ãµes de uso do solo", 
     "Recomendações de uso do solo"),
    
    ("Tipo de solo", 
     "Tipo de solo"),
    
    ("Plano de estruturaÃ§Ã£o urbana: equipamento de saÃºde/educaÃ§Ã£o", 
     "Plano de estruturação urbana: equipamento de saúde/educação"),
    
    ("Plano de estruturaÃ§Ã£o urbana: zoneamento de uso do solo", 
     "Plano de estruturação urbana: zoneamento de uso do solo"),
    
    ("Plano de estruturaÃ§Ã£o urbana: sistema viÃ¡rio bÃ¡sico", 
     "Plano de estruturação urbana: sistema viário básico"),
    
    ("Arruamento e energia elÃ©trica", 
     "Arruamento e energia elétrica"),
    
    ("Mapa geolÃ³gico da regiÃ£o de BocaiÃºva do Sul", 
     "Mapa geológico da região de Bocaiúva do Sul"),
    
    ("Declividades", 
     "Declividades"),
    
    ("AtualizaÃ§Ã£o da base", 
     "Atualização da base"),
    
    ("Ãrea de tarifa bÃ¡sica: TELEPAR", 
     "Área de tarifa básica: TELEPAR"),
    
    ("Jardim Guaraituba: planta de loteamento", 
     "Jardim Guaraituba: planta de loteamento"),
    
    ("Centro industrial de Campina Grande do Sul", 
     "Centro industrial de Campina Grande do Sul"),
    
    ("Fundos de vale e Ã¡rea verde", 
     "Fundos de vale e área verde"),
    
    ("Planta geral de loteamentos", 
     "Planta geral de loteamentos"),
    
    ("Hidrologia", 
     "Hidrologia"),
    
    ("Condicionantes naturais e antrÃ³picos", 
     "Condicionantes naturais e antrópicos"),
    
    ("Uso do solo atual", 
     "Uso do solo atual"),
    
    ("Densidade de ocupaÃ§Ã£o", 
     "Densidade de ocupação"),
    
    ("Infra - estrutura I", 
     "Infra - estrutura I"),
    
    ("Infra - estrutura II", 
     "Infra - estrutura II"),
    
    ("Transporte coletivo", 
     "Transporte coletivo"),
    
    ("EducaÃ§Ã£o", 
     "Educação"),
    
    ("Atendimento Ã  infÃ¢ncia", 
     "Atendimento à infância"),
    
    ("SaÃºde", 
     "Saúde"),
    
    ("Condicionantes naturais e antrÃ³picos: Ferraria", 
     "Condicionantes naturais e antrópicos: Ferraria"),
    
    ("Infra - estrutura: Ferraria", 
     "Infra - estrutura: Ferraria"),
    
    ("ServiÃ§os pÃºblicos: Ferraria", 
     "Serviços públicos: Ferraria"),
    
    ("Condicionantes naturais e antrÃ³picos: Bateias", 
     "Condicionantes naturais e antrópicos: Bateias"),
    
    ("Infra - estrutura: Bateias", 
     "Infra - estrutura: Bateias"),
    
    ("SaÃºde e educaÃ§Ã£o: Bateias", 
     "Saúde e educação: Bateias"),
    
    ("Zonas homogÃªneas", 
     "Zonas homogêneas"),
    
    ("Sistema viÃ¡rio bÃ¡sico", 
     "Sistema viário básico"),
    
    ("Uso do solo: Ferraria", 
     "Uso do solo: Ferraria"),
    
    ("Uso do solo: Bateias", 
     "Uso do solo: Bateias"),
    
    ("ReferÃªncias", 
     "Referências"),
    
    ("Limpeza urbana", 
     "Limpeza urbana"),
    
    ("Rede de Ã¡gua", 
     "Rede de água"),
    
    ("Taxa de ocupaÃ§Ã£o por quadra", 
     "Taxa de ocupação por quadra"),
    
    ("Faixa de proteÃ§Ã£o de fundos de vale", 
     "Faixa de proteção de fundos de vale"),
    
    ("Densidade demogrÃ¡fica por lote", 
     "Densidade demográfica por lote"),
    
    ("Suporte natural", 
     "Suporte natural"),
    
    ("Levantamento hipsomÃ©trico", 
     "Levantamento hipsométrico"),
    
    ("Drenagem", 
     "Drenagem"),
    
    ("Represa PassaÃºna", 
     "Represa Passaúna"),
    
    ("Bacia do Rio Verde: padrÃµes e recomendaÃ§Ãµes de uso de solo", 
     "Bacia do Rio Verde: padrões e recomendações de uso de solo"),
    
    ("Bacia do Rio Verde: hidrografia", 
     "Bacia do Rio Verde: hidrografia"),
    
    ("Bacia do Rio Verde: hipsometria", 
     "Bacia do Rio Verde: hipsometria"),
    
    ("Mapa do MunicÃ­pio: levantamento Paranacidade", 
     "Mapa do Município: levantamento Paranacidade"),
    
    ("UTP Campo Magro: base", 
     "UTP Campo Magro: base"),
    
    ("UTP Campo Magro: geologia", 
     "UTP Campo Magro: geologia"),
    
    ("UTP Campo Magro: estudo de zoneamento", 
     "UTP Campo Magro: estudo de zoneamento"),
    
    ("UTP Campo Magro: hipsometria", 
     "UTP Campo Magro: hipsometria"),
    
    ("UTP Campo Magro: evoluÃ§Ã£o da ocupaÃ§Ã£o urbana", 
     "UTP Campo Magro: evolução da ocupação urbana"),
    
    ("UTP Campo Magro: uso solo", 
     "UTP Campo Magro: uso solo"),
    
    ("UTP Campo Magro: uso do solo atual", 
     "UTP Campo Magro: uso do solo atual"),
    
    ("UTP Campo Magro: manejo florestal", 
     "UTP Campo Magro: manejo florestal"),
    
    ("AnÃ¡lise de expansÃ£o urbana", 
     "Análise de expansão urbana"),
    
    ("Sede do municÃ­pio", 
     "Sede do município"),
    
    ("Planta da cidade", 
     "Planta da cidade"),
    
    ("PDM (Plano de Desenvolvimento Municipal) de Colombo", 
     "PDM (Plano de Desenvolvimento Municipal) de Colombo"),
    
    ("Origem e destinos de passageiros", 
     "Origem e destinos de passageiros"),
    
    ("Origem e destino de passageiros", 
     "Origem e destino de passageiros"),
    
    ("Rede elÃ©trica", 
     "Rede elétrica"),
    
    ("Galerias de Ã¡guas pluviais", 
     "Galerias de águas pluviais"),
    
    ("ComÃ©rcio e serviÃ§os", 
     "Comércio e serviços"),
    
    ("Cadastro", 
     "Cadastro"),
    
    ("Mapa de zoneamento do municÃ­pio de Contenda", 
     "Mapa de zoneamento do município de Contenda"),
    
    ("Zoneamento do uso do solo/sistema viÃ¡rio bÃ¡sico", 
     "Zoneamento do uso do solo/sistema viário básico"),
    
    ("Poligonal com curvas", 
     "Poligonal com curvas"),
    
    ("Planta da cidade de Contenda", 
     "Planta da cidade de Contenda"),
    
    ("Carta geolÃ³gica compilada", 
     "Carta geológica compilada"),
    
    ("BolsÃµes de carÃªncia", 
     "Bolsões de carência"),
    
    ("Mapa digital de arruamento", 
     "Mapa digital de arruamento"),
    
    ("Mapa digital Ã¡reas de subabitaÃ§Ã£o", 
     "Mapa digital áreas de subabitação"),
    
    ("Vilas de ofÃ­cio/liceus de ofÃ­cio", 
     "Vilas de ofício/liceus de ofício"),
    
    ("Ãreas de subabitaÃ§Ã£o", 
     "Áreas de subabitação"),
    
    ("OcupaÃ§Ãµes irregulares", 
     "Ocupações irregulares"),
    
    ("Sistema de unidades de conservaÃ§Ã£o", 
     "Sistema de unidades de conservação"),
    
    ("Planta planialtimÃ©trica: Ã¡rea de expansÃ£o", 
     "Planta planialtimétrica: área de expansão"),
    
    ("Potencialidades", 
     "Potencialidades"),
    
    ("Condicionantes", 
     "Condicionantes"),
    
    ("RestriÃ§Ãµes", 
     "Restrições"),
    
    ("LinhÃ£o do emprego", 
     "Linhão do emprego"),
    
    ("RestituiÃ§Ã£o aerofotogramÃ©trica: processo numÃ©rico", 
     "Restituição aerofotogramétrica: processo numérico"),
    
    ("EstratÃ©gia regional da regiÃ£o metropolitana em 1982 e subsistemas regionais", 
     "Estratégia regional da região metropolitana em 1982 e subsistemas regionais"),
    
    ("Estrutura urbana Curitiba e regiÃ£o metropolitana 1985", 
     "Estrutura urbana Curitiba e região metropolitana 1985"),
    
    ("Planta da colÃ´nia Argelina e e terrenos do governo federal", 
     "Planta da colônia Argelina e terrenos do governo federal"),
    
    ("Loteamentos e sistema viÃ¡rio", 
     "Loteamentos e sistema viário"),
    
    ("Ãrea  Aterro Sul", 
     "Área Aterro Sul"),
    
    ("IndÃºstrias e serviÃ§os pÃºblicos", 
     "Indústrias e serviços públicos"),
    
    ("Programa de transporte urbano", 
     "Programa de transporte urbano"),
    
    ("Arruamento do Centro CÃ­vico", 
     "Arruamento do Centro Cívico"),
    
    ("Planta planimÃ©trica semi - cadastral", 
     "Planta planimétrica semi - cadastral"),
    
    ("DivisÃ£o de bairros e arruamento", 
     "Divisão de bairros e arruamento"),
    
    ("Mapa fiscal e arruamento", 
     "Mapa fiscal e arruamento"),
    
    ("Zoneamento:espacializaÃ§Ã£o da legislaÃ§Ã£o municipal do uso do solo", 
     "Zoneamento: espacialização da legislação municipal do uso do solo"),
    
    ("Plano de desenvolvimento municipal de Fazenda Rio Grande - PDM", 
     "Plano de desenvolvimento municipal de Fazenda Rio Grande - PDM"),
    
    ("Proposta do sistema viÃ¡rio bÃ¡sico", 
     "Proposta do sistema viário básico"),
    
    ("Divisas", 
     "Divisas"),
    
    ("Planta do nÃºcleo urbano de ItaperuÃ§u", 
     "Planta do núcleo urbano de Itaperuçu"),
    
    ("Mapa rodoviÃ¡rio do municÃ­pio de Mandirituba", 
     "Mapa rodoviário do município de Mandirituba"),
    
    ("Planta planimÃ©trica da sede do municÃ­pio", 
     "Planta planimétrica da sede do município"),
    
    ("Planta do municÃ­pio: zoneamento de uso e ocupaÃ§Ã£o do solo urbano", 
     "Planta do município: zoneamento de uso e ocupação do solo urbano"),
    
    ("Planta do municÃ­pio: perÃ­metro urbano", 
     "Planta do município: perímetro urbano"),
    
    ("Base planialtimÃ©trica", 
     "Base planialtimétrica"),
    
    ("Zoneamento: sede", 
     "Zoneamento: sede"),
    
    ("Planta cadastral do distrito de Areia Branca dos Assis", 
     "Planta cadastral do distrito de Areia Branca dos Assis"),
    
    ("Ã€reas invadidas do municÃ­pio de Mandirituba", 
     "Áreas invadidas do município de Mandirituba"),
    
    ("Localidades de Pinhais e  V. Ma. Antonieta", 
     "Localidades de Pinhais e V. Ma. Antonieta"),
    
    ("Moradias Bonilauri", 
     "Moradias Bonilauri"),
    
    ("IntervenÃ§Ãµes propostas: Parque IguaÃ§u", 
     "Intervenções propostas: Parque Iguaçu"),
    
    ("UTP Guarituba: ocupaÃ§Ãµes irregulares", 
     "UTP Guarituba: ocupações irregulares"),
    
    ("UTP Guarituba: situaÃ§Ã£o de ocupaÃ§Ã£o/macrozoneamento", 
     "UTP Guarituba: situação de ocupação/macrozoneamento"),
    
    ("UTP Guarituba: lotes aprovados", 
     "UTP Guarituba: lotes aprovados"),
    
    ("UTP Guarituba: macro - drenagem", 
     "UTP Guarituba: macro - drenagem"),
    
    ("UTP Guarituba: maiores proprietÃ¡rios", 
     "UTP Guarituba: maiores proprietários"),
    
    ("EvoluÃ§Ã£o da ocupaÃ§Ã£o", 
     "Evolução da ocupação"),
    
    ("Guarituba: AISO Ãrea de Interesse Social de OcupaÃ§Ã£o", 
     "Guarituba: AISO Área de Interesse Social de Ocupação"),
    
    ("Ãreas verdes", 
     "Áreas verdes"),
    
    ("Zoneamento do municÃ­pio de Piraquara", 
     "Zoneamento do município de Piraquara"),
    
    ("Projeto Guarituba", 
     "Projeto Guarituba"),
    
    ("Fazenda Guarituba", 
     "Fazenda Guarituba"),
    
    ("Alternativas de inundaÃ§Ã£o do lago IraÃ­ e divisÃ£o das bacias", 
     "Alternativas de inundação do lago Iraí e divisão das bacias"),
    
    ("Cadastro de imÃ³veis do patrimÃ´nio estadual", 
     "Cadastro de imóveis do patrimônio estadual"),
    
    ("Plano de EstruturaÃ§Ã£o Municipal: uso do solo", 
     "Plano de Estruturação Municipal: uso do solo"),
    
    ("Estudos sobre Piraquara: suporte natural", 
     "Estudos sobre Piraquara: suporte natural"),
    
    ("Estudos sobre Piraquara: abastecimento de Ã¡gua e esgoto", 
     "Estudos sobre Piraquara: abastecimento de água e esgoto"),
    
    ("Estudos sobre Piraquara: sistema viÃ¡rio", 
     "Estudos sobre Piraquara: sistema viário"),
    
    ("Mapa identidade", 
     "Mapa identidade"),
    
    ("EsboÃ§o geolÃ³gico de Pinhais e Piraquara", 
     "Esboço geológico de Pinhais e Piraquara"),
    
    ("Usos potenciais", 
     "Usos potenciais"),
    
    ("UTP (unidade territorial de planejamento): mapa de vegetaÃ§Ã£o", 
     "UTP (unidade territorial de planejamento): mapa de vegetação"),
    
    ("UTP (unidade territorial de planejamento):restriÃ§Ãµes naturais para extraÃ§Ã£o mineral", 
     "UTP (unidade territorial de planejamento): restrições naturais para extração mineral"),
    
    ("UTP (unidade territorial de planejamento): geologia/declividades", 
     "UTP (unidade territorial de planejamento): geologia/declividades"),
    
    ("UTP (unidade territorial de planejamento): evoluÃ§Ã£o da ocupaÃ§Ã£o urbana", 
     "UTP (unidade territorial de planejamento): evolução da ocupação urbana"),
    
    ("Ãreas urbanas e Ã¡reas inundÃ¡veis", 
     "Áreas urbanas e áreas inundáveis"),
    
    ("Altimetria/rios", 
     "Altimetria/rios"),
    
    ("NÃºcleo urbano: arruamento", 
     "Núcleo urbano: arruamento"),
    
    ("Cobertura vegetal", 
     "Cobertura vegetal"),
    
    ("Mapeamento geolÃ³gico", 
     "Mapeamento geológico"),
    
    ("Levantamento de reconhecimento de grutas na RegiÃ£o Metropolitana de Curitiba", 
     "Levantamento de reconhecimento de grutas na Região Metropolitana de Curitiba"),
    
    ("Cidade de SÃ£o JosÃ© dos Pinhais", 
     "Cidade de São José dos Pinhais"),
    
    ("Cadastramento aerofotogramÃ©trico de SÃ£o JosÃ© dos Pinhais", 
     "Cadastramento aerofotogramétrico de São José dos Pinhais"),
    
    ("Rede de esgoto", 
     "Rede de esgoto"),
    
    ("Levantamento aerofotogramÃ©trico: represa do rio Miringuava", 
     "Levantamento aerofotogramétrico: represa do rio Miringuava"),
    
    ("Densidade urbana", 
     "Densidade urbana"),
    
    ("Equipamentos sociais e de saÃºde", 
     "Equipamentos sociais e de saúde"),
    
    ("Acervo cultural", 
     "Acervo cultural"),
    
    ("Rendimento mÃ©dio domiciliar", 
     "Rendimento médio domiciliar"),
    
    ("ServiÃ§os e utilidades pÃºblicas", 
     "Serviços e utilidades públicas"),
    
    ("Caminho do leite", 
     "Caminho do leite"),
    
    ("Rede de Ã¡gua: Ã¡reas com deficiÃªncia", 
     "Rede de água: áreas com deficiência"),
    
    ("Rede de esgoto: Ã¡reas com deficiÃªncia", 
     "Rede de esgoto: áreas com deficiência"),
    
    ("Rendimento mÃ©dio domiciliar abaixo do r.m.d. do municÃ­pio", 
     "Rendimento médio domiciliar abaixo do r.m.d. do município"),
    
    ("Rede elÃ©trica: Ã¡reas com deficiÃªncia", 
     "Rede elétrica: áreas com deficiência"),
    
    ("IndÃºstrias: distribuiÃ§Ã£o espacial", 
     "Indústrias: distribuição espacial"),
    
    ("Templos", 
     "Templos"),
    
    ("OcupaÃ§Ã£o por quadra", 
     "Ocupação por quadra"),
    
    ("Plano de desenvolvimento municipal de SÃ£o JosÃ© dos Pinhais: PDM", 
     "Plano de desenvolvimento municipal de São José dos Pinhais: PDM"),
    
    ("Lixo urbano", 
     "Lixo urbano"),
    
    ("Densidade urbana lÃ­quida", 
     "Densidade urbana líquida"),
    
    ("DistribuiÃ§Ã£o percentual de lotes vagos", 
     "Distribuição percentual de lotes vagos"),
    
    ("Tipologia do uso do solo", 
     "Tipologia do uso do solo"),
    
    ("Rede escolar", 
     "Rede escolar"),
    
    ("Aspectos naturais - sÃ­ntese", 
     "Aspectos naturais - síntese"),
    
    ("Plano de zoneamento de ruÃ­do: curvas isofÃ´nicas", 
     "Plano de zoneamento de ruído: curvas isofônicas"),
    
    ("Plano de zonas de proteÃ§Ã£o: cones de aproximaÃ§Ã£o", 
     "Plano de zonas de proteção: cones de aproximação"),
    
    ("Zoneamento:proposta", 
     "Zoneamento: proposta"),
    
    ("Unidade territorial de planejamento Itaqui: Ã¡reas verdes", 
     "Unidade territorial de planejamento Itaqui: áreas verdes"),
    
    ("Unidade territorial de planejamento Itaqui: loteamentos", 
     "Unidade territorial de planejamento Itaqui: loteamentos"),
    
    ("Distrito de Campo Largo da Roseira: AUDI/WOLKS", 
     "Distrito de Campo Largo da Roseira: AUDI/WOLKS"),
    
    ("Levantamento do uso e ocupaÃ§Ã£o do solo", 
     "Levantamento do uso e ocupação do solo"),
    
    ("Setores censitÃ¡rios (urbanos)", 
     "Setores censitários (urbanos)"),
    
    ("Distrito industrial de SÃ£o JosÃ© dos Pinhais: loteamentos aprovados e populaÃ§Ã£o", 
     "Distrito industrial de São José dos Pinhais: loteamentos aprovados e população"),
    
    ("Distrito industrial de SÃ£o JosÃ© dos Pinhais: zoneamento atual e ocupaÃ§Ã£o industrial", 
     "Distrito industrial de São José dos Pinhais: zoneamento atual e ocupação industrial"),
    
    ("Estudo de zoneamento", 
     "Estudo de zoneamento"),
    
    ("Distrito industrial de SÃ£o JosÃ© dos Pinhais: mapa de localizaÃ§Ã£o", 
     "Distrito industrial de São José dos Pinhais: mapa de localização"),
    
    ("NÃ­veis de cheias", 
     "Níveis de cheias"),
    
    ("LocalizaÃ§Ã£o de escolas, creches e postos de saÃºde", 
     "Localização de escolas, creches e postos de saúde"),
    
    ("Ralf's (reator anairÃ³bico de leito fluidizado) e rede de distribuiÃ§Ã£o", 
     "Ralf's (reator anaeróbico de leito fluidizado) e rede de distribuição"),
    
    ("Bacia do rio PassaÃºna: classes de declividade", 
     "Bacia do rio Passaúna: classes de declividade"),
    
    ("Programa perfil da cidade", 
     "Programa perfil da cidade"),
    
    ("EmancipaÃ§Ã£o do distrito de Tunas", 
     "Emancipação do distrito de Tunas"),
    
    ("Zoneamento de uso do solo", 
     "Zoneamento de uso do solo"),
    
    ("Distrito de Tunas: perÃ­metro urbano", 
     "Distrito de Tunas: perímetro urbano"),
    
    ("RestituiÃ§Ã£o aerofotogramÃ©trica: Distrito de Tunas", 
     "Restituição aerofotogramétrica: Distrito de Tunas"),
    
    ("Recursos hÃ­dricos", 
     "Recursos hídricos"),
    
    ("HabitaÃ§Ã£o social", 
     "Habitação social"),
    
    ("Desenvolvimento viÃ¡rio e de transporte", 
     "Desenvolvimento viário e de transporte"),
    
    ("Desenvolvimento econÃ´mico", 
     "Desenvolvimento econômico"),
    
    ("Desenvolvimento social", 
     "Desenvolvimento social"),
    
    ("Desenvolvimento fÃ­sico e territorial", 
     "Desenvolvimento físico e territorial"),
    
    ("Levantamento plani - altimÃ©trico dos rios: Isabel Alves, Rio das Almas e Rio do SebastiÃ£o", 
     "Levantamento plani - altimétrico dos rios: Isabel Alves, Rio das Almas e Rio do Sebastião"),
    
    ("Perfis de poligonais auxiliares e fundos dos rios: Isabel Alves, Rio das Almas e Rio do SebastiÃ£o", 
     "Perfis de poligonais auxiliares e fundos dos rios: Isabel Alves, Rio das Almas e Rio do Sebastião"),
    
    ("Canal extravasor do rio IguaÃ§u: trecho BR 277/Av. Torres", 
     "Canal extravasor do rio Iguaçu: trecho BR 277/Av. Torres"),
    
    ("LocalizaÃ§Ã£o de terminais metropolitanos (atual e futuro) e itinerÃ¡rio (linhas urbanas e metropolitanas)", 
     "Localização de terminais metropolitanos (atual e futuro) e itinerário (linhas urbanas e metropolitanas)"),
    
    ("UnificaÃ§Ã£o da Ã¡rea do distrito industrial de SÃ£o JosÃ© dos Pinhais: lei 03/96", 
     "Unificação da área do distrito industrial de São José dos Pinhais: lei 03/96"),
    
    ("Mapa hÃ­drico: faixas de proteÃ§Ã£o", 
     "Mapa hídrico: faixas de proteção"),
    
    ("Levantamento de conjuntos habitacionais e condiÃ§Ãµes de habitabilidade", 
     "Levantamento de conjuntos habitacionais e condições de habitabilidade"),
    
    ("LocalizaÃ§Ã£o de conjuntos habitacionais e favelas", 
     "Localização de conjuntos habitacionais e favelas"),
    
    ("Mapa base:municÃ­pios limÃ­trofes", 
     "Mapa base: municípios limítrofes"),
    
    ("Setores censitÃ¡rios IBGE: parte 1", 
     "Setores censitários IBGE: parte 1"),
    
    ("Setores censitÃ¡rios IBGE: parte 2", 
     "Setores censitários IBGE: parte 2"),
    
    ("RMC e litoral (parte 1): sistema viÃ¡rio metropolitano", 
     "RMC e litoral (parte 1): sistema viário metropolitano"),
    
    ("RMC e litoral (parte 2): sistema viÃ¡rio metropolitano", 
     "RMC e litoral (parte 2): sistema viário metropolitano"),
    
    ("ArticulaÃ§Ã£o das cartas", 
     "Articulação das cartas"),
    
    ("RegiÃ£o Metropolitana de Curitiba", 
     "Região Metropolitana de Curitiba"),
    
    ("ProteÃ§Ã£o dos mananciais hÃ­dricos (parte 1): anexo ao decreto nÃºmero 2964/80", 
     "Proteção dos mananciais hídricos (parte 1): anexo ao decreto número 2964/80"),
    
    ("ProteÃ§Ã£o dos mananciais hÃ­dricos (parte 2): anexo ao decreto nÃºmero 1751/96", 
     "Proteção dos mananciais hídricos (parte 2): anexo ao decreto número 1751/96"),
    
    ("Mananciais hÃ­dricos: mapa anexo ao decreto Estadual nÃºmero 6390 de 05/04/2006", 
     "Mananciais hídricos: mapa anexo ao decreto Estadual número 6390 de 05/04/2006"),
    
    ("Esquema de apoio bÃ¡sico e suplementar vertical", 
     "Esquema de apoio básico e suplementar vertical"),
    
    ("Esquema de apoio bÃ¡sico e suplementar horizontal para aerotriangulaÃ§Ã£o", 
     "Esquema de apoio básico e suplementar horizontal para aerotriangulação"),
    
    ("Contorno leste", 
     "Contorno leste"),
    
    ("Contorno norte", 
     "Contorno norte"),
    
    ("Base cartogrÃ¡fica para controle da ocupaÃ§Ã£o territorial - COT", 
     "Base cartográfica para controle da ocupação territorial - COT"),
    
    ("Programa metropolitano de reservatÃ³rios de emergÃªncia", 
     "Programa metropolitano de reservatórios de emergência"),
    
    ("Perfil transversal ao Rio IraÃ­", 
     "Perfil transversal ao Rio Iraí"),
    
    ("IluminaÃ§Ã£o pÃºblica: pesquisa COMEC 1985", 
     "Iluminação pública: pesquisa COMEC 1985"),
    
    ("Equipamentos de comÃ©rcio e serviÃ§os", 
     "Equipamentos de comércio e serviços"),
    
    ("ComunicaÃ§Ãµes", 
     "Comunicações"),
    
    ("Equipamentos diversos de utilidade pÃºblica, comÃ©rcio e serviÃ§os", 
     "Equipamentos diversos de utilidade pública, comércio e serviços"),
    
    ("Equipamentos diversos de utilidade pÃºblica", 
     "Equipamentos diversos de utilidade pública"),
    
    ("Equipamento de educaÃ§Ã£o, saÃºde e assistÃªncia social", 
     "Equipamento de educação, saúde e assistência social"),
    
    ("Projeto Curitiba: Carta de densidade de lineaÃ§Ãµes em sÃ©rie", 
     "Projeto Curitiba: Carta de densidade de lineações em série"),
    
    ("Projeto Curitiba: Carta de densidade de drenagem", 
     "Projeto Curitiba: Carta de densidade de drenagem"),
    
    ("Projeto leste do ParanÃ¡: folha Palmeira", 
     "Projeto leste do Paraná: folha Palmeira"),
    
    ("Projeto Curitiba: Carta de localizaÃ§Ã£o de formas cÃ¡rsticas (dolinas e sumidouros de drenagem)", 
     "Projeto Curitiba: Carta de localização de formas cársticas (dolinas e sumidouros de drenagem)"),
    
    ("Projeto Curitiba: Carta morfoestrutural", 
     "Projeto Curitiba: Carta morfoestrutural"),
    
    ("Projeto Curitiba: Cartas de padrÃµes de relevo", 
     "Projeto Curitiba: Cartas de padrões de relevo"),
    
    ("Projeto Curitiba: Carta de grau de estruturaÃ§Ã£o das lineaÃ§Ãµes em feixe (tropia)", 
     "Projeto Curitiba: Carta de grau de estruturação das lineações em feixe (tropia)"),
    
    ("Projeto Curitiba: Carta morfolitostrutural e fragilidade do meio fÃ­sico", 
     "Projeto Curitiba: Carta morfolitostrutural e fragilidade do meio físico"),
    
    ("Projeto Curitiba: Carta de uso e ocupaÃ§Ã£o do solo", 
     "Projeto Curitiba: Carta de uso e ocupação do solo"),
    
    ("Projeto Curitiba: Ãreas naturais sob proteÃ§Ã£o na regiÃ£o metropolitana de Curitiba", 
     "Projeto Curitiba: Áreas naturais sob proteção na região metropolitana de Curitiba"),
    
    ("Projeto leste do ParanÃ¡: folha PiraÃ­ do Sul", 
     "Projeto leste do Paraná: folha Piraí do Sul"),
    
    ("Projeto leste do ParanÃ¡: folha Cerro Azul", 
     "Projeto leste do Paraná: folha Cerro Azul"),
    
    ("Projeto leste do ParanÃ¡: folha ApiaÃ­", 
     "Projeto leste do Paraná: folha Apiaí"),
    
    ("Projeto leste do ParanÃ¡: folha Ponta Grossa", 
     "Projeto leste do Paraná: folha Ponta Grossa"),
    
    ("Projeto leste do ParanÃ¡: folha Campo Largo", 
     "Projeto leste do Paraná: folha Campo Largo"),
    
    ("Projeto leste do ParanÃ¡: folha Curitiba", 
     "Projeto leste do Paraná: folha Curitiba"),
    
    ("Projeto leste do ParanÃ¡: folha GuaraqueÃ§aba", 
     "Projeto leste do Paraná: folha Guaraqueçaba"),
    
    ("Projeto leste do ParanÃ¡: folha Barra do rio PitanguÃ­", 
     "Projeto leste do Paraná: folha Barra do rio Pitanguí"),
    
    ("Dados bÃ¡sicos da RMC: hipsometria", 
     "Dados básicos da RMC: hipsometria"),
    
    ("Dados bÃ¡sicos da RMC: geologia", 
     "Dados básicos da RMC: geologia"),
    
    ("LocalizaÃ§Ã£o de indÃºstrias e poluiÃ§Ã£o hÃ­drica por metais pesados", 
     "Localização de indústrias e poluição hídrica por metais pesados"),
    
    ("Dados bÃ¡sicos da RMC: solos", 
     "Dados básicos da RMC: solos"),
    
    ("ClassificaÃ§Ã£o dos rios", 
     "Classificação dos rios"),
    
    ("Rios fora da classe", 
     "Rios fora da classe"),
    
    ("Bacias hidrogrÃ¡ficas: situaÃ§Ã£o atual", 
     "Bacias hidrográficas: situação atual"),
    
    ("OrganizaÃ§Ã£o Territorial - Diretrizes BÃ¡sicas: Ãreas indÃºstriais", 
     "Organização Territorial - Diretrizes Básicas: Áreas industriais"),
    
    ("Ventos dominantes", 
     "Ventos dominantes"),
    
    ("Despejo de resÃ­duos sÃ³lidos", 
     "Despejo de resíduos sólidos"),
    
    ("APA Estadual PassaÃºna: zoneamento", 
     "APA Estadual Passaúna: zoneamento"),
    
    ("LocalizaÃ§Ã£o de indÃºstrias e poluiÃ§Ã£o atmosfÃ©rica", 
     "Localização de indústrias e poluição atmosférica"),
    
    ("PoluiÃ§Ã£o atmosfÃ©rica: situaÃ§Ã£o atual", 
     "Poluição atmosférica: situação atual"),
    
    ("Mapa hidrolÃ³gico e do sistema viÃ¡rio bÃ¡sico da RMC", 
     "Mapa hidrológico e do sistema viário básico da RMC"),
    
    ("Macro bacias hidrogrÃ¡ficas da RMC", 
     "Macro bacias hidrográficas da RMC"),
    
    ("CompartimentaÃ§Ã£o fisiogrÃ¡fica preliminar: articulaÃ§Ã£o", 
     "Compartimentação fisiográfica preliminar: articulação"),
    
    ("CompartimentaÃ§Ã£o fisiogrÃ¡fica preliminar: compartimentos fisiogrÃ¡ficos", 
     "Compartimentação fisiográfica preliminar: compartimentos fisiográficos"),
    
    ("Sistema viÃ¡rio bÃ¡sico da RegiÃ£o Metropolitana de Curitiba: rede analÃ­tica geral considerada no carregamento de viagens", 
     "Sistema viário básico da Região Metropolitana de Curitiba: rede analítica geral considerada no carregamento de viagens"),
    
    ("Sistema viÃ¡rio bÃ¡sico da RegiÃ£o Metropolitana de Curitiba: detalhe da rede analÃ­tica considerada no carregamento de viagens", 
     "Sistema viário básico da Região Metropolitana de Curitiba: detalhe da rede analítica considerada no carregamento de viagens"),
    
    ("Qualidade das Ã¡guas superficiais na RMC", 
     "Qualidade das águas superficiais na RMC"),
    
    ("Estudos de concepÃ§Ã£o de aproveitamentos futuros de mananciais", 
     "Estudos de concepção de aproveitamentos futuros de mananciais"),
    
    ("Ãrea de abrangÃªncia da enchente de janeiro de 1995", 
     "Área de abrangência da enchente de janeiro de 1995"),
    
    ("Mapa geotÃ©cnico geral", 
     "Mapa geotécnico geral"),
    
    ("Monitoramento da qualidade da Ã¡gua/Rios da RMC: toxicidade aguda para Daphnia magna - perÃ­odo: jul/93 a jul/97", 
     "Monitoramento da qualidade da água/Rios da RMC: toxicidade aguda para Daphnia magna - período: jul/93 a jul/97"),
    
    ("Monitoramento da qualidade da Ã¡gua/Rios da RMC: parÃ¢metros fÃ­sico - quÃ­micos", 
     "Monitoramento da qualidade da água/Rios da RMC: parâmetros físico - químicos"),
    
    ("Unidades de conservaÃ§Ã£o da RMC, serra do mar e litoral", 
     "Unidades de conservação da RMC, serra do mar e litoral"),
    
    ("Distrito mineiro - industrial da cerÃ¢mica vermelha", 
     "Distrito mineiro - industrial da cerâmica vermelha"),
    
    ("PopulaÃ§Ã£o de 7 a 14 anos por setor censitÃ¡rio na RMC (parte1)", 
     "População de 7 a 14 anos por setor censitário na RMC (parte1)"),
    
    ("PopulaÃ§Ã£o de 7 a 14 anos por setor censitÃ¡rio na RMC (parte 2)", 
     "População de 7 a 14 anos por setor censitário na RMC (parte 2)"),
    
    ("GrÃ¡fico de demanda convergente", 
     "Gráfico de demanda convergente"),
    
    ("Caderno de cobertura cartogrÃ¡fica", 
     "Caderno de cobertura cartográfica"),
    
    ("sistema viÃ¡rioviarodoviaitinerÃ¡rioÃ´nibuspavimentaÃ§Ã£o", 
     "sistema viário via rodovia itinerário ônibus pavimentação"),  # Caso estranho, tentamos separar
    
    ("Zoneamento ecolÃ³gico-econÃ´mico da APA do PassaÃºna", 
     "Zoneamento ecológico-econômico da APA do Passaúna"),
    
    ("Geomorfologia da Ã¡rea de vÃ¡rzea do Alto IguaÃ§u", 
     "Geomorfologia da área de várzea do Alto Iguaçu"),
    
    ("Mapa de adequabilidade para loteamentos residenciais", 
     "Mapa de adequabilidade para loteamentos residenciais"),
    
    ("Mapa de documentaÃ§Ã£o 01", 
     "Mapa de documentação 01"),
    
    ("Mapa de documentaÃ§Ã£o 02", 
     "Mapa de documentação 02"),
    
    ("Mapa de erodibilidade", 
     "Mapa de erodibilidade"),
    
    ("Mapa de materiais inconsolidados", 
     "Mapa de materiais inconsolidados"),
    
    ("Mapa de profundidade do lenÃ§ol freÃ¡tico", 
     "Mapa de profundidade do lençol freático"),
    
    ("Mapa de cadastramento de atividade mineral, pontos de poluiÃ§Ã£o e lenÃ§ol freÃ¡tico", 
     "Mapa de cadastramento de atividade mineral, pontos de poluição e lençol freático"),
    
    ("Mapa de zoneamento geotÃ©cnico", 
     "Mapa de zoneamento geotécnico"),
    
    ("Mapa de Ã¡reas inundÃ¡veis", 
     "Mapa de áreas inundáveis"),
    
    ("Perfil longitudinal das unidades de terreno UT.A", 
     "Perfil longitudinal das unidades de terreno UT.A"),
    
    ("Planta geolÃ³gica da cidade de Curitiba", 
     "Planta geológica da cidade de Curitiba"),
    
    ("Planta geolÃ³gica de partes dos municÃ­pios de Rio Branco do Sul, Almirante TamandarÃ© e Colombo", 
     "Planta geológica de partes dos municípios de Rio Branco do Sul, Almirante Tamandaré e Colombo"),
    
    ("Mapa geolÃ³gico compilado", 
     "Mapa geológico compilado"),
    
    ("Geologia e mineraÃ§Ã£o", 
     "Geologia e mineração"),
    
    ("Recursos naturais - sÃ­ntese", 
     "Recursos naturais - síntese"),
    
    ("Uso e ocupaÃ§Ã£o do solo da Ã¡rea de proteÃ§Ã£o de Guaratuba", 
     "Uso e ocupação do solo da área de proteção de Guaratuba"),
    
    ("Ãrea de proteÃ§Ã£o ambiental do Marumbi", 
     "Área de proteção ambiental do Marumbi"),
    
    ("APA Estadual do Pequeno: decreto nÃºmero 1752/96", 
     "APA Estadual do Pequeno: decreto número 1752/96"),
    
    ("APA Estadual do Piraquara: decreto nÃºmero 1754/96", 
     "APA Estadual do Piraquara: decreto número 1754/96"),
    
    ("APA Estadual do IraÃ­: decreto nÃºmero 1753/96", 
     "APA Estadual do Iraí: decreto número 1753/96"),
    
    ("Zoneamento das APA's e UTP's", 
     "Zoneamento das APA's e UTP's"),
    
    ("Zoneamento da APA do Piraquara", 
     "Zoneamento da APA do Piraquara"),
    
    ("Zoneamento ecolÃ³gico-econÃ´mico da APA do IraÃ­", 
     "Zoneamento ecológico-econômico da APA do Iraí"),
    
    ("Levantamento aerofotogramÃ©trico da APA do PassaÃºna", 
     "Levantamento aerofotogramétrico da APA do Passaúna"),
    
    ("PassaÃºna - declividade: determinaÃ§Ã£o de Ã¡reas de risco", 
     "Passaúna - declividade: determinação de áreas de risco"),
    
    ("PassaÃºna: divisor de Ã¡guas", 
     "Passaúna: divisor de águas"),
    
    ("PassaÃºna: loteamentos aprovados e Ã¡reas ocupadas", 
     "Passaúna: loteamentos aprovados e áreas ocupadas"),
    
    ("PassaÃºna: nÃ­veis de renda familiar", 
     "Passaúna: níveis de renda familiar"),
    
    ("Bacia do rio PassaÃºna: mapa base/reservatÃ³rio", 
     "Bacia do rio Passaúna: mapa base/reservatório"),
    
    ("PassaÃºna: hipsometria", 
     "Passaúna: hipsometria"),
    
    ("PassaÃºna: uso do solo", 
     "Passaúna: uso do solo"),
    
    ("PassaÃºna: relevo e drenagem superficial", 
     "Passaúna: relevo e drenagem superficial"),
    
    ("PassaÃºna: estudo grÃ¡fico", 
     "Passaúna: estudo gráfico"),
    
    ("PassaÃºna: estudo para o zoneamento", 
     "Passaúna: estudo para o zoneamento"),
    
    ("PassaÃºna: limites municipais", 
     "Passaúna: limites municipais"),
    
    ("PassaÃºna: sucetibilidade Ã  erosÃ£o", 
     "Passaúna: suscetibilidade à erosão"),
    
    ("PassaÃºna: zoneamento-minuta", 
     "Passaúna: zoneamento-minuta"),
    
    ("Projeto reservatÃ³rio Rio PassaÃºna", 
     "Projeto reservatório Rio Passaúna"),
    
    ("Bacia do rio PassaÃºna: mapa base/hidrografia", 
     "Bacia do rio Passaúna: mapa base/hidrografia"),
    
    ("Bacia do rio PassaÃºna: geologia", 
     "Bacia do rio Passaúna: geologia"),
    
    ("Bacia do rio PassaÃºna: cobertura vegetal", 
     "Bacia do rio Passaúna: cobertura vegetal"),
    
    ("Bacia do rio PassaÃºna: sÃ­ntese da cobertura vegetal", 
     "Bacia do rio Passaúna: síntese da cobertura vegetal"),
    
    ("Bacia do rio PassaÃºna: divisores de Ã¡gua", 
     "Bacia do rio Passaúna: divisores de água"),
    
    ("Bacia do rio PassaÃºna: linhas de transporte", 
     "Bacia do rio Passaúna: linhas de transporte"),
    
    ("Bacia do rio PassaÃºna: sistema viÃ¡rio, ferrovias e linhas de alta tensÃ£o", 
     "Bacia do rio Passaúna: sistema viário, ferrovias e linhas de alta tensão"),
    
    ("Bacia do rio PassaÃºna: zoneamento do uso do solo", 
     "Bacia do rio Passaúna: zoneamento do uso do solo"),
    
    ("Bacia do rio PassaÃºna: ocupaÃ§Ã£o do solo em 1980", 
     "Bacia do rio Passaúna: ocupação do solo em 1980"),
    
    ("Bacia do rio PassaÃºna: ocupaÃ§Ã£o do solo em 1985", 
     "Bacia do rio Passaúna: ocupação do solo em 1985"),
    
    ("Loteamento na bacia do PassaÃºna no municÃ­pio de Campo Largo", 
     "Loteamento na bacia do Passaúna no município de Campo Largo"),
    
    ("EstaÃ§Ãµes hidromÃ©tricas", 
     "Estações hidrométricas"),
    
    ("Sub-bacia do rio PassaÃºna", 
     "Sub-bacia do rio Passaúna"),
    
    ("PerÃ­metro do zoneamento geolÃ³gico-econÃ´mico da Ãrea de PreservaÃ§Ã£o Ambiental Estadual do PassaÃºna", 
     "Perímetro do zoneamento geológico-econômico da Área de Preservação Ambiental Estadual do Passaúna"),
    
    ("Atividade minerÃ¡ria na bacia do PassaÃºna", 
     "Atividade minerária na bacia do Passaúna"),
    
    ("Projeto reservatÃ³rio: desapropriaÃ§Ãµes/estrutura fundiÃ¡ria", 
     "Projeto reservatório: desapropriações/estrutura fundiária"),
    
    ("Bacia do PassaÃºna: vegetaÃ§Ã£o e ocupaÃ§Ã£o", 
     "Bacia do Passaúna: vegetação e ocupação"),
    
    ("Bacia do PassaÃºna: pontos crÃ­ticos", 
     "Bacia do Passaúna: pontos críticos"),
    
    ("ReivindicaÃ§Ãµes", 
     "Reivindicações"),
    
    ("FormaÃ§Ã£o superficial", 
     "Formação superficial"),
    
    ("EsboÃ§o geolÃ³gico", 
     "Esboço geológico"),
    
    ("Zoneamento de Ã¡reas quanto a susceptibilidade Ã  erosÃ£o", 
     "Zoneamento de áreas quanto a susceptibilidade à erosão"),
    
    ("Zoneamento de Ã¡reas quanto Ã  suscetibilidade Ã  escorregamentos", 
     "Zoneamento de áreas quanto à suscetibilidade a escorregamentos"),
    
    ("Equipamentos urbanos", 
     "Equipamentos urbanos"),
    
    ("Infra-estrutura", 
     "Infra-estrutura"),
    
    ("APA Estadual do PassaÃºna: zoneamento ecolÃ³gico-econÃ´mico", 
     "APA Estadual do Passaúna: zoneamento ecológico-econômico"),
    
    ("APA Estadual do PassaÃºna: classes de aptidÃ£o do solo e capacidade de uso", 
     "APA Estadual do Passaúna: classes de aptidão do solo e capacidade de uso"),
    
    ("APA Estadual do PassaÃºna: conflitos", 
     "APA Estadual do Passaúna: conflitos"),
    
    ("APA Estadual do PassaÃºna: diretrizes", 
     "APA Estadual do Passaúna: diretrizes"),
    
    ("Setores censitÃ¡rios da bacia hidrogrÃ¡fica do rio Palmital", 
     "Setores censitários da bacia hidrográfica do rio Palmital"),
    
    ("Cartograma da rede de drenagem do rio Palmital - com expansÃ£o antrÃ³pica do ano de 1976 e 1980", 
     "Cartograma da rede de drenagem do rio Palmital - com expansão antrópica do ano de 1976 e 1980"),
    
    ("Cartograma de vegetaÃ§Ã£o da bacia hidrogrÃ¡fica do rio Palmital do ano de 1976", 
     "Cartograma de vegetação da bacia hidrográfica do rio Palmital do ano de 1976"),
    
    ("Plano Diretor de Manejo Florestal: florestal", 
     "Plano Diretor de Manejo Florestal: florestal"),
    
    ("Plano Diretor de Manejo Florestal: bacias hidrogrÃ¡ficas", 
     "Plano Diretor de Manejo Florestal: bacias hidrográficas"),
    
    ("Plano Diretor de Manejo Florestal: zonas de uso predominantemente industrial - ZUPI", 
     "Plano Diretor de Manejo Florestal: zonas de uso predominantemente industrial - ZUPI"),
    
    ("Plano Diretor de Manejo Florestal: unidades de conservaÃ§Ã£o", 
     "Plano Diretor de Manejo Florestal: unidades de conservação"),
    
    ("Plano Diretor de Manejo Florestal: cobertura florestal", 
     "Plano Diretor de Manejo Florestal: cobertura florestal"),
    
    ("Plano Diretor de Manejo Florestal: estrutura fundiÃ¡ria", 
     "Plano Diretor de Manejo Florestal: estrutura fundiária"),
    
    ("Plano Diretor de Manejo Florestal: abastecimento de Ã¡gua", 
     "Plano Diretor de Manejo Florestal: abastecimento de água"),
    
    ("Plano Diretor de Manejo Florestal: subsistemas", 
     "Plano Diretor de Manejo Florestal: subsistemas"),
    
    ("Plano Diretor de Manejo Florestal: sistema viÃ¡rio bÃ¡sico regional", 
     "Plano Diretor de Manejo Florestal: sistema viário básico regional"),
    
    ("Plano Diretor de Manejo Florestal: sedes municipais", 
     "Plano Diretor de Manejo Florestal: sedes municipais"),
    
    ("Plano Diretor de Manejo Florestal: macro compartimentaÃ§Ã£o urbana", 
     "Plano Diretor de Manejo Florestal: macro compartimentação urbana"),
    
    ("Plano Diretor de Manejo Florestal: remanescentes florestais", 
     "Plano Diretor de Manejo Florestal: remanescentes florestais"),
    
    ("Plano Diretor de Manejo Florestal: situaÃ§Ã£o", 
     "Plano Diretor de Manejo Florestal: situação"),
    
    ("Plano Diretor de Manejo Florestal: proposta", 
     "Plano Diretor de Manejo Florestal: proposta"),
    
    ("Plano de EstruturaÃ§Ã£o FÃ­sica: corredor sul - Curitiba", 
     "Plano de Estruturação Física: corredor sul - Curitiba"),
    
    ("Plano de EstruturaÃ§Ã£o FÃ­sica: corredor sul - Fazenda Rio Grande", 
     "Plano de Estruturação Física: corredor sul - Fazenda Rio Grande"),
    
    ("Plano de EstruturaÃ§Ã£o FÃ­sica: corredor sudeste - AraucÃ¡ria", 
     "Plano de Estruturação Física: corredor sudeste - Araucária"),
    
    ("Plano de EstruturaÃ§Ã£o FÃ­sica: Ã¡rea de urbanizaÃ§Ã£o contÃ­nua", 
     "Plano de Estruturação Física: área de urbanização contínua"),
    
    ("Centro de Apoio Ã  CrianÃ§a - CIAC: Almirante TamandarÃ©", 
     "Centro de Apoio à Criança - CIAC: Almirante Tamandaré"),
    
    ("Centro de Apoio Ã  CrianÃ§a - CIAC: AraucÃ¡ria", 
     "Centro de Apoio à Criança - CIAC: Araucária"),
    
    ("Centro de Apoio Ã  CrianÃ§a - CIAC: Campina Grande do Sul (Quatro Barras)", 
     "Centro de Apoio à Criança - CIAC: Campina Grande do Sul (Quatro Barras)"),
    
    ("Centro de Apoio Ã  CrianÃ§a - CIAC: Campo Largo", 
     "Centro de Apoio à Criança - CIAC: Campo Largo"),
    
    ("Centro de Apoio Ã  CrianÃ§a - CIAC: Colombo", 
     "Centro de Apoio à Criança - CIAC: Colombo"),
    
    ("Centro de Apoio Ã  CrianÃ§a - CIAC: Fazenda Rio Grande", 
     "Centro de Apoio à Criança - CIAC: Fazenda Rio Grande"),
    
    ("Centro de Apoio Ã  CrianÃ§a - CIAC: Piraquara", 
     "Centro de Apoio à Criança - CIAC: Piraquara"),
    
    ("Centro de Apoio Ã  CrianÃ§a - CIAC: SÃ£o JosÃ© dos Pinhais", 
     "Centro de Apoio à Criança - CIAC: São José dos Pinhais"),
    
    ("Ãreas a desapropriar", 
     "Áreas a desapropriar"),
    
    ("Detalhe", 
     "Detalhe"),
    
    ("Sistema viÃ¡rio - perfis", 
     "Sistema viário - perfis"),
    
    ("Ãreas de preservaÃ§Ã£o", 
     "Áreas de preservação"),
    
    ("LocalizaÃ§Ã£o das propostas: bombeiros", 
     "Localização das propostas: bombeiros"),
    
    ("LocalizaÃ§Ã£o das propostas: saÃºde", 
     "Localização das propostas: saúde"),
    
    ("LocalizaÃ§Ã£o das propostas: creches", 
     "Localização das propostas: creches"),
    
    ("LocalizaÃ§Ã£o das propostas: educaÃ§Ã£o", 
     "Localização das propostas: educação"),
    
    ("EspaÃ§o regional", 
     "Espaço regional"),
    
    ("EvoluÃ§Ã£o da ocupaÃ§Ã£o urbana", 
     "Evolução da ocupação urbana"),
    
    ("Ãreas prioritÃ¡rias de atuaÃ§Ã£o", 
     "Áreas prioritárias de atuação"),
    
    ("Abastecimento alimentar", 
     "Abastecimento alimentar"),
    
    ("SÃ­ntese de propostas: equipamentos de saÃºde", 
     "Síntese de propostas: equipamentos de saúde"),
    
    ("HabitaÃ§Ã£o e urbanismo", 
     "Habitação e urbanismo"),
    
    ("SeguranÃ§a pÃºblica - descentralizaÃ§Ã£o do corpo de bombeiros", 
     "Segurança pública - descentralização do corpo de bombeiros"),
    
    ("Mobilidade e acessibilidade", 
     "Mobilidade e acessibilidade"),
    
    ("Institucional", 
     "Institucional"),
    
    ("PopulaÃ§Ã£o", 
     "População"),
    
    ("Rede pÃºblica de serviÃ§o de saÃºde", 
     "Rede pública de serviço de saúde"),
    
    ("Equipamentos comunitÃ¡rios", 
     "Equipamentos comunitários"),
    
    ("LocalizaÃ§Ã£o das propostas: projetos em tramitaÃ§Ã£o junto a CEF/1988", 
     "Localização das propostas: projetos em tramitação junto a CEF/1988"),
    
    ("Sistema viÃ¡rio: nov/1991", 
     "Sistema viário: nov/1991"),
    
    ("Sistema viÃ¡rio: sistema de transporte pÃºblico de passageiros", 
     "Sistema viário: sistema de transporte público de passageiros"),
    
    ("LocalizaÃ§Ã£o das propostas: projetos em transiÃ§Ã£o junto a C.E.F./1988", 
     "Localização das propostas: projetos em transição junto a C.E.F./1988"),
    
    ("LocalizaÃ§Ã£o das propostas", 
     "Localização das propostas"),
    
    ("Limites municipais", 
     "Limites municipais"),
    
    ("OcupaÃ§Ã£o urbana em 1965", 
     "Ocupação urbana em 1965"),
    
    ("OcupaÃ§Ã£o urbana em 1975.", 
     "Ocupação urbana em 1975."),
    
    ("OcupaÃ§Ã£o urbana em 1980", 
     "Ocupação urbana em 1980"),
    
    ("OcupaÃ§Ã£o urbana em", 
     "Ocupação urbana em"),
    
    ("Demografia", 
     "Demografia"),
    
    ("Cultura, esporte e recreaÃ§Ã£o", 
     "Cultura, esporte e recreação"),
    
    ("DistribuiÃ§Ã£o do nÃºmero de empresas e linhas por municÃ­pio.", 
     "Distribuição do número de empresas e linhas por município."),
    
    ("Passageiros transportados mensalmente por municÃ­pio", 
     "Passageiros transportados mensalmente por município"),
    
    ("Hidrografia e ?", 
     "Hidrografia e ?"),  # mantém interrogação se for o caso
    
    ("EstruturaÃ§Ã£o urbana metropolitana", 
     "Estruturação urbana metropolitana"),
    
    ("Conflitos/problemas", 
     "Conflitos/problemas"),
    
    ("ResÃ­duo sÃ³lido", 
     "Resíduo sólido"),
    
    ("Problemas do transporte pÃºblico", 
     "Problemas do transporte público"),
    
    ("Principais problemas", 
     "Principais problemas"),
    
    ("Densidade populacional", 
     "Densidade populacional"),
    
    ("Sistema de transportes coletivo de passageiros", 
     "Sistema de transportes coletivo de passageiros"),
    
    ("Climatologia", 
     "Climatologia"),
    
    ("NÃ­veis de relaÃ§Ãµes entre: metrÃ³pole e regiÃµes administrativas 2 E", 
     "Níveis de relações entre: metrópole e regiões administrativas 2 E"),
    
    ("Mapa base: quadrÃ­culas de 1km", 
     "Mapa base: quadrículas de 1km"),
    
    ("Lay - out geral do sistema", 
     "Lay - out geral do sistema"),
    
    ("Ãreas invadidas na RMC", 
     "Áreas invadidas na RMC"),
    
    ("Suporte natural, infra-estrutura e Ã¡reas invadidas", 
     "Suporte natural, infra-estrutura e áreas invadidas"),
    
    ("PreÃ§o de propriedade mÂ² de lote padrÃ£o", 
     "Preço de propriedade m² de lote padrão"),
    
    ("Suporte natural, infra-estrutura e uso do solo", 
     "Suporte natural, infra-estrutura e uso do solo"),
    
    ("Eixo leste: situaÃ§Ã£o - inclinaÃ§Ã£o das encostas", 
     "Eixo leste: situação - inclinação das encostas"),
    
    ("Levantamento de ocupaÃ§Ãµes irregulares", 
     "Levantamento de ocupações irregulares"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 45", 
     "Seção topobatimétrica - 45"),
    
    ("Ãreas invadidas, suporte natural, infra-estrutura, uso do solo e proposta", 
     "Áreas invadidas, suporte natural, infra-estrutura, uso do solo e proposta"),
    
    ("Ãreas invadidas, suporte natural e infra-estrutura", 
     "Áreas invadidas, suporte natural e infra-estrutura"),
    
    ("Eixo leste: situaÃ§Ã£o (mapa base)", 
     "Eixo leste: situação (mapa base)"),
    
    ("Eixo leste: situaÃ§Ã£o", 
     "Eixo leste: situação"),
    
    ("SituaÃ§Ã£o: regiÃµes geogrÃ¡ficas naturais", 
     "Situação: regiões geográficas naturais"),
    
    ("SituaÃ§Ã£o: mapa geolÃ³gico", 
     "Situação: mapa geológico"),
    
    ("SituaÃ§Ã£o: caudal subterrÃ¢neo", 
     "Situação: caudal subterrâneo"),
    
    ("SituaÃ§Ã£o: mapa fitogeogrÃ¡fico", 
     "Situação: mapa fitogeográfico"),
    
    ("SituaÃ§Ã£o: carta florestal", 
     "Situação: carta florestal"),
    
    ("SituaÃ§Ã£o: sistema elÃ©trico", 
     "Situação: sistema elétrico"),
    
    ("SituaÃ§Ã£o: sistema rodoferroviÃ¡rio e aeroviÃ¡rio", 
     "Situação: sistema rodoferroviário e aeroviário"),
    
    ("Eixo leste: proposta - diretrizes para o desenvolvimento", 
     "Eixo leste: proposta - diretrizes para o desenvolvimento"),
    
    ("Eixo leste: proposta - proteÃ§Ã£o de bacias", 
     "Eixo leste: proposta - proteção de bacias"),
    
    ("Eixo leste: situaÃ§Ã£o - energia elÃ©trica e telecomunicaÃ§Ãµes", 
     "Eixo leste: situação - energia elétrica e telecomunicações"),
    
    ("ArticulaÃ§Ã£o das cartas 1:50.000", 
     "Articulação das cartas 1:50.000"),
    
    ("Planta de situaÃ§Ã£o", 
     "Planta de situação"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 01", 
     "Seção topobatimétrica - 01"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 02", 
     "Seção topobatimétrica - 02"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 03", 
     "Seção topobatimétrica - 03"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 04", 
     "Seção topobatimétrica - 04"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 05", 
     "Seção topobatimétrica - 05"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 06", 
     "Seção topobatimétrica - 06"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 07", 
     "Seção topobatimétrica - 07"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 08", 
     "Seção topobatimétrica - 08"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 09", 
     "Seção topobatimétrica - 09"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 10", 
     "Seção topobatimétrica - 10"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 11", 
     "Seção topobatimétrica - 11"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 12", 
     "Seção topobatimétrica - 12"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 13", 
     "Seção topobatimétrica - 13"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 14", 
     "Seção topobatimétrica - 14"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 15", 
     "Seção topobatimétrica - 15"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 16", 
     "Seção topobatimétrica - 16"),
    
    ("InventÃ¡rio dos solos de vÃ¡rzea da bacia hidrogrÃ¡fica do rio IguaÃ§u", 
     "Inventário dos solos de várzea da bacia hidrográfica do rio Iguaçu"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 17", 
     "Seção topobatimétrica - 17"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 18", 
     "Seção topobatimétrica - 18"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 19", 
     "Seção topobatimétrica - 19"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 20", 
     "Seção topobatimétrica - 20"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 21", 
     "Seção topobatimétrica - 21"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 22", 
     "Seção topobatimétrica - 22"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 23", 
     "Seção topobatimétrica - 23"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 24", 
     "Seção topobatimétrica - 24"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 25", 
     "Seção topobatimétrica - 25"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 26", 
     "Seção topobatimétrica - 26"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 27", 
     "Seção topobatimétrica - 27"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 28", 
     "Seção topobatimétrica - 28"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 29", 
     "Seção topobatimétrica - 29"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 30", 
     "Seção topobatimétrica - 30"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 31", 
     "Seção topobatimétrica - 31"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 32", 
     "Seção topobatimétrica - 32"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 33", 
     "Seção topobatimétrica - 33"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 34", 
     "Seção topobatimétrica - 34"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 35", 
     "Seção topobatimétrica - 35"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 36", 
     "Seção topobatimétrica - 36"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 37", 
     "Seção topobatimétrica - 37"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 37A", 
     "Seção topobatimétrica - 37A"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 38", 
     "Seção topobatimétrica - 38"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 38A", 
     "Seção topobatimétrica - 38A"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 39", 
     "Seção topobatimétrica - 39"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 40", 
     "Seção topobatimétrica - 40"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 41", 
     "Seção topobatimétrica - 41"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 42", 
     "Seção topobatimétrica - 42"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 43", 
     "Seção topobatimétrica - 43"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 44", 
     "Seção topobatimétrica - 44"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 45", 
     "Seção topobatimétrica - 45"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 46", 
     "Seção topobatimétrica - 46"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 47", 
     "Seção topobatimétrica - 47"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 48", 
     "Seção topobatimétrica - 48"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 49", 
     "Seção topobatimétrica - 49"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 50", 
     "Seção topobatimétrica - 50"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 51", 
     "Seção topobatimétrica - 51"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 51A", 
     "Seção topobatimétrica - 51A"),
    
    ("Lay - out geral de esgotamento sanitÃ¡rio", 
     "Lay - out geral de esgotamento sanitário"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 52", 
     "Seção topobatimétrica - 52"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 53", 
     "Seção topobatimétrica - 53"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 54", 
     "Seção topobatimétrica - 54"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 55", 
     "Seção topobatimétrica - 55"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 56", 
     "Seção topobatimétrica - 56"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 57", 
     "Seção topobatimétrica - 57"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 58", 
     "Seção topobatimétrica - 58"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 58A", 
     "Seção topobatimétrica - 58A"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 58B", 
     "Seção topobatimétrica - 58B"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 58C", 
     "Seção topobatimétrica - 58C"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 59", 
     "Seção topobatimétrica - 59"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 60", 
     "Seção topobatimétrica - 60"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 61", 
     "Seção topobatimétrica - 61"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 62", 
     "Seção topobatimétrica - 62"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 63", 
     "Seção topobatimétrica - 63"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 64", 
     "Seção topobatimétrica - 64"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 65", 
     "Seção topobatimétrica - 65"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 66", 
     "Seção topobatimétrica - 66"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 66P", 
     "Seção topobatimétrica - 66P"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 67", 
     "Seção topobatimétrica - 67"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 67P", 
     "Seção topobatimétrica - 67P"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 68", 
     "Seção topobatimétrica - 68"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 69", 
     "Seção topobatimétrica - 69"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 70", 
     "Seção topobatimétrica - 70"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 71", 
     "Seção topobatimétrica - 71"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 72", 
     "Seção topobatimétrica - 72"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 73", 
     "Seção topobatimétrica - 73"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 74", 
     "Seção topobatimétrica - 74"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 75", 
     "Seção topobatimétrica - 75"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 76", 
     "Seção topobatimétrica - 76"),
    
    ("SeÃ§Ã£o topobatimÃ©trica - 77", 
     "Seção topobatimétrica - 77"),
    
    ("Principais propostas", 
     "Principais propostas"),
    
    ("Sub-bacia Alto IguaÃ§u: Alternativa de projeto otimizando o combate a enchentes", 
     "Sub-bacia Alto Iguaçu: Alternativa de projeto otimizando o combate a enchentes"),
    
    ("Sub-bacia Alto IguaÃ§u: Projeto do canal de Ã¡gua limpa - elaborado pela SANEPAR", 
     "Sub-bacia Alto Iguaçu: Projeto do canal de água limpa - elaborado pela SANEPAR"),
    
    ("Projeto - PrevenÃ§Ã£o de acidentes rodoviÃ¡rios com cargas perigosas", 
     "Projeto - Prevenção de acidentes rodoviários com cargas perigosas"),
    
    ("Proposta: resÃ­duos sÃ³lidos urbanos", 
     "Proposta: resíduos sólidos urbanos"),
    
    ("TraÃ§ado geomÃ©trico do canal extravasor", 
     "Traçado geométrico do canal extravasor"),
    
    ("Estrutura viÃ¡ria metropolitana", 
     "Estrutura viária metropolitana"),
    
    ("Via metropolitana 1: diretriz bÃ¡sica", 
     "Via metropolitana 1: diretriz básica"),
    
    ("MineraÃ§Ã£o nas vÃ¡rzeas do Alto IguaÃ§u: aÃ§Ãµes de fomento e economia mineral", 
     "Mineração nas várzeas do Alto Iguaçu: ações de fomento e economia mineral"),
    
    ("Mapa de zoneamento", 
     "Mapa de zoneamento"),
    
    ("SituaÃ§Ã£o fundiÃ¡ria: mapa de desapropriaÃ§Ãµes", 
     "Situação fundiária: mapa de desapropriações"),
    
    ("Ãreas das sub-bacias do Alto IguaÃ§u", 
     "Áreas das sub-bacias do Alto Iguaçu"),
    
    ("Mapa de vegetaÃ§Ã£o", 
     "Mapa de vegetação"),
    
    ("Mapa geral: bacias contribuintes para o abastecimento d' Ã¡gua - Alto IguaÃ§u", 
     "Mapa geral: bacias contribuintes para o abastecimento d' água - Alto Iguaçu"),
    
    ("Drenagem natural: sub-bacias do Alto IguaÃ§u", 
     "Drenagem natural: sub-bacias do Alto Iguaçu"),
    
    ("Sub-bacia Alto IguaÃ§u", 
     "Sub-bacia Alto Iguaçu"),
    
    ("Levantamento planialtimÃ©trico da Ã¡rea do memorial do IguaÃ§u", 
     "Levantamento planialtimétrico da área do memorial do Iguaçu"),
    
    ("Parque Regional do IguaÃ§u: desapropriaÃ§Ãµes", 
     "Parque Regional do Iguaçu: desapropriações"),
    
    ("Plano Diretor Alto IguaÃ§u", 
     "Plano Diretor Alto Iguaçu"),
    
    ("Mapa de Ã¡reas verdes", 
     "Mapa de áreas verdes"),
    
    ("Parque IguaÃ§u", 
     "Parque Iguaçu"),
    
    ("Base cartogrÃ¡fica do Parque do IguaÃ§u", 
     "Base cartográfica do Parque do Iguaçu"),
    
    ("Proposta da delimitaÃ§Ã£o do Parque do IguaÃ§u", 
     "Proposta da delimitação do Parque do Iguaçu"),
    
    ("DesapropriaÃ§Ãµes Parque IguaÃ§u", 
     "Desapropriações Parque Iguaçu"),
    
    ("UtilizaÃ§Ã£o dos recursos hÃ­dricos da bacia do IguaÃ§u", 
     "Utilização dos recursos hídricos da bacia do Iguaçu"),
    
    ("OcupaÃ§Ã£o urbana: situaÃ§Ã£o em 1981", 
     "Ocupação urbana: situação em 1981"),
    
    ("Sistema viÃ¡rio bÃ¡sico regional: projetos e obras a serem executados - proposta", 
     "Sistema viário básico regional: projetos e obras a serem executados - proposta"),
    
    ("Transportes: sistema viÃ¡rio rural - situaÃ§Ã£o", 
     "Transportes: sistema viário rural - situação"),
    
    ("PreservaÃ§Ã£o e valorizaÃ§Ã£o do potencial natural e paisagÃ­stico: proposta", 
     "Preservação e valorização do potencial natural e paisagístico: proposta"),
    
    ("Abastecimento de Ã¡gua e saneamento ambiental: proposta", 
     "Abastecimento de água e saneamento ambiental: proposta"),
    
    ("Uso do solo: situaÃ§Ã£o - 1981", 
     "Uso do solo: situação - 1981"),
    
    ("Densidade: situaÃ§Ã£o - 1981", 
     "Densidade: situação - 1981"),
    
    ("Transporte intermunicipal: movimentaÃ§Ã£o diÃ¡ria de passageiros - situaÃ§Ã£o 1976", 
     "Transporte intermunicipal: movimentação diária de passageiros - situação 1976"),
    
    ("Transporte intermunicipal: movimentaÃ§Ã£o diÃ¡ria de passageiros - situaÃ§Ã£o 1981", 
     "Transporte intermunicipal: movimentação diária de passageiros - situação 1981"),
    
    ("Transportes: rede de transporte coletivo - situaÃ§Ã£o 1976", 
     "Transportes: rede de transporte coletivo - situação 1976"),
    
    ("Transportes: sistema viÃ¡rio bÃ¡sico - situaÃ§Ã£o 1976", 
     "Transportes: sistema viário básico - situação 1976"),
    
    ("Transportes: sistema viÃ¡rio bÃ¡sico - situaÃ§Ã£o 1981", 
     "Transportes: sistema viário básico - situação 1981"),
    
    ("Sistema viÃ¡rio bÃ¡sico rural: proposta", 
     "Sistema viário básico rural: proposta"),
    
    ("HabitaÃ§Ã£o: prioridades de ocupaÃ§Ã£o para fins habitacionais - proposta", 
     "Habitação: prioridades de ocupação para fins habitacionais - proposta"),
    
    ("EstratÃ©gia regional: proposta", 
     "Estratégia regional: proposta"),
    
    ("OrganizaÃ§Ã£o espacial: proposta", 
     "Organização espacial: proposta"),
    
    ("ReorganizaÃ§Ã£o do transporte coletivo por Ã´nibus: situaÃ§Ã£o", 
     "Reorganização do transporte coletivo por ônibus: situação"),
    
    ("ReorganizaÃ§Ã£o do transporte coletivo por Ã´nibus: proposta", 
     "Reorganização do transporte coletivo por ônibus: proposta"),
    
    ("Mancha urbana em 1953", 
     "Mancha urbana em 1953"),
    
    ("Mancha urbana em 1963", 
     "Mancha urbana em 1963"),
    
    ("Mancha urbana em 1976", 
     "Mancha urbana em 1976"),
    
    ("OcupaÃ§Ã£o urbana: situaÃ§Ã£o em 1976", 
     "Ocupação urbana: situação em 1976"),
    
    ("OrganizaÃ§Ã£o espacial", 
     "Organização espacial"),
    
    ("OrganizaÃ§Ã£o espacial: parte 1", 
     "Organização espacial: parte 1"),
    
    ("OrganizaÃ§Ã£o espacial: parte 2", 
     "Organização espacial: parte 2"),
    
    ("PreservaÃ§Ã£o e valorizaÃ§Ã£o do potencial natural e paisagÃ­stico: situaÃ§Ã£o em 1977", 
     "Preservação e valorização do potencial natural e paisagístico: situação em 1977"),
    
    ("Conjuntos habitacionais - estoques de terras: situaÃ§Ã£o em 1977", 
     "Conjuntos habitacionais - estoques de terras: situação em 1977"),
    
    ("Vazios urbanos em loteamentos", 
     "Vazios urbanos em loteamentos"),
    
    ("Contorno do limite", 
     "Contorno do limite"),
    
    ("Suscetibilidade Ã  erosÃ£o: zonas de risco", 
     "Suscetibilidade à erosão: zonas de risco"),
    
    ("Tipos de canais", 
     "Tipos de canais"),
    
    ("Planta geral de abastecimento de Ã¡gua e esgotamento sanitÃ¡rio", 
     "Planta geral de abastecimento de água e esgotamento sanitário"),
    
    ("Ãreas prioritÃ¡rias de ocupaÃ§Ã£o urbana", 
     "Áreas prioritárias de ocupação urbana"),
    
    ("Sistema de energia elÃ©trica", 
     "Sistema de energia elétrica"),
    
    ("Mapa rodoviÃ¡rio da RMC", 
     "Mapa rodoviário da RMC"),
    
    ("Planta do nÃºcleo urbano de ItaperuÃ§u: pavimentaÃ§Ã£o", 
     "Planta do núcleo urbano de Itaperuçu: pavimentação"),
    
    ("Proposta Plano Diretor de Curitiba 1966: zoneamento de uso do solo", 
     "Proposta Plano Diretor de Curitiba 1966: zoneamento de uso do solo"),
    
    ("Diretrizes para proposta: uso do solo, ordenamento territorial por Ã¡reas diferenciadas", 
     "Diretrizes para proposta: uso do solo, ordenamento territorial por áreas diferenciadas"),
    
    ("Zoneamento: espacializaÃ§Ã£o da legislaÃ§Ã£o municipal de uso do solo - lei nÂ°584/81", 
     "Zoneamento: espacialização da legislação municipal de uso do solo - lei n°584/81"),
    
    ("Mapa de ocupaÃ§Ã£o urbana", 
     "Mapa de ocupação urbana"),
    
    ("Mapa de Ã¡reas com carÃªncia em infraestrutura: favelas", 
     "Mapa de áreas com carência em infraestrutura: favelas"),
    
    ("Mapa de polos e sub-centros", 
     "Mapa de polos e sub-centros"),
    
    ("Mapa de Ã¡reas de urbanizaÃ§Ã£o", 
     "Mapa de áreas de urbanização"),
    
    ("Mapa de setorizaÃ§Ã£o", 
     "Mapa de setorização"),
    
    ("Mapa de equipamentos pÃºblicos de saÃºde", 
     "Mapa de equipamentos públicos de saúde"),
    
    ("Mapa de sistema de esgoto sanitÃ¡rio", 
     "Mapa de sistema de esgoto sanitário"),
    
    ("Mapa do programa de habitaÃ§Ã£o e urbanismo na RMC", 
     "Mapa do programa de habitação e urbanismo na RMC"),
    
    ("Plano Diretor de MineraÃ§Ã£o", 
     "Plano Diretor de Mineração"),
    
    ("Mapa de prevenÃ§Ã£o de acidentes rodoviÃ¡rios com cargas perigosas", 
     "Mapa de prevenção de acidentes rodoviários com cargas perigosas"),
    
    ("Mapa de pavimentaÃ§Ã£o e terminais", 
     "Mapa de pavimentação e terminais"),
    
    ("Mapa de microdrenagem", 
     "Mapa de microdrenagem"),
    
    ("Mapa de educaÃ§Ã£o", 
     "Mapa de educação"),
    
    ("Mapa de investimentos", 
     "Mapa de investimentos"),
    
    ("Mapa de localizaÃ§Ã£o de escolas e creches", 
     "Mapa de localização de escolas e creches"),
    
    ("Mapa polÃ­tico rodoviÃ¡rio da RMC", 
     "Mapa político rodoviário da RMC"),
    
    ("Mapa da ocupaÃ§Ã£o", 
     "Mapa da ocupação"),
    
    ("Carta temÃ¡tica da ocupaÃ§Ã£o urbana: evoluÃ§Ã£o da ocupaÃ§Ã£o urbana", 
     "Carta temática da ocupação urbana: evolução da ocupação urbana"),
    
    ("Mapa temÃ¡tico da ocupaÃ§Ã£o urbana: expansÃ£o urbana - ano 1995", 
     "Mapa temático da ocupação urbana: expansão urbana - ano 1995"),
    
    ("Mapa temÃ¡tico da ocupaÃ§Ã£o urbana: expansÃ£o urbana - ano 1999", 
     "Mapa temático da ocupação urbana: expansão urbana - ano 1999"),
    
    ("Mapa temÃ¡tico da ocupaÃ§Ã£o urbana: expansÃ£o urbana - ano 1985", 
     "Mapa temático da ocupação urbana: expansão urbana - ano 1985"),
    
    ("Mapa temÃ¡tico da ocupaÃ§Ã£o urbana: expansÃ£o urbana - ano 1990", 
     "Mapa temático da ocupação urbana: expansão urbana - ano 1990"),
    
    ("Densidade demogrÃ¡fica - 1991", 
     "Densidade demográfica - 1991"),
    
    ("Densidade demogrÃ¡fica - 1996", 
     "Densidade demográfica - 1996"),
    
    ("DivisÃ£o polÃ­tica", 
     "Divisão política"),
    
    ("ArticulaÃ§Ã£o", 
     "Articulação"),
    
    ("Zoneamento e uso do solo", 
     "Zoneamento e uso do solo"),
    
    ("Ãreas urbanas", 
     "Áreas urbanas"),
    
    ("Ãreas urbanas e urbanizadas", 
     "Áreas urbanas e urbanizadas"),
    
    ("Ãreas verdes: apa's e parques", 
     "Áreas verdes: APA's e parques"),
    
    ("Centros tecnolÃ³gicos e distritos industriais", 
     "Centros tecnológicos e distritos industriais"),
    
    ("Projeto Alto do Ribeira", 
     "Projeto Alto do Ribeira"),
    
    ("Mapa das principais Ã¡reas de proteÃ§Ã£o especial de interesse para os recursos hÃ­dricos", 
     "Mapa das principais áreas de proteção especial de interesse para os recursos hídricos"),
    
    ("Divisas de municÃ­pios da RMC", 
     "Divisas de municípios da RMC"),
    
    ("RevisÃ£o da divisa municipal", 
     "Revisão da divisa municipal"),
    
    ("Vias e municÃ­pios", 
     "Vias e municípios"),
    
    ("Mapa geral de locaÃ§Ã£o de projetos", 
     "Mapa geral de locação de projetos"),
    
    ("OrganizaÃ§Ã£o espacial - mapa base reduzido", 
     "Organização espacial - mapa base reduzido"),
    
    ("Plano de preservaÃ§Ã£o do acervo cultural da regiÃ£o metropolitana de Curitiba", 
     "Plano de preservação do acervo cultural da região metropolitana de Curitiba"),
    
    ("Mapa do sistema viÃ¡rio bÃ¡sico e zoneamento do uso do solo", 
     "Mapa do sistema viário básico e zoneamento do uso do solo"),
    
    ("Mapa do valor da terra: preÃ§o por mÂ²", 
     "Mapa do valor da terra: preço por m²"),
    
    ("Mapa de sistema de coleta de esgoto", 
     "Mapa de sistema de coleta de esgoto"),
    
    ("Plano de aÃ§Ã£o visando o combate Ã s enchentes", 
     "Plano de ação visando o combate às enchentes"),
    
    ("BolsÃµes de carÃªncia por tipologia de infra-estrutura", 
     "Bolsões de carência por tipologia de infra-estrutura"),
    
    ("Mapa de cobertura vegetal", 
     "Mapa de cobertura vegetal"),
    
    ("Mapa base com quadrÃ­culas: subdivisÃ£o para densidades", 
     "Mapa base com quadrículas: subdivisão para densidades"),
    
    ("DisposiÃ§Ã£o final de resÃ­duos sÃ³lidos", 
     "Disposição final de resíduos sólidos"),
    
    ("VegetaÃ§Ã£o da RMC", 
     "Vegetação da RMC"),
    
    ("Planta planialtimÃ©trica do Parque Marumbi", 
     "Planta planialtimétrica do Parque Marumbi"),
    
    ("Rios e localidades da RMC", 
     "Rios e localidades da RMC"),
    
    ("Projetos habitacionais da COHAB e da INOCOOP", 
     "Projetos habitacionais da COHAB e da INOCOOP"),
    
    ("Ãreas ocupadas e urbanizadas", 
     "Áreas ocupadas e urbanizadas"),
    
    ("Esgoto sanitÃ¡rio", 
     "Esgoto sanitário"),
    
    ("EvoluÃ§Ã£o da ocupaÃ§Ã£o urbana", 
     "Evolução da ocupação urbana"),
    
    ("BolsÃµes de carÃªncia: nÃ­veis de carÃªncia em obras urbanizadas", 
     "Bolsões de carência: níveis de carência em obras urbanizadas"),
    
    ("Abastecimento de Ã¡gua", 
     "Abastecimento de água"),
    
    ("Ãreas pÃºblicas municipais", 
     "Áreas públicas municipais"),
    
    ("Densidade futura por setor censitÃ¡rio", 
     "Densidade futura por setor censitário"),
    
    ("Densidade demogrÃ¡fica: levantamento preliminar", 
     "Densidade demográfica: levantamento preliminar"),
    
    ("Zoneamento do uso do solo municipal", 
     "Zoneamento do uso do solo municipal"),
    
    ("COHAB - empreendimentos habitacionais", 
     "COHAB - empreendimentos habitacionais"),
    
    ("ClassificaÃ§Ã£o dos rios", 
     "Classificação dos rios"),
    
    ("LocalizaÃ§Ã£o de conjuntos habitacionais - INOCOOP", 
     "Localização de conjuntos habitacionais - INOCOOP"),
    
    ("Modelo estruturante regional", 
     "Modelo estruturante regional"),
    
    ("OcupaÃ§Ã£o de municÃ­pios da RMC", 
     "Ocupação de municípios da RMC"),
    
    ("Plano Diretor de Abastecimento de Ãgua: setor leste", 
     "Plano Diretor de Abastecimento de Água: setor leste"),
    
    ("Circuito de turismo rural", 
     "Circuito de turismo rural"),
    
    ("Reserva hÃ­drica regional", 
     "Reserva hídrica regional"),
    
    ("Faixa de proteÃ§Ã£o de fundo de vale", 
     "Faixa de proteção de fundo de vale"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: mapa base", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: mapa base"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: modelo digital do terreno", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: modelo digital do terreno"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: macrozoneamento de uso e ocupaÃ§Ã£o do solo", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: macrozoneamento de uso e ocupação do solo"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: Ã¡reas de conservaÃ§Ã£o", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: áreas de conservação"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: conflitos entre Ã¡reas minerais e grutas", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: conflitos entre áreas minerais e grutas"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: localizaÃ§Ã£o de conflitos", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: localização de conflitos"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: produÃ§Ã£o econÃ´mica local", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: produção econômica local"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: zonas e pontos de recarga", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: zonas e pontos de recarga"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: uso e ocupaÃ§Ã£o do solo e vetores de ocupaÃ§Ã£o", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: uso e ocupação do solo e vetores de ocupação"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: equipamentos urbanos", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: equipamentos urbanos"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: sistema viÃ¡rio regional", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: sistema viário regional"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: densidades demogrÃ¡ficas urbanas e rurais", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: densidades demográficas urbanas e rurais"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: aptidÃ£o agrÃ­cola dos solos", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: aptidão agrícola dos solos"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: patrimÃ´nio espeleolÃ³gico", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: patrimônio espeleológico"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: adequabilidade para uso e ocupaÃ§Ã£o do solo", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: adequabilidade para uso e ocupação do solo"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: hidrogeologia", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: hidrogeologia"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: declividades", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: declividades"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: materiais inconsolidados", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: materiais inconsolidados"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: geologia", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: geologia"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: hidrografia e captaÃ§Ãµes", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: hidrografia e captações"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: compartimentaÃ§Ã£o geomorfolÃ³gica", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: compartimentação geomorfológica"),
    
    ("Zoneamento de uso e ocupaÃ§Ã£o do solo na regiÃ£o do Karst na RMC: Ã¡reas do karst de interesse para abastecimento pÃºblico", 
     "Zoneamento de uso e ocupação do solo na região do Karst na RMC: áreas do karst de interesse para abastecimento público"),
    
    ("Densidade urbana lÃ­quida", 
     "Densidade urbana líquida"),
]

# (continuação do código...)
# Como a lista é muito extensa, vamos truncar. O importante é que a função já contém todas essas substituições no dicionário.

# ========== 4. APLICAR CORREÇÃO (já aplicado acima) ==========
# O código já aplicou a correção usando a função definida.

# ========== 5. COMPARAR ANTES/DEPOIS (já feito) ==========

# ========== 6. ESTATÍSTICAS DOS DADOS CORRIGIDOS ==========
print("\n📊 ESTATÍSTICAS DOS DADOS CORRIGIDOS:")

# Contar palavras-chave frequentes
palavras_chave = [
    'aerofotogramétrico', 'topográfico', 'planialtimétrico', 'hipsométrico',
    'geológico', 'geomorfológico', 'cartográfico', 'cadastral',
    'esgotamento', 'saneamento', 'abastecimento', 'ocupação',
    'legislação', 'estruturação', 'implantação', 'delimitação',
    'desapropriação', 'localização', 'conservação', 'preservação',
    'valorização', 'caracterização', 'compartimentação', 'erodibilidade',
    'suscetibilidade', 'geotécnico', 'litologia', 'pedologia',
    'vegetação', 'reflorestamento', 'declividade', 'drenagem',
    'hidrografia', 'hidrologia', 'topografia', 'geomorfologia',
    'hipsometria', 'bacias', 'mananciais', 'reservatório',
    'captações', 'perímetro', 'zoneamento', 'macrozoneamento',
    'uso', 'solo', 'aptidão', 'adequabilidade', 'recomendação',
    'padrões', 'proposta', 'situação', 'diagnóstico', 'diretriz',
    'estratégia', 'planejamento', 'gestão', 'desenvolvimento',
    'crescimento', 'expansão', 'evolução', 'urbanização', 'rural',
    'urbano', 'municipal', 'município', 'distrito', 'sede',
    'núcleo', 'limite', 'divisa', 'regional', 'metropolitano',
    'RMC', 'COMEC', 'URBS', 'SANEPAR', 'IAP', 'IBGE', 'PDM',
    'PEU', 'UTP', 'APA', 'APP', 'ZEIS', 'ZUPI', 'COHAB',
    'INOCOOP', 'PROSAM', 'CIAC', 'CEF', 'TELEPAR', 'Karst',
    'Carste', 'AUDI', 'WOLKS', 'Paraná', 'Curitiba', 'São José',
    'Pinhais', 'Iguaçu', 'Passaúna', 'Iraí', 'Piraquara', 'Barigui',
    'Colombo', 'Tamandaré', 'Campo Largo', 'Campo Magro', 'Fazenda Rio Grande',
    'Itaperuçu', 'Quatro Barras', 'Almirante Tamandaré', 'Campina Grande do Sul',
    'Araucária', 'Balsa Nova', 'Contenda', 'Mandirituba', 'Bocaiúva do Sul',
    'Guarituba', 'Itaqui', 'Guadalupe', 'Maracanã', 'Renault', 'Alphaville',
    'Graciosa', 'Guajuvira', 'Bugre', 'Recife', 'Porto Alegre', 'Salvador',
    'Londrina', 'Maringá', 'Florianópolis', 'Monterrey', 'Bolívia', 'Brasil'
]

print("\n📈 FREQUÊNCIA DE PALAVRAS-CHAVE NOS TÍTULOS:")
titulos_texto = ' '.join(df[coluna_alvo].fillna('').astype(str)).lower()

for palavra in palavras_chave:
    palavra_lower = palavra.lower()
    count = titulos_texto.count(palavra_lower)
    if count > 0:
        print(f"  • {palavra.upper():30} → {count:4d} ocorrências")

# Projetos por tipo/categoria
print(f"\n🏢 CATEGORIZAÇÃO DOS TÍTULOS:")
categorias = {
    'Levantamento': df[coluna_alvo].str.contains('(?i)levantamento', na=False).sum(),
    'Carta': df[coluna_alvo].str.contains('(?i)carta', na=False).sum(),
    'Mapa': df[coluna_alvo].str.contains('(?i)mapa', na=False).sum(),
    'Planta': df[coluna_alvo].str.contains('(?i)planta', na=False).sum(),
    'Plano': df[coluna_alvo].str.contains('(?i)plano', na=False).sum(),
    'Estudo': df[coluna_alvo].str.contains('(?i)estudo', na=False).sum(),
    'Relatório': df[coluna_alvo].str.contains('(?i)relatório', na=False).sum(),
    'Projeto': df[coluna_alvo].str.contains('(?i)projeto', na=False).sum(),
    'Diretrizes': df[coluna_alvo].str.contains('(?i)diretrizes', na=False).sum(),
    'Diagnóstico': df[coluna_alvo].str.contains('(?i)diagnóstico', na=False).sum(),
    'Zoneamento': df[coluna_alvo].str.contains('(?i)zoneamento', na=False).sum(),
    'UTP': df[coluna_alvo].str.contains('(?i)UTP', na=False).sum(),
    'APA': df[coluna_alvo].str.contains('(?i)APA', na=False).sum(),
    'PDM': df[coluna_alvo].str.contains('(?i)PDM', na=False).sum(),
    'RMC': df[coluna_alvo].str.contains('(?i)RMC', na=False).sum(),
}

for categoria, quantidade in categorias.items():
    if quantidade > 0:
        print(f"  {categoria:20} → {quantidade:4d} títulos")

# ========== 7. SALVAR RESULTADO ==========
print(f"\n💾 SALVANDO ARQUIVO CORRIGIDO: {output_file}")

# Opcional: remover coluna auxiliar
if 'titulo_original' in df.columns:
    df_salvar = df.drop('titulo_original', axis=1)
else:
    df_salvar = df

# Salvar para Excel
df_salvar.to_excel(output_file, index=False)
print(f"✅ Arquivo salvo com sucesso!")

# ========== 8. VERIFICAÇÃO DE QUALIDADE ==========
print("\n🔍 VERIFICAÇÃO DE QUALIDADE:")

# Verificar caracteres problemáticos remanescentes
caracteres_problematicos = ['Ã', 'Â', 'â‚¬', 'â', 'â€', 'Å', 'â„¢', 'Ã¶', 'Ã¤', 'Ã¼', 'Ã©', 'Ã³', 'Ãº', 'Ã§', 'Ã£', 'Ãµ', 'Ãª', 'Ã´']
problemas_encontrados = []

for idx, titulo in enumerate(df[coluna_alvo].head(100)):  # Verificar apenas os 100 primeiros
    if any(char in str(titulo) for char in caracteres_problematicos):
        # Encontrar qual caractere problemático
        for char in caracteres_problematicos:
            if char in str(titulo):
                problemas_encontrados.append((idx, char, str(titulo)[:60]))
                break

if problemas_encontrados:
    print(f"⚠️  Foram encontrados {len(problemas_encontrados)} títulos com caracteres ainda problemáticos:")
    for idx, char, titulo in problemas_encontrados[:10]:  # Mostrar só os 10 primeiros
        print(f"  Linha {idx+1}: Caractere '{char}' em '{titulo}...'")
    
    print(f"\n💡 Recomendação: Execute novamente a correção ou ajuste a função de correção.")
else:
    print("🎉 TODOS os caracteres foram corrigidos com sucesso nos 100 primeiros títulos!")

# Taxa de correção (comparar antes/depois)
total_titulos = len(df)
titulos_modificados = (df['titulo_original'] != df[coluna_alvo]).sum()
taxa_correcao = (titulos_modificados / total_titulos) * 100

print(f"\n📈 ESTATÍSTICAS FINAIS:")
print(f"   Total de títulos processados: {total_titulos}")
print(f"   Títulos modificados: {titulos_modificados}")
print(f"   Taxa de correção: {taxa_correcao:.1f}%")

# ========== 9. VISUALIZAÇÃO DOS RESULTADOS ==========
print("\n📋 AMOSTRA DOS 20 PRIMEIROS TÍTULOS CORRIGIDOS:")
print("=" * 100)

for i in range(min(20, len(df))):
    original = df['titulo_original'].iloc[i][:60] + "..." if len(df['titulo_original'].iloc[i]) > 60 else df['titulo_original'].iloc[i]
    corrigido = df[coluna_alvo].iloc[i][:60] + "..." if len(df[coluna_alvo].iloc[i]) > 60 else df[coluna_alvo].iloc[i]
    
    # Destacar se houve mudança
    if original != corrigido:
        print(f"{i+1:3d}. ✓ {corrigido}")
    else:
        print(f"{i+1:3d}.   {corrigido}")

print("=" * 100)

# ========== 10. CÓDIGO PARA USO FUTURO ==========
print("\n⚡ CÓDIGO RESUMIDO PARA USO FUTURO:")

codigo_resumido = """
# CÓDIGO RESUMIDO PARA CORRIGIR TITULO_MAPAS.xlsx

import pandas as pd

def corrigir_titulo_mapas(texto):
    if pd.isna(texto):
        return ""
    
    texto = str(texto)
    
    # Correção básica de encoding
    try:
        texto = texto.encode('latin-1', errors='ignore').decode('utf-8', errors='ignore')
    except:
        pass
    
    # Principais substituições (versão resumida)
    substituicoes = {
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú',
        'Ã£': 'ã', 'Ãµ': 'õ', 'Ã§': 'ç', 'Ãª': 'ê', 'Ã´': 'ô',
        'Ã ': 'à', 'Ã¢': 'â', 'Ã§Ã£': 'ção', 'Ã§Ãµ': 'ções',
        'Ã¡rio': 'ário', 'Ã¡ria': 'ária', 'Ã³rio': 'ório', 'Ã³ria': 'ória',
        'aerofotogramÃ©trico': 'aerofotogramétrico',
        'topogrÃ¡fica': 'topográfica',
        'planialtimÃ©trico': 'planialtimétrico',
        'hipsomÃ©trico': 'hipsométrico',
        'hidrogrÃ¡fica': 'hidrográfica',
        'geolÃ³gico': 'geológico',
        'geomorfolÃ³gica': 'geomorfológica',
        'cartogrÃ¡fica': 'cartográfica',
        'cadastral': 'cadastral',
        'esgotamento': 'esgotamento',
        'saneamento': 'saneamento',
        'abastecimento': 'abastecimento',
        'ocupaÃ§Ã£o': 'ocupação',
        'legislaÃ§Ã£o': 'legislação',
        'estruturaÃ§Ã£o': 'estruturação',
        'implantaÃ§Ã£o': 'implantação',
        'delimitaÃ§Ã£o': 'delimitação',
        'desapropriaÃ§Ã£o': 'desapropriação',
        'localizaÃ§Ã£o': 'localização',
        'conservaÃ§Ã£o': 'conservação',
        'preservaÃ§Ã£o': 'preservação',
        'valorizaÃ§Ã£o': 'valorização',
        'caracterizaÃ§Ã£o': 'caracterização',
        'compartimentaÃ§Ã£o': 'compartimentação',
        'erodibilidade': 'erodibilidade',
        'suscetibilidade': 'suscetibilidade',
        'geotÃ©cnico': 'geotécnico',
        'vegetaÃ§Ã£o': 'vegetação',
        'reflorestamento': 'reflorestamento',
        'declividade': 'declividade',
        'drenagem': 'drenagem',
        'hidrografia': 'hidrografia',
        'topografia': 'topografia',
        'geomorfologia': 'geomorfologia',
        'hipsometria': 'hipsometria',
        'bacias': 'bacias',
        'mananciais': 'mananciais',
        'reservatÃ³rio': 'reservatório',
        'captaÃ§Ãµes': 'captações',
        'perÃ­metro': 'perímetro',
        'zoneamento': 'zoneamento',
        'macrozoneamento': 'macrozoneamento',
        'uso': 'uso',
        'solo': 'solo',
        'aptidÃ£o': 'aptidão',
        'adequabilidade': 'adequabilidade',
        'recomendaÃ§Ã£o': 'recomendação',
        'padrÃµes': 'padrões',
        'proposta': 'proposta',
        'situaÃ§Ã£o': 'situação',
        'diagnÃ³stico': 'diagnóstico',
        'diretriz': 'diretriz',
        'estratÃ©gia': 'estratégia',
        'planejamento': 'planejamento',
        'gestÃ£o': 'gestão',
        'desenvolvimento': 'desenvolvimento',
        'crescimento': 'crescimento',
        'expansÃ£o': 'expansão',
        'evoluÃ§Ã£o': 'evolução',
        'urbanizaÃ§Ã£o': 'urbanização',
        'rural': 'rural',
        'urbano': 'urbano',
        'municipal': 'municipal',
        'municÃ­pio': 'município',
        'distrito': 'distrito',
        'sede': 'sede',
        'nÃºcleo': 'núcleo',
        'limite': 'limite',
        'divisa': 'divisa',
        'regional': 'regional',
        'metropolitano': 'metropolitano',
        'RMC': 'RMC',
        'COMEC': 'COMEC',
        'ParanÃ¡': 'Paraná',
        'Curitiba': 'Curitiba',
        'SÃ£o': 'São',
        'JosÃ©': 'José',
        'Pinhais': 'Pinhais',
        'IguaÃ§u': 'Iguaçu',
        'PassaÃºna': 'Passaúna',
        'IraÃ­': 'Iraí',
        'Piraquara': 'Piraquara',
        'Barigui': 'Barigui',
        'Colombo': 'Colombo',
        'TamandarÃ©': 'Tamandaré',
        'Campo Largo': 'Campo Largo',
        'Campo Magro': 'Campo Magro',
        'Fazenda Rio Grande': 'Fazenda Rio Grande',
        'ItaperuÃ§u': 'Itaperuçu',
        'Quatro Barras': 'Quatro Barras',
        'Almirante TamandarÃ©': 'Almirante Tamandaré',
        'Campina Grande do Sul': 'Campina Grande do Sul',
        'AraucÃ¡ria': 'Araucária',
        'Balsa Nova': 'Balsa Nova',
        'Contenda': 'Contenda',
        'Mandirituba': 'Mandirituba',
        'BocaiÃºva do Sul': 'Bocaiúva do Sul',
        'Guarituba': 'Guarituba',
        'Guajuvira': 'Guajuvira',
        'Bugre': 'Bugre',
        'Lapa': 'Lapa',
        'Morretes': 'Morretes',
        'ParanaguÃ¡': 'Paranaguá',
        'Rio Branco do Sul': 'Rio Branco do Sul',
        'Tunas do ParanÃ¡': 'Tunas do Paraná',
    }
    
    for errado, correto in substituicoes.items():
        texto = texto.replace(errado, correto)
    
    # Remover espaços extras
    import re
    texto = re.sub(r'\s+', ' ', texto)
    
    return texto.strip()

# Uso:
df = pd.read_excel('TITULO_MAPAS.xlsx')
df['titulo_corrigido'] = df['titulo_mapa'].apply(corrigir_titulo_mapas)  # ajuste o nome da coluna
df.to_excel('TITULO_MAPAS_CORRIGIDO.xlsx', index=False)
"""

print(codigo_resumido)

print("\n" + "="*100)
print("🎯 PROCESSO DE CORREÇÃO CONCLUÍDO!")
print(f"📁 Arquivo original: {input_file}")
print(f"📁 Arquivo corrigido: {output_file}")
print(f"📊 Total de registros: {len(df)}")
print(f"📈 Taxa de correção: {taxa_correcao:.1f}%")
print("="*100)

# Opcional: criar um arquivo de log com as correções
log_correcoes = pd.DataFrame({
    'linha': range(1, len(df) + 1),
    'original': df['titulo_original'],
    'corrigido': df[coluna_alvo],
    'alterado': df['titulo_original'] != df[coluna_alvo]
})

log_correcoes.to_csv('log_correcoes_titulos_mapas.csv', index=False, encoding='utf-8-sig')
print(f"\n📝 Log detalhado salvo em: 'log_correcoes_titulos_mapas.csv'")

<>:3364: SyntaxWarning: invalid escape sequence '\s'
<>:3364: SyntaxWarning: invalid escape sequence '\s'
C:\Users\felipewada\AppData\Local\Temp\ipykernel_15748\2465636805.py:3364: SyntaxWarning: invalid escape sequence '\s'
  codigo_resumido = """


🔧 CONFIGURANDO AMBIENTE PARA TITULO_MAPAS.xlsx...

📂 CARREGANDO ARQUIVO: TITULO_MAPAS.xlsx
✅ Arquivo carregado com sucesso!
📊 Dimensões: 4470 linhas × 1 colunas
📋 Primeiras 3 linhas antes da correção:
  1. Levantamento aerofotogramÃ©trico da RegiÃ£o Metropolitana de Curitiba 
  2. Levantamento aerofotogramÃ©trico da RegiÃ£o Metropolitana de Curitiba 
  3. Levantamento aerofotogramÃ©trico da RegiÃ£o Metropolitana de Curitiba 

🔄 APLICANDO CORREÇÃO DE CARACTERES...
Colunas disponíveis: ['titulo_mapas']
Nome da primeira coluna: 'titulo_mapas'
📝 Coluna alvo para correção: 'titulo_mapas'
✅ Correção aplicada em 4470 registros

📋 Exemplo de correção:
   ANTES: Levantamento aerofotogramÃ©trico da RegiÃ£o Metropolitana de Curitiba ...
   DEPOIS: Levantamento aerofotogramétrico da Região Metropolitana de Curitiba...

🔍 COMPARAÇÃO DETALHADA ANTES/DEPOIS:


,ORIGINAL,CORRIGIDO
0,Levantamento aerofotogramÃ©trico da RegiÃ£o Metropolitana de Curitiba,Levantamento aerofotogramétrico da Região Metropolitana de Curitiba
1,Levantamento aerofotogramÃ©trico da RegiÃ£o Metropolitana de Curitiba,Levantamento aerofotogramétrico da Região Metropolitana de Curitiba
2,Levantamento aerofotogramÃ©trico da RegiÃ£o Metropolitana de Curitiba,Levantamento aerofotogramétrico da Região Metropolitana de Curitiba
3,Levantamento aerofotogramÃ©trico da RegiÃ£o Metropolitana de Curitiba,Levantamento aerofotogramétrico da Região Metropolitana de Curitiba
4,Levantamento aerofotogramÃ©trico da RegiÃ£o Metropolitana de Curitiba,Levantamento aerofotogramétrico da Região Metropolitana de Curitiba
5,Levantamento aerofotogramÃ©trico da RegiÃ£o Metropolitana de Curitiba,Levantamento aerofotogramétrico da Região Metropolitana de Curitiba
6,Levantamento aerofotogramÃ©trico da RegiÃ£o Metropolitana de Curitiba,Levantamento aerofotogramétrico da Região Metropolitana de Curitiba
7,Levantamento aerofotogramÃ©trico da RegiÃ£o Metropolitana de Curitiba,Levantamento aerofotogramétrico da Região Metropolitana de Curitiba
8,Levantamento aerofotogramÃ©trico da RegiÃ£o Metropolitana de Curitiba,Levantamento aerofotogramétrico da Região Metropolitana de Curitiba
9,AtualizaÃ§Ã£o topogrÃ¡fica,Atualização topográfica



📋 EXEMPLOS ESPECÍFICOS DE CORREÇÃO (baseados na amostra):

📊 ESTATÍSTICAS DOS DADOS CORRIGIDOS:

📈 FREQUÊNCIA DE PALAVRAS-CHAVE NOS TÍTULOS:
  • AEROFOTOGRAMÉTRICO             → 1114 ocorrências
  • PLANIALTIMÉTRICO               →    6 ocorrências
  • HIPSOMÉTRICO                   →    1 ocorrências
  • GEOLÓGICO                      →   46 ocorrências
  • CADASTRAL                      →   62 ocorrências
  • ESGOTAMENTO                    →    2 ocorrências
  • SANEAMENTO                     →   12 ocorrências
  • ABASTECIMENTO                  →   39 ocorrências
  • OCUPAÇÃO                       →  408 ocorrências
  • LEGISLAÇÃO                     →   10 ocorrências
  • ESTRUTURAÇÃO                   →   15 ocorrências
  • IMPLANTAÇÃO                    →    2 ocorrências
  • DELIMITAÇÃO                    →    1 ocorrências
  • LOCALIZAÇÃO                    →   34 ocorrências
  • CONSERVAÇÃO                    →   32 ocorrências
  • PRESERVAÇÃO                    →   11 ocorrê

In [5]:
import pandas as pd

def corrigir_titulo_mapas(texto):
    if pd.isna(texto):
        return ""
    
    texto = str(texto)
    
    # Correção básica de encoding
    try:
        texto = texto.encode('latin-1', errors='ignore').decode('utf-8', errors='ignore')
    except:
        pass
    
    # Principais substituições (versão resumida)
    substituicoes = {
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú',
        'Ã£': 'ã', 'Ãµ': 'õ', 'Ã§': 'ç', 'Ãª': 'ê', 'Ã´': 'ô',
        'Ã ': 'à', 'Ã¢': 'â', 'Ã§Ã£': 'ção', 'Ã§Ãµ': 'ções',
        'Ã¡rio': 'ário', 'Ã¡ria': 'ária', 'Ã³rio': 'ório', 'Ã³ria': 'ória',
        'aerofotogramÃ©trico': 'aerofotogramétrico',
        'topogrÃ¡fica': 'topográfica',
        'planialtimÃ©trico': 'planialtimétrico',
        'hipsomÃ©trico': 'hipsométrico',
        'hidrogrÃ¡fica': 'hidrográfica',
        'geolÃ³gico': 'geológico',
        'geomorfolÃ³gica': 'geomorfológica',
        'cartogrÃ¡fica': 'cartográfica',
        'cadastral': 'cadastral',
        'esgotamento': 'esgotamento',
        'saneamento': 'saneamento',
        'abastecimento': 'abastecimento',
        'ocupaÃ§Ã£o': 'ocupação',
        'legislaÃ§Ã£o': 'legislação',
        'estruturaÃ§Ã£o': 'estruturação',
        'implantaÃ§Ã£o': 'implantação',
        'delimitaÃ§Ã£o': 'delimitação',
        'desapropriaÃ§Ã£o': 'desapropriação',
        'localizaÃ§Ã£o': 'localização',
        'conservaÃ§Ã£o': 'conservação',
        'preservaÃ§Ã£o': 'preservação',
        'valorizaÃ§Ã£o': 'valorização',
        'caracterizaÃ§Ã£o': 'caracterização',
        'compartimentaÃ§Ã£o': 'compartimentação',
        'erodibilidade': 'erodibilidade',
        'suscetibilidade': 'suscetibilidade',
        'geotÃ©cnico': 'geotécnico',
        'vegetaÃ§Ã£o': 'vegetação',
        'reflorestamento': 'reflorestamento',
        'declividade': 'declividade',
        'drenagem': 'drenagem',
        'hidrografia': 'hidrografia',
        'topografia': 'topografia',
        'geomorfologia': 'geomorfologia',
        'hipsometria': 'hipsometria',
        'bacias': 'bacias',
        'mananciais': 'mananciais',
        'reservatÃ³rio': 'reservatório',
        'captaÃ§Ãµes': 'captações',
        'perÃ­metro': 'perímetro',
        'zoneamento': 'zoneamento',
        'macrozoneamento': 'macrozoneamento',
        'uso': 'uso',
        'solo': 'solo',
        'aptidÃ£o': 'aptidão',
        'adequabilidade': 'adequabilidade',
        'recomendaÃ§Ã£o': 'recomendação',
        'padrÃµes': 'padrões',
        'proposta': 'proposta',
        'situaÃ§Ã£o': 'situação',
        'diagnÃ³stico': 'diagnóstico',
        'diretriz': 'diretriz',
        'estratÃ©gia': 'estratégia',
        'planejamento': 'planejamento',
        'gestÃ£o': 'gestão',
        'desenvolvimento': 'desenvolvimento',
        'crescimento': 'crescimento',
        'expansÃ£o': 'expansão',
        'evoluÃ§Ã£o': 'evolução',
        'urbanizaÃ§Ã£o': 'urbanização',
        'rural': 'rural',
        'urbano': 'urbano',
        'municipal': 'municipal',
        'municÃ­pio': 'município',
        'distrito': 'distrito',
        'sede': 'sede',
        'nÃºcleo': 'núcleo',
        'limite': 'limite',
        'divisa': 'divisa',
        'regional': 'regional',
        'metropolitano': 'metropolitano',
        'RMC': 'RMC',
        'COMEC': 'COMEC',
        'ParanÃ¡': 'Paraná',
        'Curitiba': 'Curitiba',
        'SÃ£o': 'São',
        'JosÃ©': 'José',
        'Pinhais': 'Pinhais',
        'IguaÃ§u': 'Iguaçu',
        'PassaÃºna': 'Passaúna',
        'IraÃ­': 'Iraí',
        'Piraquara': 'Piraquara',
        'Barigui': 'Barigui',
        'Colombo': 'Colombo',
        'TamandarÃ©': 'Tamandaré',
        'Campo Largo': 'Campo Largo',
        'Campo Magro': 'Campo Magro',
        'Fazenda Rio Grande': 'Fazenda Rio Grande',
        'ItaperuÃ§u': 'Itaperuçu',
        'Quatro Barras': 'Quatro Barras',
        'Almirante TamandarÃ©': 'Almirante Tamandaré',
        'Campina Grande do Sul': 'Campina Grande do Sul',
        'AraucÃ¡ria': 'Araucária',
        'Balsa Nova': 'Balsa Nova',
        'Contenda': 'Contenda',
        'Mandirituba': 'Mandirituba',
        'BocaiÃºva do Sul': 'Bocaiúva do Sul',
        'Guarituba': 'Guarituba',
        'Guajuvira': 'Guajuvira',
        'Bugre': 'Bugre',
        'Lapa': 'Lapa',
        'Morretes': 'Morretes',
        'ParanaguÃ¡': 'Paranaguá',
        'Rio Branco do Sul': 'Rio Branco do Sul',
        'Tunas do ParanÃ¡': 'Tunas do Paraná',
    }
    
    for errado, correto in substituicoes.items():
        texto = texto.replace(errado, correto)
    
    # Remover espaços extras
    import re
    texto = re.sub(r'\s+', ' ', texto)
    
    return texto.strip()

# Uso:
df = pd.read_excel('TITULO_MAPAS.xlsx')
df['titulo_corrigido'] = df['titulo_mapas'].apply(corrigir_titulo_mapas)  # ajuste o nome da coluna
df.to_excel('TITULO_MAPAS_CORRIGIDO.xlsx', index=False)

# CÓDIGO DE CONVERSÃO DE DATAS PARA DATETIME PARA INSERÇÃO NO BANCO DE DADOS

In [6]:
import pandas as pd
import re
from datetime import datetime

# ========== CONFIGURAÇÃO ==========
print("🔧 PROCESSANDO DATAS DE MAPAS...")
input_file = "DATA_MAPAS.xlsx"
output_file = "DATA_MAPAS_CONVERTIDO.xlsx"

# ========== CARREGAR ARQUIVO ==========
print(f"\n📂 CARREGANDO ARQUIVO: {input_file}")
try:
    df = pd.read_excel(input_file)
    print(f"✅ Arquivo carregado com sucesso!")
    print(f"📊 Total de registros: {len(df)}")
    
    # Verificar nome da coluna
    coluna_original = df.columns[0]
    print(f"🔍 Coluna identificada: '{coluna_original}'")
    
except Exception as e:
    print(f"❌ Erro ao carregar: {e}")
    # Tentar sem cabeçalho
    try:
        df = pd.read_excel(input_file, header=None)
        print("✅ Carregado sem cabeçalho")
        coluna_original = 0
    except Exception as e2:
        print(f"❌ Erro alternativo: {e2}")
        raise

# ========== FUNÇÃO DE CONVERSÃO APERFEIÇOADA ==========
def converter_data_mapa(valor):
    """
    Converte datas variadas para formato padrão YYYY-MM-DD
    Lida com: 1999, [1999], 1999[?], [1999?], -1986, NULL, S.D, etc.
    """
    if pd.isna(valor):
        return None
    
    valor_str = str(valor).strip()
    
    # Casos especiais
    if valor_str == '' or valor_str.upper() in ['NULL', 'NAN', 'NONE', 'S.D', 'S.D.', 'SEM DATA']:
        return None
    
    # Limpar caracteres especiais como colchetes, parênteses, hífens (mantendo dígitos)
    # Mas queremos extrair o ano, então podemos remover tudo que não é dígito e verificar se sobra 4 dígitos
    # No entanto, cuidado com textos como "83x135cm" que não são datas
    # Vamos primeiro tentar extrair um ano de 4 dígitos com regex
    padrao_ano = r'(\d{4})'
    match = re.search(padrao_ano, valor_str)
    
    if match:
        ano = int(match.group(1))
        
        # Validar ano razoável (assumindo mapas entre 1500 e ano atual + 2)
        ano_atual = datetime.now().year
        if 1500 <= ano <= (ano_atual + 2):
            return f"{ano}-01-01"  # Padronizar como 1º de janeiro
    
    # Tentar ano de 2 dígitos (ex: 86 para 1986) - mas cuidado com ambiguidade
    # Muitos valores são anos completos, então essa etapa pode ser menos necessária
    padrao_ano_2digitos = r'(\d{2})'
    match_2 = re.search(padrao_ano_2digitos, valor_str)
    
    if match_2:
        ano_2dig = int(match_2.group(1))
        if 0 <= ano_2dig <= 99:
            # Heurística: se for <= 30, assume 2000+, senão 1900+
            if ano_2dig <= 30:
                ano = 2000 + ano_2dig
            else:
                ano = 1900 + ano_2dig
            return f"{ano}-01-01"
    
    return None

# ========== ANALISAR DADOS ORIGINAIS ==========
print(f"\n🔍 ANALISANDO FORMATOS DE DATA...")

# Coletar amostras de diferentes formatos
formatos_encontrados = {}
amostras = df[coluna_original].dropna().head(50).tolist()

for amostra in amostras:
    formato = str(amostra).strip()
    if formato not in formatos_encontrados:
        formatos_encontrados[formato] = 0
    formatos_encontrados[formato] += 1

print(f"📋 Formatos únicos encontrados: {len(formatos_encontrados)}")
print("\n📊 TOP 10 FORMATOS MAIS COMUNS:")
for formato, count in sorted(formatos_encontrados.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  '{formato}' → {count} ocorrências")

# ========== APLICAR CONVERSÃO ==========
print(f"\n🔄 CONVERTENDO DATAS...")

# Criar DataFrame com resultado
df_resultado = pd.DataFrame({
    'DATA_MAPAS_ORIGINAL': df[coluna_original],
    'DATA_MAPAS_CONVERTIDA': df[coluna_original].apply(converter_data_mapa)
})

# Extrair apenas o ano para análise
df_resultado['ANO_MAPA'] = df_resultado['DATA_MAPAS_CONVERTIDA'].str[:4]

# ========== ANÁLISE DOS RESULTADOS ==========
print(f"\n📊 ESTATÍSTICAS DA CONVERSÃO:")

# Contar conversões bem sucedidas
total_registros = len(df_resultado)
conversoes_validas = df_resultado['DATA_MAPAS_CONVERTIDA'].notna().sum()
conversoes_invalidas = total_registros - conversoes_validas

print(f"✅ Datas convertidas: {conversoes_validas} ({conversoes_validas/total_registros*100:.1f}%)")
print(f"❌ Datas não convertidas: {conversoes_invalidas} ({conversoes_invalidas/total_registros*100:.1f}%)")

# Distribuição por década
if conversoes_validas > 0:
    print(f"\n📅 DISTRIBUIÇÃO POR DÉCADA:")
    
    # Extrair década
    def extrair_decada(data_str):
        if pd.isna(data_str):
            return None
        ano = int(data_str[:4])
        return f"{ano//10*10}s"
    
    df_resultado['DECADA'] = df_resultado['DATA_MAPAS_CONVERTIDA'].apply(extrair_decada)
    
    # Contar por década
    decadas = df_resultado['DECADA'].value_counts().sort_index()
    for decada, count in decadas.items():
        if pd.notna(decada):
            print(f"  {decada}: {count} mapas ({count/conversoes_validas*100:.1f}%)")

# Mostrar exemplos problemáticos
print(f"\n🔍 EXEMPLOS DE DADOS NÃO CONVERTIDOS:")
problemas = df_resultado[df_resultado['DATA_MAPAS_CONVERTIDA'].isna()].head(10)
for idx, row in problemas.iterrows():
    if pd.notna(row['DATA_MAPAS_ORIGINAL']):
        print(f"  Linha {idx+1}: '{row['DATA_MAPAS_ORIGINAL']}'")

# ========== FUNÇÃO PARA CORRIGIR CASOS ESPECIAIS ==========
def converter_casos_especiais(valor):
    """
    Trata casos especiais que a função principal não capturou
    """
    if pd.isna(valor):
        return None
    
    valor_str = str(valor).strip().lower()
    
    # Mapeamento de casos especiais
    casos_especiais = {
        '197': '1970',  # Ano truncado
        '198': '1980',  # Exemplo: '198?' talvez?
        '199': '1990',
        '200': '2000',
        '1891, doaÃ§': '1891',  # Corrigir caracteres + texto
        '1971.': '1971',  # Ponto no final
        's.d': None,  # Sem data
        '[s.d]': None,
        '83x135cm': None,  # Dimensão, não data
    }
    
    for padrao, substituicao in casos_especiais.items():
        if padrao in valor_str:
            return substituicao + "-01-01" if substituicao else None
    
    return None

# ========== TENTATIVA DE CORREÇÃO ADICIONAL ==========
print(f"\n🔄 APLICANDO CORREÇÃO ADICIONAL PARA CASOS ESPECIAIS...")

# Aplicar correção adicional nos casos não convertidos
mascara_nao_convertidos = df_resultado['DATA_MAPAS_CONVERTIDA'].isna()
df_resultado.loc[mascara_nao_convertidos, 'DATA_MAPAS_CONVERTIDA_CORRIGIDA'] = \
    df_resultado.loc[mascara_nao_convertidos, 'DATA_MAPAS_ORIGINAL'].apply(converter_casos_especiais)

# Combinar resultados
def combinar_resultados(row):
    if pd.notna(row.get('DATA_MAPAS_CONVERTIDA_CORRIGIDA')):
        return row['DATA_MAPAS_CONVERTIDA_CORRIGIDA']
    return row['DATA_MAPAS_CONVERTIDA']

if 'DATA_MAPAS_CONVERTIDA_CORRIGIDA' in df_resultado.columns:
    df_resultado['DATA_MAPAS_FINAL'] = df_resultado.apply(combinar_resultados, axis=1)
    
    # Nova contagem
    novas_conversoes = df_resultado['DATA_MAPAS_FINAL'].notna().sum()
    melhoria = novas_conversoes - conversoes_validas
    
    if melhoria > 0:
        print(f"✅ {melhoria} novas datas convertidas!")
        print(f"🎯 Total final: {novas_conversoes} ({novas_conversoes/total_registros*100:.1f}%)")
else:
    df_resultado['DATA_MAPAS_FINAL'] = df_resultado['DATA_MAPAS_CONVERTIDA']

# ========== SALVAR RESULTADOS ==========
print(f"\n💾 SALVANDO RESULTADOS: {output_file}")

try:
    df_resultado.to_excel(output_file, index=False)
    print(f"✅ Arquivo salvo com sucesso!")
    
    # Salvar também como CSV para fácil visualização
    csv_file = output_file.replace('.xlsx', '.csv')
    df_resultado.to_csv(csv_file, index=False, encoding='utf-8-sig')
    print(f"✅ Backup CSV: {csv_file}")
    
except Exception as e:
    print(f"❌ Erro ao salvar: {e}")
    # Tentar alternativa
    try:
        df_resultado.to_excel(output_file, index=False, engine='openpyxl')
        print("✅ Salvo com engine 'openpyxl'")
    except Exception as e2:
        print(f"❌ Erro alternativo: {e2}")

# ========== RESUMO FINAL ==========
print("\n" + "="*80)
print("🎯 CONVERSÃO DE DATAS CONCLUÍDA!")
print("="*80)
print(f"📁 Arquivo de entrada: {input_file}")
print(f"📁 Arquivo de saída: {output_file}")
print(f"📊 Total de registros: {total_registros}")

final_validas = df_resultado['DATA_MAPAS_FINAL'].notna().sum()
final_invalidas = total_registros - final_validas

print(f"✅ Datas convertidas: {final_validas} ({final_validas/total_registros*100:.1f}%)")
print(f"❌ Datas não convertidas: {final_invalidas} ({final_invalidas/total_registros*100:.1f}%)")

# Anos extremos
anos_validos = df_resultado['DATA_MAPAS_FINAL'].dropna().str[:4].astype(int)
if len(anos_validos) > 0:
    print(f"📅 Ano mais antigo: {anos_validos.min()}")
    print(f"📅 Ano mais recente: {anos_validos.max()}")
    print(f"📅 Média dos anos: {anos_validos.mean():.0f}")

print("="*80)

# ========== AMOSTRA DOS RESULTADOS ==========
print(f"\n📋 AMOSTRA DOS RESULTADOS (20 primeiras linhas):")
print("="*100)
print(f"{'ORIGINAL':<30} | {'CONVERTIDO':<12} | {'ANO':<6}")
print("-" * 100)

for i in range(min(20, len(df_resultado))):
    original = str(df_resultado.iloc[i]['DATA_MAPAS_ORIGINAL'])[:28]
    convertido = str(df_resultado.iloc[i]['DATA_MAPAS_FINAL'])[:10] if pd.notna(df_resultado.iloc[i]['DATA_MAPAS_FINAL']) else "NÃO CONVERTIDO"
    ano = str(df_resultado.iloc[i]['ANO_MAPA']) if pd.notna(df_resultado.iloc[i]['ANO_MAPA']) else ""
    
    print(f"{original:<30} | {convertido:<12} | {ano:<6}")

print("="*100)

# ========== CÓDIGO PARA EXPORTAR APENAS ANOS ==========
print(f"\n⚡ BÔNUS: CRIAR ARQUIVO APENAS COM ANOS...")

# Criar arquivo simplificado apenas com anos
df_simplificado = pd.DataFrame({
    'ANO_MAPA': df_resultado['ANO_MAPA'].fillna('DESCONHECIDO')
})

# Contar frequência
contagem_anos = df_simplificado['ANO_MAPA'].value_counts().sort_index()

# Salvar contagem
arquivo_contagem = "CONTAGEM_ANOS_MAPAS.xlsx"
df_contagem = pd.DataFrame({
    'ANO': contagem_anos.index,
    'QUANTIDADE': contagem_anos.values
})

df_contagem.to_excel(arquivo_contagem, index=False)
print(f"✅ Arquivo de contagem salvo: {arquivo_contagem}")

print(f"\n🏁 PROCESSAMENTO COMPLETO!")

🔧 PROCESSANDO DATAS DE MAPAS...

📂 CARREGANDO ARQUIVO: DATA_MAPAS.xlsx
✅ Arquivo carregado com sucesso!
📊 Total de registros: 4470
🔍 Coluna identificada: 'data_mapas'

🔍 ANALISANDO FORMATOS DE DATA...
📋 Formatos únicos encontrados: 2

📊 TOP 10 FORMATOS MAIS COMUNS:
  '1976' → 45 ocorrências
  '1986' → 5 ocorrências

🔄 CONVERTENDO DATAS...

📊 ESTATÍSTICAS DA CONVERSÃO:
✅ Datas convertidas: 4421 (98.9%)
❌ Datas não convertidas: 49 (1.1%)

📅 DISTRIBUIÇÃO POR DÉCADA:
  1950s: 3 mapas (0.1%)
  1960s: 36 mapas (0.8%)
  1970s: 770 mapas (17.4%)
  1980s: 2270 mapas (51.3%)
  1990s: 1040 mapas (23.5%)
  2000s: 301 mapas (6.8%)
  2010s: 1 mapas (0.0%)

🔍 EXEMPLOS DE DADOS NÃO CONVERTIDOS:

🔄 APLICANDO CORREÇÃO ADICIONAL PARA CASOS ESPECIAIS...

💾 SALVANDO RESULTADOS: DATA_MAPAS_CONVERTIDO.xlsx
✅ Arquivo salvo com sucesso!
✅ Backup CSV: DATA_MAPAS_CONVERTIDO.csv

🎯 CONVERSÃO DE DATAS CONCLUÍDA!
📁 Arquivo de entrada: DATA_MAPAS.xlsx
📁 Arquivo de saída: DATA_MAPAS_CONVERTIDO.xlsx
📊 Total de registr

# CÓDIGO PARA CORREÇÃO DE CARACTERES CORROMPIDOS NA COLUNA CONTEUDO_MAPAS

In [1]:
# CORREÇÃO DE CARACTERES - CONTEUDO_MAPAS.xlsx
# Versão adaptada para o arquivo de mapas

import pandas as pd
import numpy as np
import re
from pathlib import Path

# ========== 1. CONFIGURAÇÃO ==========
print("🔧 CONFIGURANDO AMBIENTE...")
input_file = "CONTEUDO_MAPAS.xlsx"
output_file = "CONTEUDO_MAPAS_CORRIGIDO.xlsx"

# ========== 2. CARREGAR O ARQUIVO ==========
print(f"\n📂 CARREGANDO ARQUIVO: {input_file}")
try:
    df = pd.read_excel(input_file)
    print(f"✅ Arquivo carregado com sucesso!")
    print(f"📊 Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas")
    
    print(f"\n🔍 ESTRUTURA DO ARQUIVO:")
    print(f"Colunas: {list(df.columns)}")
    print(f"Tipos de dados: {df.dtypes.tolist()}")
    
    # Mostrar primeira coluna (que é a que nos interessa)
    primeira_coluna = df.columns[0]
    print(f"\n📄 PRIMEIRA COLUNA DETECTADA: '{primeira_coluna}'")
    
    # Mostrar amostra da primeira coluna
    print(f"📄 AMOSTRA DA COLUNA '{primeira_coluna}' (ORIGINAL - primeiras 5 linhas):")
    print("-" * 100)
    for i in range(min(5, len(df))):
        valor = str(df[primeira_coluna].iloc[i])[:200] if not pd.isna(df[primeira_coluna].iloc[i]) else ""
        print(f"Linha {i+1}: {valor}")
        print("-" * 50)
        
except Exception as e:
    print(f"❌ Erro ao carregar: {e}")
    # Tentar com openpyxl
    try:
        df = pd.read_excel(input_file, engine='openpyxl')
        print("✅ Carregado com engine 'openpyxl'")
    except Exception as e2:
        print(f"❌ Erro ao carregar com openpyxl: {e2}")
        exit()

# ========== 3. IDENTIFICAR COLUNA PARA CORRIGIR ==========
print("\n🔍 IDENTIFICANDO COLUNA PARA CORRIGIR...")

# Verificar se há colunas com caracteres problemáticos
coluna_para_corrigir = None

# Padrões comuns de caracteres corrompidos
padroes_problematicos = ['Ã¡', 'Ã©', 'Ã­', 'Ã³', 'Ãº', 'Ã£', 'Ã§', 'Ãµ', 'Ã¢', 'Ãª', 'Ã´', 'Ã¼']

for coluna in df.columns:
    amostras = df[coluna].dropna().astype(str).head(10).tolist()
    texto_amostra = ' '.join(amostras)
    
    problema_count = sum(texto_amostra.count(padrao) for padrao in padroes_problematicos)
    
    if problema_count > 0:
        coluna_para_corrigir = coluna
        print(f"✅ Coluna '{coluna}' identificada com {problema_count} caracteres problemáticos")
        break

if coluna_para_corrigir is None:
    # Usar a primeira coluna por padrão
    coluna_para_corrigir = df.columns[0]
    print(f"⚠️  Nenhuma coluna com padrões óbvios encontrada. Usando primeira coluna: '{coluna_para_corrigir}'")

print(f"\n🎯 COLUNA SELECIONADA PARA CORREÇÃO: '{coluna_para_corrigir}'")

# ========== 4. FUNÇÃO DE CORREÇÃO ADAPTADA ==========
def corrigir_caracteres_mapas(texto):
    """
    Função específica para corrigir caracteres corrompidos no CONTEUDO_MAPAS.xlsx
    Baseada no conteúdo real da coluna (descrições de mapas)
    """
    if pd.isna(texto):
        return ""
    
    texto = str(texto)
    
    # ETAPA 1: Correção via encoding (tentativas de decodificação correta)
    try:
        # Tentar: texto foi salvo como UTF-8 mas lido como Latin-1
        texto_bytes = texto.encode('latin-1', errors='ignore')
        texto = texto_bytes.decode('utf-8', errors='ignore')
    except:
        try:
            # Tentar o contrário
            texto_bytes = texto.encode('utf-8', errors='ignore')
            texto = texto_bytes.decode('latin-1', errors='ignore')
        except:
            pass
    
    # ETAPA 2: Substituições específicas para os padrões mais comuns
    substituicoes = {
        # Acentuação básica
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú',
        'Ã£': 'ã', 'Ãµ': 'õ', 'Ã§': 'ç', 'Ã¢': 'â', 'Ãª': 'ê', 'Ã´': 'ô',
        'Ã¼': 'ü', 'Ã¤': 'ä', 'Ã«': 'ë', 'Ã¯': 'ï', 'Ã¶': 'ö', 'Ã»': 'û',
        'Ã ': 'à',  # Espaço após Ã pode ser à
        'Ã\xa0': 'à',
        
        # Maiúsculas
        'Ã': 'Á', 'Ã': 'É', 'Ã': 'Í', 'Ã': 'Ó', 'Ã': 'Ú',
        'Ã': 'Ã', 'Ã': 'Ç', 'Ã': 'Õ', 'Ã': 'Â', 'Ã': 'Ê', 'Ã': 'Ô',
        'Ã': 'Ü', 'Ã': 'Ä',
        
        # Caracteres especiais
        'Â°': '°', 'Âª': 'ª', 'Âº': 'º', 'Â´': '´', 'Â·': '·',
        
        # Padrões compostos comuns no conteúdo dos mapas
        'Ã§Ã£o': 'ção',
        'Ã§Ãµes': 'ções',
        'Ã£o': 'ão',
        'Ãµes': 'ões',
        'Ãªncia': 'ência',
        'Ã¢ncia': 'ância',
        'Ã³rio': 'ório',
        'Ã¡rio': 'ário',
        'Ã©rio': 'ério',
        'Ã­vel': 'ível',
        'Ã¡vel': 'ável',
        
        # Termos específicos observados no arquivo
        'hidrografia': 'hidrografia',  # já correto, mas mantido por segurança
        'vegetaÃ§Ã£o': 'vegetação',
        'planimetria': 'planimetria',
        'altimetria': 'altimetria',
        'curvas de nÃ­vel': 'curvas de nível',
        'eqÃ¼idistÃ¢ncia': 'eqüidistância',  # ou 'equidistância' (ü pode ser mantido)
        'datum horizontal': 'datum horizontal',
        'CÃ³rrego Alegre': 'Córrego Alegre',
        'SAD-69': 'SAD-69',  # já ok
        'ItararÃ©': 'Itararé',
        'Pilar AstronÃ´mico': 'Pilar Astronômico',
        'Ponta Grossa': 'Ponta Grossa',
        'AraucÃ¡ria': 'Araucária',
        'ParanÃ¡': 'Paraná',
        'RegiÃ£o Metropolitana': 'Região Metropolitana',
        'Curitiba': 'Curitiba',
        'municÃ­pio': 'município',
        'Ã¡rea': 'área',
        'Ã¡reas': 'áreas',
        'ocupaÃ§Ã£o': 'ocupação',
        'populaÃ§Ã£o': 'população',
        'caracterÃ­sticas': 'características',
        'geomorfolÃ³gicas': 'geomorfológicas',
        'litoestratigrÃ¡ficas': 'litoestratigráficas',
        'ocorrÃªncias': 'ocorrências',
        'minerais': 'minerais',
        'declividade': 'declividade',
        'intervalos': 'intervalos',
        'por cento': 'por cento',
        'bacias': 'bacias',
        'hidrogrÃ¡ficas': 'hidrográficas',
        'sub-bacias': 'sub-bacias',
        'afluentes': 'afluentes',
        'diretos': 'diretos',
        'Ribeira': 'Ribeira',
        'IguaÃ§u': 'Iguaçu',
        'AtlÃ¢ntica': 'Atlântica',
        'AÃ§ungui': 'Açungui',
        'Capivari': 'Capivari',
        'VÃ¡rzea': 'Várzea',
        'divisores': 'divisores',
        'd\'Ã¡gua': "d'água",
        'cursos d\'Ã¡gua': "cursos d'água",
        'represa': 'represa',
        'PassaÃºna': 'Passaúna',
        'IraÃ­': 'Iraí',
        'Piraquara': 'Piraquara',
        'Miringuava': 'Miringuava',
        'Palmital': 'Palmital',
        'Verde': 'Verde',
        'Pequeno': 'Pequeno',
        'Atuba': 'Atuba',
        'BelÃ©m': 'Belém',
        'BariguÃ­': 'Barigui',
        'ItaquÃ­': 'Itaqui',
        'TacaniÃ§a': 'Tacaniça',
        'Arruamento': 'Arruamento',
        'loteamentos': 'loteamentos',
        'zoneamento': 'zoneamento',
        'sistema viÃ¡rio': 'sistema viário',
        'rodovias': 'rodovias',
        'ferrovias': 'ferrovias',
        'aeroporto': 'aeroporto',
        'terminal': 'terminal',
        'estaÃ§Ã£o': 'estação',
        'tratamento': 'tratamento',
        'esgoto': 'esgoto',
        'abastecimento': 'abastecimento',
        'energia elÃ©trica': 'energia elétrica',
        'telefone': 'telefone',
        'iluminaÃ§Ã£o': 'iluminação',
        'pavimentaÃ§Ã£o': 'pavimentação',
        'asfalto': 'asfalto',
        'paralelepÃ­pedo': 'paralelepípedo',
        'blokret': 'blokret',  # nome próprio
        'anti-pÃ³': 'anti-pó',
        'coleta de lixo': 'coleta de lixo',
        'aterro sanitÃ¡rio': 'aterro sanitário',
        'escola': 'escola',
        'creche': 'creche',
        'posto de saÃºde': 'posto de saúde',
        'hospital': 'hospital',
        'centro de saÃºde': 'centro de saúde',
        'clÃ­nica': 'clínica',
        'odontolÃ³gica': 'odontológica',
        'delegacia': 'delegacia',
        'polÃ­cia': 'polícia',
        'cÃ¢mara': 'câmara',
        'prefeitura': 'prefeitura',
        'igreja': 'igreja',
        'cemitÃ©rio': 'cemitério',
        'campo de futebol': 'campo de futebol',
        'ginÃ¡sio': 'ginásio',
        'praÃ§a': 'praça',
        'parque': 'parque',
        'bosque': 'bosque',
        'mata': 'mata',
        'mata com araucÃ¡ria': 'mata com araucária',
        'bracatinga': 'bracatinga',
        'capoeira': 'capoeira',
        'reflorestamento': 'reflorestamento',
        'agricultura': 'agricultura',
        'pecuÃ¡ria': 'pecuária',
        'pastagem': 'pastagem',
        'brejo': 'brejo',
        'campo': 'campo',
        'vÃ¡rzea': 'várzea',
        'inundaÃ§Ã£o': 'inundação',
        'inundÃ¡vel': 'inundável',
        'alagado': 'alagado',
        'erosÃ£o': 'erosão',
        'declividade': 'declividade',
        'geomorfologia': 'geomorfologia',
        'geologia': 'geologia',
        'litologia': 'litologia',
        'solos': 'solos',
        'hidrologia': 'hidrologia',
        'climatologia': 'climatologia',
        'temperatura': 'temperatura',
        'precipitaÃ§Ã£o': 'precipitação',
        'vento': 'vento',
        'direÃ§Ã£o': 'direção',
        'preferencial': 'preferencial',
        'dados': 'dados',
        'informaÃ§Ãµes': 'informações',
        'cadastro': 'cadastro',
        'mapa': 'mapa',
        'carta': 'carta',
        'planta': 'planta',
        'escala': 'escala',
        'legenda': 'legenda',
        'sem legenda': 'sem legenda',
        'fotointerpretaÃ§Ã£o': 'fotointerpretação',
        'levantamento': 'levantamento',
        'aerofotogramÃ©trico': 'aerofotogramétrico',
        'RMC': 'RMC',  # sigla
        'COMEC': 'COMEC',
        'PROSAM': 'PROSAM',
        'SANEPAR': 'SANEPAR',
        'COPEL': 'COPEL',
        'TELEPAR': 'TELEPAR',
        'COHAB': 'COHAB',
        'INOCOOP': 'INOCOOP',
        'DNPM': 'DNPM',
        'DER': 'DER',
        'DETRAN': 'DETRAN',
        'BR': 'BR',  # rodovia
        'PR': 'PR',  # estado
        'MG': 'MG',  # Minas Gerais
        'SP': 'SP',  # São Paulo
    }
    
    # Aplicar substituições
    for errado, correto in substituicoes.items():
        texto = texto.replace(errado, correto)
    
    # ETAPA 3: Correções para padrões complexos (regex)
    # Substituir combinações como 'Ã§Ã£o' por 'ção' (já feito acima, mas garantia)
    texto = re.sub(r'Ã§([Ã£o]+)', lambda m: 'ç' + m.group(1).replace('Ã£', 'a').replace('Ãµ', 'o'), texto)
    
    # Corrigir 'nÃ­vel' -> 'nível' (já incluso)
    texto = re.sub(r'nÃ­vel', 'nível', texto)
    
    # Remover caracteres de controle
    texto = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', ' ', texto)
    texto = re.sub(r'�+', ' ', texto)  # caractere de substituição
    
    # Normalizar espaços múltiplos
    texto = re.sub(r'\s+', ' ', texto)
    
    return texto.strip()

# ========== 5. APLICAR CORREÇÃO NA COLUNA IDENTIFICADA ==========
print(f"\n🔄 APLICANDO CORREÇÃO DE CARACTERES NA COLUNA '{coluna_para_corrigir}'...")

# Criar cópia para comparação
df_original = df.copy()

# Contar registros não vazios
registros_nao_vazios = df[coluna_para_corrigir].notna().sum()
print(f"  Corrigindo {registros_nao_vazios} registros não vazios...")

# Aplicar correção
df[coluna_para_corrigir] = df[coluna_para_corrigir].apply(corrigir_caracteres_mapas)

print(f"✅ Correção aplicada na coluna '{coluna_para_corrigir}'")

# ========== 6. COMPARAR ANTES/DEPOIS ==========
print(f"\n🔍 COMPARAÇÃO DETALHADA ANTES/DEPOIS (coluna '{coluna_para_corrigir}'):")
print("="*120)

# Mostrar os primeiros 5 registros corrigidos
for i in range(min(5, len(df))):
    original = str(df_original[coluna_para_corrigir].iloc[i])[:200] if not pd.isna(df_original[coluna_para_corrigir].iloc[i]) else ""
    corrigido = str(df[coluna_para_corrigir].iloc[i])[:200] if not pd.isna(df[coluna_para_corrigir].iloc[i]) else ""
    
    if not original and not corrigido:
        continue
        
    print(f"\n📄 REGISTRO {i+1}:")
    print(f"ORIGINAL  : {original}")
    print(f"CORRIGIDO : {corrigido}")
    
    if original != corrigido:
        print("✅ ALTERAÇÃO APLICADA")
    else:
        print("⚠️  SEM ALTERAÇÃO (já correto ou vazio)")
    print("-" * 120)

# ========== 7. APLICAR CORREÇÃO EM OUTRAS COLUNAS (se houver) ==========
print("\n🔍 VERIFICANDO OUTRAS COLUNAS COM PROBLEMAS...")

colunas_corrigidas = [coluna_para_corrigir]

for coluna in df.columns:
    if coluna == coluna_para_corrigir:
        continue  # Já corrigimos esta
    
    # Verificar se a coluna tem texto e contém caracteres problemáticos
    if df[coluna].dtype == 'object':
        amostras = df[coluna].dropna().astype(str).head(20).tolist()
        texto_amostra = ' '.join(amostras)
        
        if any(padrao in texto_amostra for padrao in padroes_problematicos):
            print(f"  Corrigindo coluna adicional: '{coluna}'...")
            df[coluna] = df[coluna].apply(corrigir_caracteres_mapas)
            colunas_corrigidas.append(coluna)

print(f"\n✅ Total de colunas corrigidas: {len(colunas_corrigidas)}")
print(f"📋 Colunas: {colunas_corrigidas}")

# ========== 8. VERIFICAR PROBLEMAS RESIDUAIS ==========
print("\n🔍 VERIFICANDO PROBLEMAS RESIDUAIS...")

caracteres_problematicos = ['Ã', 'Â', 'â', '€', '�']

problemas_encontrados = 0
for coluna in colunas_corrigidas:
    for idx, valor in df[coluna].items():
        if pd.isna(valor):
            continue
        texto = str(valor)
        if any(char in texto for char in caracteres_problematicos):
            problemas_encontrados += 1
            if problemas_encontrados <= 3:
                print(f"  Linha {idx+1}, Coluna '{coluna}': Encontrado caractere problemático em '{texto[:80]}...'")

if problemas_encontrados == 0:
    print("🎉 Nenhum problema residual encontrado!")
else:
    print(f"⚠️  Encontrados {problemas_encontrados} problemas residuais (apenas os primeiros 3 foram mostrados)")

# ========== 9. SALVAR ARQUIVO CORRIGIDO ==========
print(f"\n💾 SALVANDO ARQUIVO CORRIGIDO: {output_file}")

try:
    df.to_excel(output_file, index=False, engine='openpyxl')
    print(f"✅ Arquivo salvo com sucesso!")
    
    # Verificar tamanhos
    if Path(input_file).exists() and Path(output_file).exists():
        tamanho_original = Path(input_file).stat().st_size
        tamanho_corrigido = Path(output_file).stat().st_size
        print(f"📁 Tamanho original : {tamanho_original:,} bytes")
        print(f"📁 Tamanho corrigido: {tamanho_corrigido:,} bytes")
        
except Exception as e:
    print(f"❌ Erro ao salvar como Excel: {e}")
    # Tentar salvar como CSV
    try:
        csv_file = output_file.replace('.xlsx', '.csv')
        df.to_csv(csv_file, index=False, encoding='utf-8-sig')
        print(f"✅ Salvo como CSV: {csv_file}")
    except Exception as e2:
        print(f"❌ Erro ao salvar CSV: {e2}")

# ========== 10. RESUMO FINAL ==========
print("\n" + "="*80)
print("🎯 PROCESSO CONCLUÍDO!")
print("="*80)
print(f"📁 Arquivo original : {input_file}")
print(f"📁 Arquivo corrigido: {output_file}")
print(f"📊 Total de registros: {len(df)}")
print(f"📊 Total de colunas: {len(df.columns)}")
print(f"📊 Colunas corrigidas: {len(colunas_corrigidas)}")
print(f"📋 Colunas: {colunas_corrigidas}")
print("="*80)

# ========== 11. AMOSTRA FINAL ==========
print(f"\n📋 AMOSTRA FINAL DOS DADOS CORRIGIDOS (coluna '{coluna_para_corrigir}'):")
print("="*100)

for i in range(min(3, len(df))):
    print(f"\n📄 LINHA {i+1}:")
    print("-" * 100)
    valor = str(df[coluna_para_corrigir].iloc[i]) if not pd.isna(df[coluna_para_corrigir].iloc[i]) else ""
    if valor:
        # Quebrar em linhas para facilitar leitura
        linhas = [valor[j:j+100] for j in range(0, len(valor), 100)]
        for j, linha in enumerate(linhas):
            print(linha)
            if j >= 2:  # Mostrar no máximo 3 linhas por registro
                print("...")
                break
    else:
        print("[VAZIO]")

print("\n" + "="*100)
print("✅ Pronto! As correções foram aplicadas com sucesso.")
print("="*100)

# ========== BÔNUS: OPÇÃO PARA EXPORTAR EM OUTROS FORMATOS ==========
print("\n⚡ BÔNUS: CÓDIGO PARA EXPORTAR EM OUTROS FORMATOS")

export_code = """
# Exportar para diferentes formatos:
import pandas as pd

# 1. CSV com encoding correto
df.to_csv('CONTEUDO_MAPAS_CORRIGIDO_UTF8.csv', index=False, encoding='utf-8-sig')

# 2. JSON
df.to_json('CONTEUDO_MAPAS_CORRIGIDO.json', orient='records', force_ascii=False)

# 3. Parquet (formato eficiente)
df.to_parquet('CONTEUDO_MAPAS_CORRIGIDO.parquet', index=False)

print("✅ Exportado em múltiplos formatos!")
"""

print(export_code)

🔧 CONFIGURANDO AMBIENTE...

📂 CARREGANDO ARQUIVO: CONTEUDO_MAPAS.xlsx
✅ Arquivo carregado com sucesso!
📊 Dimensões: 4470 linhas × 1 colunas

🔍 ESTRUTURA DO ARQUIVO:
Colunas: ['conteudo_mapas']
Tipos de dados: [dtype('O')]

📄 PRIMEIRA COLUNA DETECTADA: 'conteudo_mapas'
📄 AMOSTRA DA COLUNA 'conteudo_mapas' (ORIGINAL - primeiras 5 linhas):
----------------------------------------------------------------------------------------------------
Linha 1: Elementos de hidrografia e vegetaÃ§Ã£o, planimetria e altimetria (curvas de nÃ­vel com eqÃ¼idistÃ¢ncia de 5m), datum horizontal: CÃ³rrego Alegre, MG. 
--------------------------------------------------
Linha 2: Elementos de hidrografia e vegetaÃ§Ã£o, planimetria e altimetria (curvas de nÃ­vel com eqÃ¼idistÃ¢ncia de 5m), datum horizontal: CÃ³rrego Alegre, MG. 
--------------------------------------------------
Linha 3: Elementos de hidrografia e vegetaÃ§Ã£o, planimetria e altimetria (curvas de nÃ­vel com eqÃ¼idistÃ¢ncia de 5m), datum horizontal: 

# CÓDIGO DE CORREÇÃO DE CARACTERES NA COLUNA NOTAS_GERAIS_MAPAS

In [3]:
# CORREÇÃO DE CARACTERES - NOTAS_GERAIS_MAPAS.xlsx
# VERSÃO MELHORADA: Detecta automaticamente a coluna textual e substitui caracteres corrompidos

import pandas as pd
import numpy as np
import re
from pathlib import Path

# ========== 1. CONFIGURAÇÃO ==========
print("🔧 CONFIGURANDO AMBIENTE...")
input_file = "NOTAS_GERAIS_MAPAS.xlsx"
output_file = "NOTAS_GERAIS_MAPAS_CORRIGIDO.xlsx"

# ========== 2. CARREGAR O ARQUIVO ==========
print(f"\n📂 CARREGANDO ARQUIVO: {input_file}")
try:
    df = pd.read_excel(input_file)
    print(f"✅ Arquivo carregado com sucesso!")
    print(f"📊 Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas")
    print(f"\n🔍 ESTRUTURA DO ARQUIVO:")
    print(f"Colunas: {list(df.columns)}")
    
    if df.empty:
        print("⚠️  O arquivo está vazio (nenhuma linha de dados).")
except Exception as e:
    print(f"❌ Erro ao carregar: {e}")
    exit()

# ========== 3. IDENTIFICAR A COLUNA COM TEXTO CORROMPIDO ==========
print("\n🔍 IDENTIFICANDO COLUNA PARA CORRIGIR...")

def detectar_coluna_corrompida(df):
    """Retorna o nome da coluna com mais caracteres problemáticos ou a primeira textual."""
    if df.empty:
        return None
    
    padroes_problematicos = ['Ã¡', 'Ã©', 'Ã­', 'Ã³', 'Ãº', 'Ã£', 'Ã§', 'Ãµ', 'Ãª', 'Ã´', 'Ã¼']
    melhor_coluna = None
    max_problemas = 0

    for coluna in df.columns:
        # Pular colunas não-texto (números, datas)
        if not pd.api.types.is_string_dtype(df[coluna]) and not pd.api.types.is_object_dtype(df[coluna]):
            continue

        amostras = df[coluna].dropna().astype(str).head(50).tolist()
        texto_amostra = ' '.join(amostras)
        problemas = sum(texto_amostra.count(p) for p in padroes_problematicos)

        if problemas > max_problemas:
            max_problemas = problemas
            melhor_coluna = coluna

    # Se nenhuma coluna tiver problemas, usar a primeira coluna (mas avisar)
    if melhor_coluna is None and len(df.columns) > 0:
        melhor_coluna = df.columns[0]
        print(f"⚠️  Nenhuma coluna com caracteres corrompidos detectada. Usando primeira coluna: '{melhor_coluna}'")
    elif melhor_coluna:
        print(f"✅ Coluna selecionada: '{melhor_coluna}' com {max_problemas} ocorrências de caracteres suspeitos.")
    return melhor_coluna

coluna_alvo = detectar_coluna_corrompida(df)
if coluna_alvo is None:
    print("❌ Nenhuma coluna disponível para correção. Abortando.")
    exit()

print(f"\n🎯 COLUNA SELECIONADA: '{coluna_alvo}'")

# ========== 4. FUNÇÃO DE CORREÇÃO INTELIGENTE ==========
def corrigir_caracteres_inteligente(texto):
    """
    Função inteligente que SUBSTITUI caracteres corrompidos pelos corretos
    em vez de apenas removê-los.
    """
    if pd.isna(texto):
        return np.nan

    texto = str(texto)

    # ETAPA 1: Tentar correção via re-encoding (UTF-8 interpretado como Latin-1)
    try:
        texto_bytes = texto.encode('latin-1', errors='ignore')
        texto = texto_bytes.decode('utf-8', errors='ignore')
    except:
        pass

    # ETAPA 2: Mapeamento DIRETO de caracteres corrompidos
    mapeamentos_exatos = {
        # Acentuação básica
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú',
        'Ã£': 'ã', 'Ãµ': 'õ', 'Ã§': 'ç', 'Ãª': 'ê', 'Ã´': 'ô',
        'Ã ': 'à', 'Ã\xa0': 'à', 'Ã¢': 'â', 'Ã¨': 'è', 'Ã«': 'ë',
        'Ã¯': 'ï', 'Ã¼': 'ü', 'Ã±': 'ñ',

        # Maiúsculas
        'Ã': 'Á', 'Ã': 'É', 'Ã': 'Í', 'Ã': 'Ó', 'Ã': 'Ú',
        'Ã': 'Ã', 'Ã': 'Ç', 'Ã': 'Õ', 'Ã': 'Ä', 'Ã': 'Ö',
        'Ã': 'Ü',

        # Caracteres especiais
        'Â°': '°', 'Âª': 'ª', 'Âº': 'º', 'Â´': '´', 'Â¨': '¨',
        'Â£': '£', 'Â¢': '¢', 'Â¥': '¥', 'â‚¬': '€',

        # Termos comuns do seu arquivo (adaptáveis)
        'nÃ£o': 'não', 'mÃ¡s': 'mas', 'pÃºblico': 'público',
        'ConvÃªnio': 'Convênio', 'ConsÃ³rcio': 'Consórcio',
        'informaÃ§Ã£o': 'informação', 'informaÃ§Ãµes': 'informações',
        'formaÃ§Ã£o': 'formação', 'localizaÃ§Ã£o': 'localização',
        'construÃ§Ã£o': 'construção', 'administraÃ§Ã£o': 'administração',
        'coordenaÃ§Ã£o': 'coordenação', 'relaÃ§Ã£o': 'relação',
        'organizaÃ§Ã£o': 'organização', 'comunicaÃ§Ã£o': 'comunicação',
        'educaÃ§Ã£o': 'educação', 'execuÃ§Ã£o': 'execução',
        'instalaÃ§Ã£o': 'instalação', 'urbanizaÃ§Ã£o': 'urbanização',
        'pavimentaÃ§Ã£o': 'pavimentação', 'proteÃ§Ã£o': 'proteção',
        'revisÃ£o': 'revisão', 'versÃ£o': 'versão',
        'dissertaÃ§Ã£o': 'dissertação', 'estatÃ­stica': 'estatística',
        'apresentaÃ§Ã£o': 'apresentação', 'colaboraÃ§Ã£o': 'colaboração',
        'orientaÃ§Ã£o': 'orientação', 'participaÃ§Ã£o': 'participação',
        'publicaÃ§Ã£o': 'publicação', 'mineraÃ§Ã£o': 'mineração',
        'ParanÃ¡': 'Paraná', 'BrasÃ­lia': 'Brasília',
        'SÃ£o': 'São', 'JoÃ£o': 'João', 'JosÃ©': 'José',
        'SebastiÃ£o': 'Sebastião', 'AntÃ´nio': 'Antônio',
        'mÃ©dio': 'médio', 'empresÃ¡rio': 'empresário',
        'tÃ©cnico': 'técnico', 'tÃ©cnica': 'técnica',
        'hidrÃ¡ulico': 'hidráulico', 'elÃ©trico': 'elétrico',
        'topogrÃ¡fico': 'topográfico', 'cartogrÃ¡fico': 'cartográfico',
        'geogrÃ¡fico': 'geográfico', 'geolÃ³gico': 'geológico',
        'ambiental': 'ambiental', 'ecolÃ³gico': 'ecológico',
        'municÃ­pio': 'município', 'regiÃ£o': 'região',
        'paÃ­s': 'país', 'Ãºnico': 'único',
    }

    for errado, correto in mapeamentos_exatos.items():
        if errado in texto:
            texto = texto.replace(errado, correto)

    # ETAPA 3: Correção de padrões com regex
    texto = re.sub(r'(\w+)Ã£o\b', r'\1ão', texto)
    texto = re.sub(r'(\w+)Ãµes\b', r'\1ões', texto)
    texto = texto.replace('Ã§Ã£o', 'ção')
    texto = texto.replace('Ã§Ãµes', 'ções')

    # ETAPA 4: Tratamento de "Ã" solitário (contextual)
    if 'Ã' in texto:
        palavras = texto.split()
        palavras_corrigidas = []
        for palavra in palavras:
            if 'Ã' in palavra:
                # Tenta substituir pelas combinações mais prováveis
                if len(palavra) > 2 and palavra[-2:] == 'Ã£o':
                    palavra = palavra[:-2] + 'ão'
                elif len(palavra) > 3 and palavra[-3:] == 'Ãµes':
                    palavra = palavra[:-3] + 'ões'
                else:
                    # Substituições diretas de caracteres
                    palavra = palavra.replace('Ã¡', 'á').replace('Ã©', 'é').replace('Ã­', 'í')
                    palavra = palavra.replace('Ã³', 'ó').replace('Ãº', 'ú').replace('Ã£', 'ã')
                    palavra = palavra.replace('Ãµ', 'õ').replace('Ã§', 'ç').replace('Ãª', 'ê')
                    palavra = palavra.replace('Ã´', 'ô')
                    # Se ainda tiver 'Ã', substitui por 'a' (fallback)
                    if 'Ã' in palavra:
                        palavra = palavra.replace('Ã', 'a')
            palavras_corrigidas.append(palavra)
        texto = ' '.join(palavras_corrigidas)

    # ETAPA 5: Limpeza final
    texto = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', ' ', texto)
    texto = texto.replace('�', '')
    texto = ' '.join(texto.split())

    if texto.strip() == '' or texto.strip().lower() == 'null':
        return np.nan
    return texto.strip()

# ========== 5. APLICAR CORREÇÃO ==========
print(f"\n🔄 APLICANDO CORREÇÃO INTELIGENTE NA COLUNA '{coluna_alvo}'...")

# Criar cópia para comparação
df_original = df.copy()
original_series = df[coluna_alvo].copy()

# Aplicar correção
df[coluna_alvo] = df[coluna_alvo].apply(corrigir_caracteres_inteligente)

print("✅ Correção aplicada!")

# ========== 6. DEMONSTRAÇÃO DAS CORREÇÕES ==========
print("\n" + "="*100)
print("📊 DEMONSTRAÇÃO DAS CORREÇÕES APLICADAS")
print("="*100)

exemplos_correcao = []
for idx in range(min(100, len(df))):
    original = original_series.iloc[idx]
    corrigido = df[coluna_alvo].iloc[idx]

    if pd.isna(original) or pd.isna(corrigido):
        continue

    original_str = str(original)
    corrigido_str = str(corrigido)

    if original_str != corrigido_str and ('Ã' in original_str or 'Â' in original_str):
        exemplos_correcao.append((idx, original_str, corrigido_str))
        if len(exemplos_correcao) >= 10:
            break

if exemplos_correcao:
    print(f"\n🎯 EXEMPLOS DE SUBSTITUIÇÕES:")
    print("-" * 120)
    for i, (idx, original, corrigido) in enumerate(exemplos_correcao):
        print(f"\n📌 Exemplo {i+1} (Linha {idx+1}):")
        print(f"   🔴 ORIGINAL:  {original[:150]}{'...' if len(original)>150 else ''}")
        print(f"   🟢 CORRIGIDO: {corrigido[:150]}{'...' if len(corrigido)>150 else ''}")
else:
    print("⚠️  Nenhuma correção significativa encontrada nas primeiras 100 linhas.")
    print("   Isso pode indicar que a coluna selecionada já está correta, é numérica ou o arquivo está vazio.")

# ========== 7. ANÁLISE DE PROBLEMAS RESIDUAIS ==========
print("\n🔍 VERIFICANDO PROBLEMAS RESIDUAIS...")
caracteres_problematicos = ['Ã', 'Â', 'â‚¬', '€', '�']
problemas_residuais = 0

for idx, texto in enumerate(df[coluna_alvo].head(200)):
    if pd.isna(texto):
        continue
    texto_str = str(texto)
    for char in caracteres_problematicos:
        if char in texto_str:
            problemas_residuais += 1
            if problemas_residuais <= 3:
                print(f"   Linha {idx+1}: caractere '{char}' em '{texto_str[:80]}...'")
            break

if problemas_residuais == 0:
    print("✅ Nenhum problema residual encontrado!")
else:
    print(f"⚠️  Encontrados {problemas_residuais} problemas residuais (exibidos os primeiros).")

# ========== 8. SALVAR ARQUIVO CORRIGIDO ==========
print(f"\n💾 SALVANDO ARQUIVO CORRIGIDO: {output_file}")
try:
    df.to_excel(output_file, index=False)
    print(f"✅ Arquivo salvo com sucesso!")
    if Path(input_file).exists() and Path(output_file).exists():
        tam_orig = Path(input_file).stat().st_size
        tam_corr = Path(output_file).stat().st_size
        print(f"📁 Tamanho original:  {tam_orig:,} bytes")
        print(f"📁 Tamanho corrigido: {tam_corr:,} bytes")
except Exception as e:
    print(f"❌ Erro ao salvar: {e}")
    csv_file = output_file.replace('.xlsx', '.csv')
    df.to_csv(csv_file, index=False, encoding='utf-8-sig')
    print(f"✅ Salvo como CSV: {csv_file}")

# ========== 9. RESUMO FINAL ==========
print("\n" + "="*80)
print("🎯 PROCESSO CONCLUÍDO!")
print("="*80)
print(f"📁 Arquivo original : {input_file}")
print(f"📁 Arquivo corrigido: {output_file}")
print(f"📊 Total de registros: {len(df)}")
print(f"📊 Coluna corrigida: '{coluna_alvo}'")
print("="*80)

# ========== BÔNUS: FUNÇÃO REUTILIZÁVEL ==========
print("\n⚡ CÓDIGO BÔNUS PARA USO FUTURO (função de correção genérica):")
print("""
def corrigir_texto_portugues(texto):
    '''Corrige texto em português com problemas de encoding (UTF-8 → Latin-1)'''
    if pd.isna(texto):
        return texto
    texto = str(texto)
    try:
        texto = texto.encode('latin-1').decode('utf-8')
    except:
        pass
    correcoes = {
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú',
        'Ã£': 'ã', 'Ãµ': 'õ', 'Ã§': 'ç', 'Ãª': 'ê', 'Ã´': 'ô',
        'Ã ': 'à', 'Ã¢': 'â', 'nÃ£o': 'não', 'mÃ¡s': 'mas',
    }
    for errado, correto in correcoes.items():
        texto = texto.replace(errado, correto)
    return texto.strip()
""")

🔧 CONFIGURANDO AMBIENTE...

📂 CARREGANDO ARQUIVO: NOTAS_GERAIS_MAPAS.xlsx
✅ Arquivo carregado com sucesso!
📊 Dimensões: 4470 linhas × 1 colunas

🔍 ESTRUTURA DO ARQUIVO:
Colunas: ['nota_geral_mapas']

🔍 IDENTIFICANDO COLUNA PARA CORRIGIR...
✅ Coluna selecionada: 'nota_geral_mapas' com 25 ocorrências de caracteres suspeitos.

🎯 COLUNA SELECIONADA: 'nota_geral_mapas'

🔄 APLICANDO CORREÇÃO INTELIGENTE NA COLUNA 'nota_geral_mapas'...
✅ Correção aplicada!

📊 DEMONSTRAÇÃO DAS CORREÇÕES APLICADAS
⚠️  Nenhuma correção significativa encontrada nas primeiras 100 linhas.
   Isso pode indicar que a coluna selecionada já está correta, é numérica ou o arquivo está vazio.

🔍 VERIFICANDO PROBLEMAS RESIDUAIS...
✅ Nenhum problema residual encontrado!

💾 SALVANDO ARQUIVO CORRIGIDO: NOTAS_GERAIS_MAPAS_CORRIGIDO.xlsx
✅ Arquivo salvo com sucesso!
📁 Tamanho original:  46,502 bytes
📁 Tamanho corrigido: 43,783 bytes

🎯 PROCESSO CONCLUÍDO!
📁 Arquivo original : NOTAS_GERAIS_MAPAS.xlsx
📁 Arquivo corrigido: NOTAS_G

# CÓDIGO DE CORREÇÃO DE CARACTERES PARA A COLUNA AQUISICAO_MAPAS

In [4]:
# CORREÇÃO DE CARACTERES - METODOS_DE_AQUISICAO_MAPAS.xlsx
# Versão adaptada para a coluna 'aquisicao_mapas'

import pandas as pd
import numpy as np
import re
from pathlib import Path

# ========== 1. CONFIGURAÇÃO ==========
print("🔧 CONFIGURANDO AMBIENTE...")
input_file = "METODOS_DE_AQUISICAO_MAPAS.xlsx"
output_file = "METODOS_DE_AQUISICAO_MAPAS_CORRIGIDO.xlsx"

# ========== 2. CARREGAR O ARQUIVO ==========
print(f"\n📂 CARREGANDO ARQUIVO: {input_file}")
try:
    df = pd.read_excel(input_file)
    print(f"✅ Arquivo carregado com sucesso!")
    print(f"📊 Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas")
    print(f"\n🔍 ESTRUTURA DO ARQUIVO:")
    print(f"Colunas: {list(df.columns)}")
    
    # Se estiver vazio, aborta
    if df.empty:
        print("⚠️  O arquivo está vazio. Nada a corrigir.")
        exit()
except Exception as e:
    print(f"❌ Erro ao carregar: {e}")
    exit()

# ========== 3. IDENTIFICAR A COLUNA COM TEXTO CORROMPIDO ==========
print("\n🔍 IDENTIFICANDO COLUNA PARA CORRIGIR...")

def detectar_coluna_corrompida(df):
    padroes_problematicos = ['Ã¡', 'Ã©', 'Ã­', 'Ã³', 'Ãº', 'Ã£', 'Ã§', 'Ãµ', 'Ãª', 'Ã´', 'Ã¼']
    melhor_coluna = None
    max_problemas = 0

    for coluna in df.columns:
        if not pd.api.types.is_string_dtype(df[coluna]) and not pd.api.types.is_object_dtype(df[coluna]):
            continue

        amostras = df[coluna].dropna().astype(str).head(50).tolist()
        texto_amostra = ' '.join(amostras)
        problemas = sum(texto_amostra.count(p) for p in padroes_problematicos)

        if problemas > max_problemas:
            max_problemas = problemas
            melhor_coluna = coluna

    if melhor_coluna is None and len(df.columns) > 0:
        melhor_coluna = df.columns[0]
        print(f"⚠️  Nenhuma coluna com caracteres corrompidos detectada. Usando primeira coluna: '{melhor_coluna}'")
    elif melhor_coluna:
        print(f"✅ Coluna selecionada: '{melhor_coluna}' com {max_problemas} ocorrências de caracteres suspeitos.")
    return melhor_coluna

coluna_alvo = detectar_coluna_corrompida(df)
if coluna_alvo is None:
    print("❌ Nenhuma coluna disponível para correção. Abortando.")
    exit()

print(f"\n🎯 COLUNA SELECIONADA: '{coluna_alvo}'")

# ========== 4. FUNÇÃO DE CORREÇÃO INTELIGENTE ==========
def corrigir_caracteres_inteligente(texto):
    if pd.isna(texto):
        return np.nan

    texto = str(texto)

    # ETAPA 1: Tentar correção via re-encoding (UTF-8 interpretado como Latin-1)
    try:
        texto_bytes = texto.encode('latin-1', errors='ignore')
        texto = texto_bytes.decode('utf-8', errors='ignore')
    except:
        pass

    # ETAPA 2: Mapeamento DIRETO de caracteres corrompidos
    mapeamentos_exatos = {
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú',
        'Ã£': 'ã', 'Ãµ': 'õ', 'Ã§': 'ç', 'Ãª': 'ê', 'Ã´': 'ô',
        'Ã ': 'à', 'Ã¢': 'â', 'Ã¨': 'è', 'Ã«': 'ë', 'Ã¯': 'ï', 'Ã¼': 'ü', 'Ã±': 'ñ',
        'Ã': 'Á', 'Ã': 'É', 'Ã': 'Í', 'Ã': 'Ó', 'Ã': 'Ú',
        'Ã': 'Ã', 'Ã': 'Ç', 'Ã': 'Õ', 'Ã': 'Ä', 'Ã': 'Ö', 'Ã': 'Ü',
        'Â°': '°', 'Âª': 'ª', 'Âº': 'º', 'Â´': '´', 'Â¨': '¨', 'Â£': '£', 'Â¢': '¢', 'Â¥': '¥', 'â‚¬': '€',
        'nÃ£o': 'não', 'mÃ¡s': 'mas', 'pÃºblico': 'público',
        'ConvÃªnio': 'Convênio', 'ConsÃ³rcio': 'Consórcio',
        'informaÃ§Ã£o': 'informação', 'informaÃ§Ãµes': 'informações',
        'formaÃ§Ã£o': 'formação', 'localizaÃ§Ã£o': 'localização',
        'construÃ§Ã£o': 'construção', 'administraÃ§Ã£o': 'administração',
        'coordenaÃ§Ã£o': 'coordenação', 'relaÃ§Ã£o': 'relação',
        'organizaÃ§Ã£o': 'organização', 'comunicaÃ§Ã£o': 'comunicação',
        'educaÃ§Ã£o': 'educação', 'execuÃ§Ã£o': 'execução',
        'instalaÃ§Ã£o': 'instalação', 'urbanizaÃ§Ã£o': 'urbanização',
        'pavimentaÃ§Ã£o': 'pavimentação', 'proteÃ§Ã£o': 'proteção',
        'revisÃ£o': 'revisão', 'versÃ£o': 'versão',
        'dissertaÃ§Ã£o': 'dissertação', 'estatÃ­stica': 'estatística',
        'apresentaÃ§Ã£o': 'apresentação', 'colaboraÃ§Ã£o': 'colaboração',
        'orientaÃ§Ã£o': 'orientação', 'participaÃ§Ã£o': 'participação',
        'publicaÃ§Ã£o': 'publicação', 'mineraÃ§Ã£o': 'mineração',
        'ParanÃ¡': 'Paraná', 'BrasÃ­lia': 'Brasília',
        'SÃ£o': 'São', 'JoÃ£o': 'João', 'JosÃ©': 'José',
        'SebastiÃ£o': 'Sebastião', 'AntÃ´nio': 'Antônio',
        'mÃ©dio': 'médio', 'empresÃ¡rio': 'empresário',
        'tÃ©cnico': 'técnico', 'tÃ©cnica': 'técnica',
        'hidrÃ¡ulico': 'hidráulico', 'elÃ©trico': 'elétrico',
        'topogrÃ¡fico': 'topográfico', 'cartogrÃ¡fico': 'cartográfico',
        'geogrÃ¡fico': 'geográfico', 'geolÃ³gico': 'geológico',
        'ambiental': 'ambiental', 'ecolÃ³gico': 'ecológico',
        'municÃ­pio': 'município', 'regiÃ£o': 'região',
        'paÃ­s': 'país', 'Ãºnico': 'único',
    }

    for errado, correto in mapeamentos_exatos.items():
        if errado in texto:
            texto = texto.replace(errado, correto)

    # ETAPA 3: Correção de padrões com regex
    texto = re.sub(r'(\w+)Ã£o\b', r'\1ão', texto)
    texto = re.sub(r'(\w+)Ãµes\b', r'\1ões', texto)
    texto = texto.replace('Ã§Ã£o', 'ção')
    texto = texto.replace('Ã§Ãµes', 'ções')

    # ETAPA 4: Tratamento de "Ã" solitário (contextual)
    if 'Ã' in texto:
        palavras = texto.split()
        palavras_corrigidas = []
        for palavra in palavras:
            if 'Ã' in palavra:
                if len(palavra) > 2 and palavra[-2:] == 'Ã£o':
                    palavra = palavra[:-2] + 'ão'
                elif len(palavra) > 3 and palavra[-3:] == 'Ãµes':
                    palavra = palavra[:-3] + 'ões'
                else:
                    palavra = palavra.replace('Ã¡', 'á').replace('Ã©', 'é').replace('Ã­', 'í')
                    palavra = palavra.replace('Ã³', 'ó').replace('Ãº', 'ú').replace('Ã£', 'ã')
                    palavra = palavra.replace('Ãµ', 'õ').replace('Ã§', 'ç').replace('Ãª', 'ê')
                    palavra = palavra.replace('Ã´', 'ô')
                    if 'Ã' in palavra:
                        palavra = palavra.replace('Ã', 'a')
            palavras_corrigidas.append(palavra)
        texto = ' '.join(palavras_corrigidas)

    # ETAPA 5: Limpeza final
    texto = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', ' ', texto)
    texto = texto.replace('�', '')
    texto = ' '.join(texto.split())

    if texto.strip() == '' or texto.strip().lower() == 'null':
        return np.nan
    return texto.strip()

# ========== 5. APLICAR CORREÇÃO ==========
print(f"\n🔄 APLICANDO CORREÇÃO INTELIGENTE NA COLUNA '{coluna_alvo}'...")

df_original = df.copy()
original_series = df[coluna_alvo].copy()

df[coluna_alvo] = df[coluna_alvo].apply(corrigir_caracteres_inteligente)

print("✅ Correção aplicada!")

# ========== 6. DEMONSTRAÇÃO DAS CORREÇÕES ==========
print("\n" + "="*100)
print("📊 DEMONSTRAÇÃO DAS CORREÇÕES APLICADAS")
print("="*100)

exemplos_correcao = []
for idx in range(min(100, len(df))):
    original = original_series.iloc[idx]
    corrigido = df[coluna_alvo].iloc[idx]

    if pd.isna(original) or pd.isna(corrigido):
        continue

    original_str = str(original)
    corrigido_str = str(corrigido)

    if original_str != corrigido_str and ('Ã' in original_str or 'Â' in original_str):
        exemplos_correcao.append((idx, original_str, corrigido_str))
        if len(exemplos_correcao) >= 10:
            break

if exemplos_correcao:
    print(f"\n🎯 EXEMPLOS DE SUBSTITUIÇÕES:")
    print("-" * 120)
    for i, (idx, original, corrigido) in enumerate(exemplos_correcao):
        print(f"\n📌 Exemplo {i+1} (Linha {idx+1}):")
        print(f"   🔴 ORIGINAL:  {original[:150]}{'...' if len(original)>150 else ''}")
        print(f"   🟢 CORRIGIDO: {corrigido[:150]}{'...' if len(corrigido)>150 else ''}")
else:
    print("⚠️  Nenhuma correção significativa encontrada nas primeiras 100 linhas.")
    print("   Isso indica que a coluna selecionada já está correta ou não contém caracteres corrompidos.")

# ========== 7. ANÁLISE DE PROBLEMAS RESIDUAIS ==========
print("\n🔍 VERIFICANDO PROBLEMAS RESIDUAIS...")
caracteres_problematicos = ['Ã', 'Â', 'â‚¬', '€', '�']
problemas_residuais = 0

for idx, texto in enumerate(df[coluna_alvo].head(200)):
    if pd.isna(texto):
        continue
    texto_str = str(texto)
    for char in caracteres_problematicos:
        if char in texto_str:
            problemas_residuais += 1
            if problemas_residuais <= 3:
                print(f"   Linha {idx+1}: caractere '{char}' em '{texto_str[:80]}...'")
            break

if problemas_residuais == 0:
    print("✅ Nenhum problema residual encontrado!")
else:
    print(f"⚠️  Encontrados {problemas_residuais} problemas residuais (exibidos os primeiros).")

# ========== 8. SALVAR ARQUIVO CORRIGIDO ==========
print(f"\n💾 SALVANDO ARQUIVO CORRIGIDO: {output_file}")
try:
    df.to_excel(output_file, index=False)
    print(f"✅ Arquivo salvo com sucesso!")
    if Path(input_file).exists() and Path(output_file).exists():
        tam_orig = Path(input_file).stat().st_size
        tam_corr = Path(output_file).stat().st_size
        print(f"📁 Tamanho original:  {tam_orig:,} bytes")
        print(f"📁 Tamanho corrigido: {tam_corr:,} bytes")
except Exception as e:
    print(f"❌ Erro ao salvar: {e}")
    csv_file = output_file.replace('.xlsx', '.csv')
    df.to_csv(csv_file, index=False, encoding='utf-8-sig')
    print(f"✅ Salvo como CSV: {csv_file}")

# ========== 9. RESUMO FINAL ==========
print("\n" + "="*80)
print("🎯 PROCESSO CONCLUÍDO!")
print("="*80)
print(f"📁 Arquivo original : {input_file}")
print(f"📁 Arquivo corrigido: {output_file}")
print(f"📊 Total de registros: {len(df)}")
print(f"📊 Coluna corrigida: '{coluna_alvo}'")
print("="*80)

# ========== BÔNUS: FUNÇÃO REUTILIZÁVEL ==========
print("\n⚡ CÓDIGO BÔNUS PARA USO FUTURO (função de correção genérica):")
print("""
def corrigir_texto_portugues(texto):
    '''Corrige texto em português com problemas de encoding (UTF-8 → Latin-1)'''
    if pd.isna(texto):
        return texto
    texto = str(texto)
    try:
        texto = texto.encode('latin-1').decode('utf-8')
    except:
        pass
    correcoes = {
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú',
        'Ã£': 'ã', 'Ãµ': 'õ', 'Ã§': 'ç', 'Ãª': 'ê', 'Ã´': 'ô',
        'Ã ': 'à', 'Ã¢': 'â', 'nÃ£o': 'não', 'mÃ¡s': 'mas',
    }
    for errado, correto in correcoes.items():
        texto = texto.replace(errado, correto)
    return texto.strip()
""")

🔧 CONFIGURANDO AMBIENTE...

📂 CARREGANDO ARQUIVO: METODOS_DE_AQUISICAO_MAPAS.xlsx
✅ Arquivo carregado com sucesso!
📊 Dimensões: 4470 linhas × 1 colunas

🔍 ESTRUTURA DO ARQUIVO:
Colunas: ['aquisicao_mapas']

🔍 IDENTIFICANDO COLUNA PARA CORRIGIR...
⚠️  Nenhuma coluna com caracteres corrompidos detectada. Usando primeira coluna: 'aquisicao_mapas'

🎯 COLUNA SELECIONADA: 'aquisicao_mapas'

🔄 APLICANDO CORREÇÃO INTELIGENTE NA COLUNA 'aquisicao_mapas'...
✅ Correção aplicada!

📊 DEMONSTRAÇÃO DAS CORREÇÕES APLICADAS
⚠️  Nenhuma correção significativa encontrada nas primeiras 100 linhas.
   Isso indica que a coluna selecionada já está correta ou não contém caracteres corrompidos.

🔍 VERIFICANDO PROBLEMAS RESIDUAIS...
✅ Nenhum problema residual encontrado!

💾 SALVANDO ARQUIVO CORRIGIDO: METODOS_DE_AQUISICAO_MAPAS_CORRIGIDO.xlsx
✅ Arquivo salvo com sucesso!
📁 Tamanho original:  36,124 bytes
📁 Tamanho corrigido: 32,307 bytes

🎯 PROCESSO CONCLUÍDO!
📁 Arquivo original : METODOS_DE_AQUISICAO_MAPAS.xls

# CÓDIGO DE CORREÇÃO DE CARACTERES PARA A COLUNA METODO_ELABORACAO_MAPAS

In [5]:
# CORREÇÃO DE CARACTERES - METODO_ELABORACAO_MAPAS.xlsx
# Versão adaptada para a coluna 'metodo_elaboracao_mapas'

import pandas as pd
import numpy as np
import re
from pathlib import Path

# ========== 1. CONFIGURAÇÃO ==========
print("🔧 CONFIGURANDO AMBIENTE...")
input_file = "METODO_ELABORACAO_MAPAS.xlsx"
output_file = "METODO_ELABORACAO_MAPAS_CORRIGIDO.xlsx"

# ========== 2. CARREGAR O ARQUIVO ==========
print(f"\n📂 CARREGANDO ARQUIVO: {input_file}")
try:
    df = pd.read_excel(input_file)
    print(f"✅ Arquivo carregado com sucesso!")
    print(f"📊 Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas")
    print(f"\n🔍 ESTRUTURA DO ARQUIVO:")
    print(f"Colunas: {list(df.columns)}")

    if df.empty:
        print("⚠️  O arquivo está vazio. Nada a corrigir.")
        exit()
except Exception as e:
    print(f"❌ Erro ao carregar: {e}")
    exit()

# ========== 3. IDENTIFICAR A COLUNA COM TEXTO CORROMPIDO ==========
print("\n🔍 IDENTIFICANDO COLUNA PARA CORRIGIR...")

def detectar_coluna_corrompida(df):
    padroes_problematicos = ['Ã¡', 'Ã©', 'Ã­', 'Ã³', 'Ãº', 'Ã£', 'Ã§', 'Ãµ', 'Ãª', 'Ã´', 'Ã¼']
    melhor_coluna = None
    max_problemas = 0

    for coluna in df.columns:
        if not pd.api.types.is_string_dtype(df[coluna]) and not pd.api.types.is_object_dtype(df[coluna]):
            continue

        amostras = df[coluna].dropna().astype(str).head(50).tolist()
        texto_amostra = ' '.join(amostras)
        problemas = sum(texto_amostra.count(p) for p in padroes_problematicos)

        if problemas > max_problemas:
            max_problemas = problemas
            melhor_coluna = coluna

    if melhor_coluna is None and len(df.columns) > 0:
        melhor_coluna = df.columns[0]
        print(f"⚠️  Nenhuma coluna com caracteres corrompidos detectada. Usando primeira coluna: '{melhor_coluna}'")
    elif melhor_coluna:
        print(f"✅ Coluna selecionada: '{melhor_coluna}' com {max_problemas} ocorrências de caracteres suspeitos.")
    return melhor_coluna

coluna_alvo = detectar_coluna_corrompida(df)
if coluna_alvo is None:
    print("❌ Nenhuma coluna disponível para correção. Abortando.")
    exit()

print(f"\n🎯 COLUNA SELECIONADA: '{coluna_alvo}'")

# ========== 4. FUNÇÃO DE CORREÇÃO INTELIGENTE ==========
def corrigir_caracteres_inteligente(texto):
    if pd.isna(texto):
        return np.nan

    texto = str(texto)

    # ETAPA 1: Tentar correção via re-encoding (UTF-8 interpretado como Latin-1)
    try:
        texto_bytes = texto.encode('latin-1', errors='ignore')
        texto = texto_bytes.decode('utf-8', errors='ignore')
    except:
        pass

    # ETAPA 2: Mapeamento DIRETO de caracteres corrompidos
    mapeamentos_exatos = {
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú',
        'Ã£': 'ã', 'Ãµ': 'õ', 'Ã§': 'ç', 'Ãª': 'ê', 'Ã´': 'ô',
        'Ã ': 'à', 'Ã¢': 'â', 'Ã¨': 'è', 'Ã«': 'ë', 'Ã¯': 'ï', 'Ã¼': 'ü', 'Ã±': 'ñ',
        'Ã': 'Á', 'Ã': 'É', 'Ã': 'Í', 'Ã': 'Ó', 'Ã': 'Ú',
        'Ã': 'Ã', 'Ã': 'Ç', 'Ã': 'Õ', 'Ã': 'Ä', 'Ã': 'Ö', 'Ã': 'Ü',
        'Â°': '°', 'Âª': 'ª', 'Âº': 'º', 'Â´': '´', 'Â¨': '¨', 'Â£': '£', 'Â¢': '¢', 'Â¥': '¥', 'â‚¬': '€',
        'nÃ£o': 'não', 'mÃ¡s': 'mas', 'pÃºblico': 'público',
        'ConvÃªnio': 'Convênio', 'ConsÃ³rcio': 'Consórcio',
        'informaÃ§Ã£o': 'informação', 'informaÃ§Ãµes': 'informações',
        'formaÃ§Ã£o': 'formação', 'localizaÃ§Ã£o': 'localização',
        'construÃ§Ã£o': 'construção', 'administraÃ§Ã£o': 'administração',
        'coordenaÃ§Ã£o': 'coordenação', 'relaÃ§Ã£o': 'relação',
        'organizaÃ§Ã£o': 'organização', 'comunicaÃ§Ã£o': 'comunicação',
        'educaÃ§Ã£o': 'educação', 'execuÃ§Ã£o': 'execução',
        'instalaÃ§Ã£o': 'instalação', 'urbanizaÃ§Ã£o': 'urbanização',
        'pavimentaÃ§Ã£o': 'pavimentação', 'proteÃ§Ã£o': 'proteção',
        'revisÃ£o': 'revisão', 'versÃ£o': 'versão',
        'dissertaÃ§Ã£o': 'dissertação', 'estatÃ­stica': 'estatística',
        'apresentaÃ§Ã£o': 'apresentação', 'colaboraÃ§Ã£o': 'colaboração',
        'orientaÃ§Ã£o': 'orientação', 'participaÃ§Ã£o': 'participação',
        'publicaÃ§Ã£o': 'publicação', 'mineraÃ§Ã£o': 'mineração',
        'ParanÃ¡': 'Paraná', 'BrasÃ­lia': 'Brasília',
        'SÃ£o': 'São', 'JoÃ£o': 'João', 'JosÃ©': 'José',
        'SebastiÃ£o': 'Sebastião', 'AntÃ´nio': 'Antônio',
        'mÃ©dio': 'médio', 'empresÃ¡rio': 'empresário',
        'tÃ©cnico': 'técnico', 'tÃ©cnica': 'técnica',
        'hidrÃ¡ulico': 'hidráulico', 'elÃ©trico': 'elétrico',
        'topogrÃ¡fico': 'topográfico', 'cartogrÃ¡fico': 'cartográfico',
        'geogrÃ¡fico': 'geográfico', 'geolÃ³gico': 'geológico',
        'ambiental': 'ambiental', 'ecolÃ³gico': 'ecológico',
        'municÃ­pio': 'município', 'regiÃ£o': 'região',
        'paÃ­s': 'país', 'Ãºnico': 'único',
        # Termos específicos deste arquivo
        'RestituiÃ§Ã£o': 'Restituição',
        'fotogramÃ©trica': 'fotogramétrica',
        'aerotriangulaÃ§Ã£o': 'aerotriangulação',
        'nÃ­veis': 'níveis',
        'Ã¡gua': 'água',
        'Ã¡guas': 'águas',
        'Ã¡reas': 'áreas',
        'identificaÃ§Ã£o': 'identificação',
        'delimitaÃ§Ã£o': 'delimitação',
        'delimitaÃ§Ãµes': 'delimitações',
        'divisores': 'divisores',
        'apartir': 'a partir',  # erro comum, mas não é encoding, opcional
        'Ã  lÃ¡pis': 'a lápis',
        'reambulaÃ§Ã£o': 'reambulação',
        'aerofotogramÃ©trico': 'aerofotogramétrico',
        'aerofotos': 'aerofotos',
        'escala': 'escala',
        'apoio de campo': 'apoio de campo',
        'planialtimÃ©tricas': 'planialtimétricas',
        'cartas': 'cartas',
        'COMEC': 'COMEC',
        'levantamento': 'levantamento',
        'aerofotogrametria': 'aerofotogrametria',
        'restituiÃ§Ã£o': 'restituição',
    }

    for errado, correto in mapeamentos_exatos.items():
        if errado in texto:
            texto = texto.replace(errado, correto)

    # ETAPA 3: Correção de padrões com regex
    texto = re.sub(r'(\w+)Ã£o\b', r'\1ão', texto)
    texto = re.sub(r'(\w+)Ãµes\b', r'\1ões', texto)
    texto = texto.replace('Ã§Ã£o', 'ção')
    texto = texto.replace('Ã§Ãµes', 'ções')

    # ETAPA 4: Tratamento de "Ã" solitário (contextual)
    if 'Ã' in texto:
        palavras = texto.split()
        palavras_corrigidas = []
        for palavra in palavras:
            if 'Ã' in palavra:
                if len(palavra) > 2 and palavra[-2:] == 'Ã£o':
                    palavra = palavra[:-2] + 'ão'
                elif len(palavra) > 3 and palavra[-3:] == 'Ãµes':
                    palavra = palavra[:-3] + 'ões'
                else:
                    palavra = palavra.replace('Ã¡', 'á').replace('Ã©', 'é').replace('Ã­', 'í')
                    palavra = palavra.replace('Ã³', 'ó').replace('Ãº', 'ú').replace('Ã£', 'ã')
                    palavra = palavra.replace('Ãµ', 'õ').replace('Ã§', 'ç').replace('Ãª', 'ê')
                    palavra = palavra.replace('Ã´', 'ô')
                    if 'Ã' in palavra:
                        palavra = palavra.replace('Ã', 'a')
            palavras_corrigidas.append(palavra)
        texto = ' '.join(palavras_corrigidas)

    # ETAPA 5: Limpeza final
    texto = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', ' ', texto)
    texto = texto.replace('�', '')
    texto = ' '.join(texto.split())

    if texto.strip() == '' or texto.strip().lower() == 'null':
        return np.nan
    return texto.strip()

# ========== 5. APLICAR CORREÇÃO ==========
print(f"\n🔄 APLICANDO CORREÇÃO INTELIGENTE NA COLUNA '{coluna_alvo}'...")

df_original = df.copy()
original_series = df[coluna_alvo].copy()

df[coluna_alvo] = df[coluna_alvo].apply(corrigir_caracteres_inteligente)

print("✅ Correção aplicada!")

# ========== 6. DEMONSTRAÇÃO DAS CORREÇÕES ==========
print("\n" + "="*100)
print("📊 DEMONSTRAÇÃO DAS CORREÇÕES APLICADAS")
print("="*100)

exemplos_correcao = []
for idx in range(min(100, len(df))):
    original = original_series.iloc[idx]
    corrigido = df[coluna_alvo].iloc[idx]

    if pd.isna(original) or pd.isna(corrigido):
        continue

    original_str = str(original)
    corrigido_str = str(corrigido)

    if original_str != corrigido_str and ('Ã' in original_str or 'Â' in original_str):
        exemplos_correcao.append((idx, original_str, corrigido_str))
        if len(exemplos_correcao) >= 10:
            break

if exemplos_correcao:
    print(f"\n🎯 EXEMPLOS DE SUBSTITUIÇÕES:")
    print("-" * 120)
    for i, (idx, original, corrigido) in enumerate(exemplos_correcao):
        print(f"\n📌 Exemplo {i+1} (Linha {idx+1}):")
        print(f"   🔴 ORIGINAL:  {original[:150]}{'...' if len(original)>150 else ''}")
        print(f"   🟢 CORRIGIDO: {corrigido[:150]}{'...' if len(corrigido)>150 else ''}")
else:
    print("⚠️  Nenhuma correção significativa encontrada nas primeiras 100 linhas.")
    print("   Isso indica que a coluna selecionada já está correta ou não contém caracteres corrompidos.")

# ========== 7. ANÁLISE DE PROBLEMAS RESIDUAIS ==========
print("\n🔍 VERIFICANDO PROBLEMAS RESIDUAIS...")
caracteres_problematicos = ['Ã', 'Â', 'â‚¬', '€', '�']
problemas_residuais = 0

for idx, texto in enumerate(df[coluna_alvo].head(200)):
    if pd.isna(texto):
        continue
    texto_str = str(texto)
    for char in caracteres_problematicos:
        if char in texto_str:
            problemas_residuais += 1
            if problemas_residuais <= 3:
                print(f"   Linha {idx+1}: caractere '{char}' em '{texto_str[:80]}...'")
            break

if problemas_residuais == 0:
    print("✅ Nenhum problema residual encontrado!")
else:
    print(f"⚠️  Encontrados {problemas_residuais} problemas residuais (exibidos os primeiros).")

# ========== 8. SALVAR ARQUIVO CORRIGIDO ==========
print(f"\n💾 SALVANDO ARQUIVO CORRIGIDO: {output_file}")
try:
    df.to_excel(output_file, index=False)
    print(f"✅ Arquivo salvo com sucesso!")
    if Path(input_file).exists() and Path(output_file).exists():
        tam_orig = Path(input_file).stat().st_size
        tam_corr = Path(output_file).stat().st_size
        print(f"📁 Tamanho original:  {tam_orig:,} bytes")
        print(f"📁 Tamanho corrigido: {tam_corr:,} bytes")
except Exception as e:
    print(f"❌ Erro ao salvar: {e}")
    csv_file = output_file.replace('.xlsx', '.csv')
    df.to_csv(csv_file, index=False, encoding='utf-8-sig')
    print(f"✅ Salvo como CSV: {csv_file}")

# ========== 9. RESUMO FINAL ==========
print("\n" + "="*80)
print("🎯 PROCESSO CONCLUÍDO!")
print("="*80)
print(f"📁 Arquivo original : {input_file}")
print(f"📁 Arquivo corrigido: {output_file}")
print(f"📊 Total de registros: {len(df)}")
print(f"📊 Coluna corrigida: '{coluna_alvo}'")
print("="*80)

# ========== BÔNUS: FUNÇÃO REUTILIZÁVEL ==========
print("\n⚡ CÓDIGO BÔNUS PARA USO FUTURO (função de correção genérica):")
print("""
def corrigir_texto_portugues(texto):
    '''Corrige texto em português com problemas de encoding (UTF-8 → Latin-1)'''
    if pd.isna(texto):
        return texto
    texto = str(texto)
    try:
        texto = texto.encode('latin-1').decode('utf-8')
    except:
        pass
    correcoes = {
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú',
        'Ã£': 'ã', 'Ãµ': 'õ', 'Ã§': 'ç', 'Ãª': 'ê', 'Ã´': 'ô',
        'Ã ': 'à', 'Ã¢': 'â', 'nÃ£o': 'não', 'mÃ¡s': 'mas',
    }
    for errado, correto in correcoes.items():
        texto = texto.replace(errado, correto)
    return texto.strip()
""")

🔧 CONFIGURANDO AMBIENTE...

📂 CARREGANDO ARQUIVO: METODO_ELABORACAO_MAPAS.xlsx
✅ Arquivo carregado com sucesso!
📊 Dimensões: 4470 linhas × 1 colunas

🔍 ESTRUTURA DO ARQUIVO:
Colunas: ['metodo_elaboracao_mapas']

🔍 IDENTIFICANDO COLUNA PARA CORRIGIR...
✅ Coluna selecionada: 'metodo_elaboracao_mapas' com 250 ocorrências de caracteres suspeitos.

🎯 COLUNA SELECIONADA: 'metodo_elaboracao_mapas'

🔄 APLICANDO CORREÇÃO INTELIGENTE NA COLUNA 'metodo_elaboracao_mapas'...
✅ Correção aplicada!

📊 DEMONSTRAÇÃO DAS CORREÇÕES APLICADAS

🎯 EXEMPLOS DE SUBSTITUIÇÕES:
------------------------------------------------------------------------------------------------------------------------

📌 Exemplo 1 (Linha 1):
   🔴 ORIGINAL:  RestituiÃ§Ã£o fotogramÃ©trica com base em aerofotos 1:40.000 de 1976, com aerotriangulaÃ§Ã£o  e apoio de campo. 
   🟢 CORRIGIDO: Restituição fotogramétrica com base em aerofotos 1:40.000 de 1976, com aerotriangulação e apoio de campo.

📌 Exemplo 2 (Linha 2):
   🔴 ORIGINAL:  Restit

# CÓDIGO DE ATOMIZAÇÃO DE AUTORES CORPORATIVOS DE MAPAS

In [3]:
import pandas as pd
import numpy as np
import os

def identificar_separador_autor(valor):
    """
    Identifica separadores em valores multivalorados de autores.
    Retorna o separador encontrado ou None se não for multivalorado.
    """
    if pd.isna(valor) or str(valor).strip() == '':
        return None
    
    valor_str = str(valor).strip()
    
    # Trata explicitamente o valor "NULL"
    if valor_str.upper() == 'NULL':
        return None
    
    # Verifica separadores na ordem de prioridade
    if " --- " in valor_str:  # Padrão principal: com espaços
        return " --- "
    elif "---" in valor_str:  # Padrão secundário: sem espaços
        return "---"
    
    return None

def extrair_prefixo_autor(valor):
    """
    Extrai prefixos específicos para autores institucionais.
    Retorna (prefixo, valor_sem_prefixo)
    """
    if pd.isna(valor):
        return None, valor
    
    valor_str = str(valor).strip()
    
    # Lista de prefixos comuns em autores corporativos
    prefixos = [
        "Coordenação da",
        "Prefeitura Municipal de",
        "Prefeitura municipal de",
        "Secretaria de",
        "Secretaria do",
        "Secretaria da",
        "Instituto",
        "Companhia",
        "Universidade",
        "Ministério",
        "Departamento",
        "Superintendência",
        "Serviço",
        "Empresa",
        "Fundação"
    ]
    
    for prefixo in prefixos:
        if valor_str.lower().startswith(prefixo.lower()):
            # Se encontrar o prefixo, retorna ele e o restante
            resto = valor_str[len(prefixo):].strip()
            if resto and resto[0] in [',', 'de', 'da', 'do', '-']:
                resto = resto[1:].strip() if len(resto) > 1 else resto
            return prefixo, resto
    
    return None, valor_str

def processar_autores_multivalorados(arquivo_excel, coluna_autor='autor_corporativo_mapas'):
    """
    Processa a planilha de autores corporativos de mapas, 
    desnormalizando valores multivalorados.
    
    Args:
        arquivo_excel: Caminho do arquivo Excel
        coluna_autor: Nome da coluna com os autores corporativos
    
    Returns:
        DataFrame com autores desnormalizados
    """
    
    # Verificar se o arquivo existe
    if not os.path.exists(arquivo_excel):
        print(f"❌ Arquivo não encontrado: {arquivo_excel}")
        return None
    
    # Carregar o arquivo
    try:
        df = pd.read_excel(arquivo_excel)
        print(f"✅ Arquivo carregado: {arquivo_excel}")
        print(f"   Total de linhas: {len(df)}")
    except Exception as e:
        print(f"❌ Erro ao carregar arquivo: {e}")
        return None
    
    # Verificar se a coluna existe
    if coluna_autor not in df.columns:
        print(f"⚠️  A coluna '{coluna_autor}' não foi encontrada no arquivo.")
        print(f"   Colunas disponíveis: {list(df.columns)}")
        
        # Se houver apenas uma coluna, usar ela
        if len(df.columns) == 1:
            coluna_autor = df.columns[0]
            print(f"   Usando a coluna disponível: '{coluna_autor}'")
        else:
            return None
    
    # Criar ID único para cada linha original
    df['id_original'] = df.index + 1
    
    # Lista para armazenar resultados
    dados_desnormalizados = []
    
    # Estatísticas para análise
    total_multivalorados = 0
    total_vazios = 0
    
    for idx, row in df.iterrows():
        id_original = row['id_original']
        valor = row[coluna_autor]
        linha_excel = idx + 2  # +2 porque Excel começa em 1 e tem cabeçalho
        
        # CASO 1: Valor NULL ou vazio
        if pd.isna(valor) or str(valor).strip() == '' or str(valor).strip().upper() == 'NULL':
            total_vazios += 1
            dados_desnormalizados.append({
                'id_original': id_original,
                'linha_excel_original': linha_excel,
                'autor_original': '',
                'autor_desnormalizado': '',
                'prefixo': None,
                'tipo': 'vazio',
                'separador': None,
                'qtd_autores': 0
            })
            continue
        
        valor_str = str(valor).strip()
        
        # Extrair prefixo se existir (apenas para identificação, mas manteremos o valor original)
        prefixo, valor_sem_prefixo = extrair_prefixo_autor(valor_str)
        
        # Verificar se é multivalorado
        separador = identificar_separador_autor(valor_str)  # Usar valor_str original, não valor_sem_prefixo
        
        # CASO 2: Valor UNIVALORADO (sem separador ---)
        if not separador:
            dados_desnormalizados.append({
                'id_original': id_original,
                'linha_excel_original': linha_excel,
                'autor_original': valor_str,
                'autor_desnormalizado': valor_str,  # Manter o valor completo com prefixo
                'prefixo': prefixo,
                'tipo': 'único',
                'separador': None,
                'qtd_autores': 1
            })
            continue
        
        # CASO 3: Valor MULTIVALORADO (com separador ---)
        total_multivalorados += 1
        # Separar os autores usando o valor original
        autores = [a.strip() for a in valor_str.split(separador) if a.strip()]
        
        # Para cada autor, criar uma linha
        for i, autor in enumerate(autores, 1):
            # Extrair prefixo do autor individual (apenas para análise)
            prefixo_individual, _ = extrair_prefixo_autor(autor)
            
            dados_desnormalizados.append({
                'id_original': id_original,
                'linha_excel_original': linha_excel,
                'autor_original': valor_str,
                'autor_desnormalizado': autor,  # Manter o autor com seu prefixo original
                'prefixo': prefixo_individual if prefixo_individual else prefixo,
                'tipo': 'múltiplo',
                'separador': separador,
                'qtd_autores': len(autores),
                'posicao_na_lista': i
            })
    
    # Criar DataFrame
    df_desnormalizado = pd.DataFrame(dados_desnormalizados)
    
    print(f"\n📊 RESUMO DO PROCESSAMENTO:")
    print(f"   • Linhas processadas: {len(df)}")
    print(f"   • Registros vazios/NULL: {total_vazios}")
    print(f"   • Registros multivalorados encontrados: {total_multivalorados}")
    print(f"   • Total registros após desnormalização: {len(df_desnormalizado)}")
    
    return df_desnormalizado

def analisar_autores(df_desnormalizado):
    """
    Analisa estatísticas dos autores processados.
    """
    print("\n" + "="*70)
    print("📈 ANÁLISE DOS AUTORES CORPORATIVOS - MAPAS")
    print("="*70)
    
    total_linhas_originais = df_desnormalizado['id_original'].nunique()
    total_registros = len(df_desnormalizado)
    
    # Estatísticas por tipo
    vazios = len(df_desnormalizado[df_desnormalizado['tipo'] == 'vazio'])
    unicos = len(df_desnormalizado[df_desnormalizado['tipo'] == 'único'])
    multiplos = len(df_desnormalizado[df_desnormalizado['tipo'] == 'múltiplo'])
    
    # Autores únicos identificados (excluindo vazios)
    autores_unicos_df = df_desnormalizado[df_desnormalizado['autor_desnormalizado'] != '']
    qtd_autores_unicos = autores_unicos_df['autor_desnormalizado'].nunique()
    
    print(f"\n📊 ESTATÍSTICAS GERAIS:")
    print(f"  • Linhas originais: {total_linhas_originais}")
    print(f"  • Registros após desnormalização: {total_registros}")
    print(f"  • Registros vazios/NULL: {vazios}")
    print(f"  • Autores únicos (originais): {unicos}")
    print(f"  • Autores de listas múltiplas: {multiplos}")
    print(f"  • Autores distintos identificados: {qtd_autores_unicos}")
    
    # Análise de prefixos
    print(f"\n🏷️  DISTRIBUIÇÃO DE PREFIXOS:")
    prefixos_counts = df_desnormalizado['prefixo'].value_counts(dropna=False).head(15)
    for prefixo, count in prefixos_counts.items():
        if pd.isna(prefixo):
            print(f"  • Sem prefixo identificado: {count} ({count/total_registros*100:.1f}%)")
        else:
            print(f"  • {prefixo}: {count} ({count/total_registros*100:.1f}%)")
    
    # Top 20 autores mais frequentes
    print(f"\n🏆 TOP 20 AUTORES MAIS FREQUENTES:")
    top_20 = autores_unicos_df['autor_desnormalizado'].value_counts().head(20)
    for i, (autor, count) in enumerate(top_20.items(), 1):
        # Abreviar autores muito longos para exibição
        autor_exib = autor if len(autor) <= 60 else autor[:57] + "..."
        print(f"  {i:2d}. {autor_exib:<60} ({count} ocorrências)")
    
    # Análise de variações do mesmo autor
    print(f"\n🔄 POSSÍVEIS VARIAÇÕES DO MESMO AUTOR:")
    
    # Agrupar por termos comuns
    termos_busca = ['COMEC', 'SANEPAR', 'MINEROPAR', 'IPPUC', 'IBGE', 'UFPR', 'Prefeitura']
    
    for termo in termos_busca:
        variacoes = autores_unicos_df[autores_unicos_df['autor_desnormalizado'].str.contains(termo, case=False, na=False)]
        if len(variacoes) > 1:
            print(f"\n  • Variações para '{termo}':")
            autores_termo = variacoes['autor_desnormalizado'].unique()
            for autor_var in autores_termo[:5]:  # Mostrar até 5 variações
                print(f"    - {autor_var}")
            if len(autores_termo) > 5:
                print(f"    ... e mais {len(autores_termo)-5} variações")
    
    # Análise de listas multivaloradas (se houver)
    multivalorados = df_desnormalizado[df_desnormalizado['tipo'] == 'múltiplo']
    if not multivalorados.empty:
        print(f"\n🔗 LISTAS MULTIVALORADAS:")
        print(f"  • Total de listas: {multivalorados['id_original'].nunique()}")
        
        # Distribuição por tamanho de lista
        tamanhos = multivalorados.groupby('id_original')['qtd_autores'].first().value_counts().sort_index()
        print(f"  • Distribuição por tamanho:")
        for tamanho, count in tamanhos.items():
            print(f"    - Listas com {tamanho} autores: {count}")
    else:
        print(f"\n🔗 LISTAS MULTIVALORADAS:")
        print("  • Nenhuma lista multivalorada encontrada neste arquivo")

def exportar_autores_resultados(df_desnormalizado, nome_arquivo='AUTORES_MAPAS_DESNORMALIZADOS.xlsx'):
    """
    Exporta os resultados para Excel com múltiplas abas.
    """
    try:
        with pd.ExcelWriter(nome_arquivo, engine='openpyxl') as writer:
            # 1. Dados completos desnormalizados
            df_desnormalizado.to_excel(writer, sheet_name='Dados Completos', index=False)
            
            # 2. Estatísticas gerais
            percentual_preenchimento = (len(df_desnormalizado[df_desnormalizado['autor_desnormalizado'] != '']) / len(df_desnormalizado) * 100)
            
            stats = {
                'Métrica': [
                    'Linhas originais',
                    'Registros desnormalizados',
                    'Registros vazios/NULL',
                    'Autores únicos (original)',
                    'Autores de listas múltiplas',
                    'Autores distintos (catálogo)',
                    'Listas multivaloradas',
                    'Percentual de preenchimento'
                ],
                'Valor': [
                    df_desnormalizado['id_original'].nunique(),
                    len(df_desnormalizado),
                    len(df_desnormalizado[df_desnormalizado['tipo'] == 'vazio']),
                    len(df_desnormalizado[df_desnormalizado['tipo'] == 'único']),
                    len(df_desnormalizado[df_desnormalizado['tipo'] == 'múltiplo']),
                    df_desnormalizado[df_desnormalizado['autor_desnormalizado'] != '']['autor_desnormalizado'].nunique(),
                    df_desnormalizado[df_desnormalizado['tipo'] == 'múltiplo']['id_original'].nunique(),
                    f"{percentual_preenchimento:.1f}%"
                ]
            }
            
            stats_df = pd.DataFrame(stats)
            stats_df.to_excel(writer, sheet_name='Estatísticas', index=False)
            
            # 3. Autores únicos (catálogo) - ordenado
            autores_unicos = df_desnormalizado[['autor_desnormalizado']].drop_duplicates()
            autores_unicos = autores_unicos[autores_unicos['autor_desnormalizado'] != '']
            autores_unicos = autores_unicos.sort_values('autor_desnormalizado')
            autores_unicos.to_excel(writer, sheet_name='Catálogo Autores', index=False)
            
            # 4. Distribuição de prefixos
            prefixos_df = df_desnormalizado['prefixo'].value_counts(dropna=False).reset_index()
            prefixos_df.columns = ['Prefixo', 'Quantidade']
            prefixos_df['Percentual'] = (prefixos_df['Quantidade'] / len(df_desnormalizado) * 100).round(1)
            prefixos_df.to_excel(writer, sheet_name='Prefixos', index=False)
            
            # 5. Frequência de autores
            frequencia = df_desnormalizado['autor_desnormalizado'].value_counts().reset_index()
            frequencia.columns = ['Autor', 'Frequência']
            frequencia = frequencia[frequencia['Autor'] != '']
            frequencia['Percentual'] = (frequencia['Frequência'] / len(df_desnormalizado) * 100).round(1)
            frequencia.to_excel(writer, sheet_name='Frequência Autores', index=False)
            
            # 6. Listas multivaloradas (se houver)
            multivalorados = df_desnormalizado[df_desnormalizado['tipo'] == 'múltiplo']
            if not multivalorados.empty:
                listas_agrupadas = multivalorados.groupby('id_original').agg({
                    'linha_excel_original': 'first',
                    'autor_original': 'first',
                    'prefixo': 'first',
                    'qtd_autores': 'first',
                    'autor_desnormalizado': lambda x: ' | '.join(x)
                }).reset_index()
                
                listas_agrupadas = listas_agrupadas.sort_values('qtd_autores', ascending=False)
                listas_agrupadas.to_excel(writer, sheet_name='Listas Multivaloradas', index=False)
            
            # 7. Resumo por tipo de instituição (análise adicional)
            # Criar categorias baseadas em palavras-chave
            df_temp = df_desnormalizado[df_desnormalizado['autor_desnormalizado'] != ''].copy()
            
            def categorizar_instituicao(autor):
                autor_lower = str(autor).lower()
                if 'prefeitura' in autor_lower or 'municipal' in autor_lower:
                    return 'Prefeitura Municipal'
                elif 'comec' in autor_lower:
                    return 'COMEC'
                elif 'sanepar' in autor_lower:
                    return 'SANEPAR'
                elif 'mineropar' in autor_lower:
                    return 'MINEROPAR'
                elif 'ippuc' in autor_lower:
                    return 'IPPUC'
                elif 'ufpr' in autor_lower or 'universidade federal' in autor_lower:
                    return 'Universidade'
                elif 'ibge' in autor_lower:
                    return 'IBGE'
                elif 'secretaria' in autor_lower or 'sepl' in autor_lower:
                    return 'Secretaria Estadual'
                elif 'ministério' in autor_lower or 'ministerio' in autor_lower:
                    return 'Órgão Federal'
                else:
                    return 'Outros'
            
            df_temp['categoria'] = df_temp['autor_desnormalizado'].apply(categorizar_instituicao)
            categoria_counts = df_temp['categoria'].value_counts().reset_index()
            categoria_counts.columns = ['Categoria', 'Quantidade']
            categoria_counts['Percentual'] = (categoria_counts['Quantidade'] / len(df_temp) * 100).round(1)
            categoria_counts.to_excel(writer, sheet_name='Categorias', index=False)
        
        print(f"\n✅ Arquivo exportado com sucesso: {nome_arquivo}")
        print(f"   Local: {os.path.abspath(nome_arquivo)}")
        return True
        
    except Exception as e:
        print(f"\n❌ Erro ao exportar arquivo: {e}")
        return False

def main():
    """
    Função principal para processar o arquivo de mapas.
    """
    print("\n" + "="*70)
    print("🗺️  PROCESSADOR DE AUTORES CORPORATIVOS - MAPAS")
    print("="*70)
    
    # Nome do arquivo (assumindo que está no mesmo diretório)
    arquivo = 'AUTOR_CORPORATIVO_MAPAS.xlsx'
    
    print(f"\n📂 Processando arquivo: {arquivo}")
    
    # Processar a planilha
    df_desnormalizado = processar_autores_multivalorados(
        arquivo, 
        coluna_autor='autor_corporativo_mapas'
    )
    
    if df_desnormalizado is not None and not df_desnormalizado.empty:
        # Análise detalhada
        analisar_autores(df_desnormalizado)
        
        # Exportar resultados
        nome_saida = 'AUTORES_MAPAS_PROCESSADOS.xlsx'
        exportar_autores_resultados(df_desnormalizado, nome_saida)
        
        print("\n" + "="*70)
        print("✅ PROCESSAMENTO CONCLUÍDO COM SUCESSO!")
        print("="*70)
        print("\n📁 O arquivo gerado contém as seguintes abas:")
        print("   1. Dados Completos - Todos os registros processados")
        print("   2. Estatísticas - Métricas gerais do processamento")
        print("   3. Catálogo Autores - Lista única de todos os autores")
        print("   4. Prefixos - Distribuição de prefixos identificados")
        print("   5. Frequência Autores - Autores ordenados por ocorrência")
        print("   6. Categorias - Classificação por tipo de instituição")
        
        # Se houver listas multivaloradas, avisar
        if df_desnormalizado['tipo'].eq('múltiplo').any():
            print("   7. Listas Multivaloradas - Agrupamento de listas (quando houver)")
        
        # Mostrar informações sobre o arquivo original
        print(f"\n📊 RESUMO DO ARQUIVO ORIGINAL:")
        total_linhas = df_desnormalizado['id_original'].nunique()
        total_preenchidos = len(df_desnormalizado[df_desnormalizado['autor_desnormalizado'] != ''])
        print(f"   • Total de linhas no arquivo: {total_linhas}")
        print(f"   • Linhas com autores preenchidos: {total_preenchidos}")
        print(f"   • Linhas vazias/NULL: {len(df_desnormalizado[df_desnormalizado['tipo'] == 'vazio'])}")
        print(f"   • Percentual de preenchimento: {total_preenchidos/total_linhas*100:.1f}%")
        
    else:
        print("\n❌ Nenhum dado foi processado. Verifique o arquivo de entrada.")

# Executar o programa
if __name__ == "__main__":
    main()


🗺️  PROCESSADOR DE AUTORES CORPORATIVOS - MAPAS

📂 Processando arquivo: AUTOR_CORPORATIVO_MAPAS.xlsx
✅ Arquivo carregado: AUTOR_CORPORATIVO_MAPAS.xlsx
   Total de linhas: 4470

📊 RESUMO DO PROCESSAMENTO:
   • Linhas processadas: 4470
   • Registros vazios/NULL: 17
   • Registros multivalorados encontrados: 20
   • Total registros após desnormalização: 4490

📈 ANÁLISE DOS AUTORES CORPORATIVOS - MAPAS

📊 ESTATÍSTICAS GERAIS:
  • Linhas originais: 4470
  • Registros após desnormalização: 4490
  • Registros vazios/NULL: 17
  • Autores únicos (originais): 4433
  • Autores de listas múltiplas: 40
  • Autores distintos identificados: 62

🏷️  DISTRIBUIÇÃO DE PREFIXOS:
  • Coordenação da: 3643 (81.1%)
  • Sem prefixo identificado: 477 (10.6%)
  • Companhia: 185 (4.1%)
  • Instituto: 117 (2.6%)
  • Secretaria de: 22 (0.5%)
  • Universidade: 20 (0.4%)
  • Departamento: 14 (0.3%)
  • Superintendência: 4 (0.1%)
  • Empresa: 4 (0.1%)
  • Serviço: 3 (0.1%)
  • Ministério: 1 (0.0%)

🏆 TOP 20 AUTORES 

# CÓDIGO DE ATOMIZAÇÃO DE EXECUTORES DE MAPAS

In [6]:
import pandas as pd
import numpy as np
import os
import re

# Dicionário de normalização para executores de mapas
DICIONARIO_NORMALIZACAO = {
    # Siglas e abreviações -> Nome completo padrão
    'COMEC': 'Coordenação da Região Metropolitana de Curitiba - COMEC',
    'ITCF': 'Instituto de Terras, Cartografia e Florestas - ITCF',
    'ITC': 'Instituto de Terras e Cartografia - ITC',
    'FITC': 'Fundação Instituto de Terras e Cartografia - FITC',
    'IBDF': 'Instituto Brasileiro de Desenvolvimento Florestal - IBDF',
    'COPEL': 'Companhia Paranaense de Energia - COPEL',
    'SANEPAR': 'Companhia de Saneamento do Paraná - SANEPAR',
    'MINEROPAR': 'Minerais do Paraná S.A. - MINEROPAR',
    'UFPR': 'Universidade Federal do Paraná - UFPR',
    'IPARDES': 'Instituto Paranaense de Desenvolvimento Econômico e Social - IPARDES',
    'IPPUC': 'Instituto de Pesquisa e Planejamento Urbano de Curitiba - IPPUC',
    'IBGE': 'Instituto Brasileiro de Geografia e Estatística - IBGE',
    'DER': 'Departamento de Estradas de Rodagem - DER',
    'EMBRAPA': 'Empresa Brasileira de Pesquisa Agropecuária - EMBRAPA',
    'CPRM': 'Companhia de Pesquisa de Recursos Minerais - CPRM',
    'SUREHMA': 'Superintendência dos Recursos Hídricos e Meio Ambiente - SUREHMA',
    'IAP': 'Instituto Ambiental do Paraná - IAP',
    'URBS': 'Urbanização de Curitiba S.A. - URBS',
    'FUPEF': 'Fundação de Pesquisas Florestais do Paraná - FUPEF',
    'CELEPAR': 'Companhia de Informática do Paraná - CELEPAR',
    'SEMA': 'Secretaria de Estado do Meio Ambiente e Recursos Hídricos - SEMA',
    'SEPL': 'Secretaria de Planejamento e Coordenação Geral - SEPL',
    'GEIPOT': 'Empresa Brasileira de Planejamento de Transportes - GEIPOT',
    
    # Variações de nomes de empresas
    'Aerofoto Cruzeiro S.A': 'Aerofoto Cruzeiro S.A.',
    'Aerofoto Cruzeiro S.A.': 'Aerofoto Cruzeiro S.A.',
    'Geofotos': 'Geofotos Engenharia de Aerolevantamentos S.A.',
    'Geofotos Engenharia de Aerolevantamentos S.A': 'Geofotos Engenharia de Aerolevantamentos S.A.',
    'Geofotos Engenharia de Aerolevantamentos S.A.': 'Geofotos Engenharia de Aerolevantamentos S.A.',
    
    'AGRITEC': 'AGRITEC S.A. Agrimensura e Aerofotogrametria',
    'AGRITEC S.A': 'AGRITEC S.A. Agrimensura e Aerofotogrametria',
    'AGRITEC S.A.': 'AGRITEC S.A. Agrimensura e Aerofotogrametria',
    'AGRITEC S.A. Agrimensura e Aerofotogrametria': 'AGRITEC S.A. Agrimensura e Aerofotogrametria',
    
    'Aerodata': 'Aerodata S.A. Engenharia de Aerolevantamentos',
    'Aerodata S.A': 'Aerodata S.A. Engenharia de Aerolevantamentos',
    'Aerodata S.A.': 'Aerodata S.A. Engenharia de Aerolevantamentos',
    'Aerodata S.A Engenharia de Aerolevantamentos': 'Aerodata S.A. Engenharia de Aerolevantamentos',
    
    'Esteio': 'Esteio Engenharia e Aerolevantamentos S.A.',
    'Esteio Engenharia e Aerolevantamentos S.A': 'Esteio Engenharia e Aerolevantamentos S.A.',
    'Esteio Engenharia e Aerolevantamentos S.A.': 'Esteio Engenharia e Aerolevantamentos S.A.',
    
    'SAESP': 'SAESP LTDA',
    'SAESP LTDA': 'SAESP LTDA',
    
    'Aerosul': 'Aerosul S.A. Aerofotogrametria Sul do Brasil LTDA.',
    'Aerosul S.A': 'Aerosul S.A. Aerofotogrametria Sul do Brasil LTDA.',
    'Aerosul S.A.': 'Aerosul S.A. Aerofotogrametria Sul do Brasil LTDA.',
    
    'Engefoto': 'Engefoto Engenharia e Aerolevantamentos S.A.',
    'Aeroimagem': 'Aeroimagem Aerolevantamentos S.A.',
    'Conspar': 'Consórcio Conspar',
    'Consórcio Conspar': 'Consórcio Conspar',
    
    'Saporski': 'Saporski',
    'Polycópias': 'Polycópias',
    
    'PROENSI': 'Projeto e Engenharia de Sistemas LTDA - PROENSI',
    'GEOMAG': 'GEOMAG LTDA. Engenharia Civil',
    
    'SENAGRO': 'SENAGRO. Sensoriamento Remoto',
    
    # Variações de instituições públicas
    'Instituto de Terras e Cartografia': 'Instituto de Terras e Cartografia - ITC',
    'Instituto de Terras, Cartografia e Florestas': 'Instituto de Terras, Cartografia e Florestas - ITCF',
    'Fundação Instituto de Terras e Cartografia': 'Fundação Instituto de Terras e Cartografia - FITC',
    
    'UFPR - Departamento de Geografia': 'Universidade Federal do Paraná - UFPR - Departamento de Geografia',
    'UFPR. Instituto de Geologia': 'Universidade Federal do Paraná - UFPR - Instituto de Geologia',
    
    'Minerais do Paraná S.A': 'Minerais do Paraná S.A. - MINEROPAR',
    'Minerais do Paraná S.A.': 'Minerais do Paraná S.A. - MINEROPAR',
    'Minerais do Paraná S.A. MINEROPAR': 'Minerais do Paraná S.A. - MINEROPAR',
    
    'Companhia de Pesquisa de Recursos Minerais': 'Companhia de Pesquisa de Recursos Minerais - CPRM',
    'CPMR': 'Companhia de Pesquisa de Recursos Minerais - CPRM',  # Corrigindo erro de digitação
    
    'DE/PR': 'Departamento de Estradas de Rodagem - DER/PR',
    
    # Consórcios e parcerias
    'ITCF --- COPEL --- IBDF --- DE/PR --- FUPEF': 'Consórcio ITCF/COPEL/IBDF/DER/FUPEF',
    'Esteio --- Engefoto --- Aerosul --- Aeroimagem --- Aerodata': 'Consórcio Esteio/Engefoto/Aerosul/Aeroimagem/Aerodata',
}

# Lista de padrões para extração de siglas
PADROES_SIGLAS = [
    r'-\s*([A-Z]{2,}(?:\.[A-Z]{2,})?)',  # Padrão: - SIGLA
    r'\b([A-Z]{2,}(?:\.[A-Z]{2,})?)\b',   # Palavras com 2+ letras maiúsculas
    r'\(([A-Z]{2,}(?:\.[A-Z]{2,})?)\)',    # (SIGLA)
]

def identificar_separador(valor):
    """
    Identifica separadores em valores multivalorados.
    """
    if pd.isna(valor) or str(valor).strip() == '':
        return None
    
    valor_str = str(valor).strip()
    
    if valor_str.upper() == 'NULL':
        return None
    
    # Verificar diferentes tipos de separadores
    if " --- " in valor_str:
        return " --- "
    elif "---" in valor_str:
        return "---"
    elif " / " in valor_str:
        return " / "
    elif "/" in valor_str and len(valor_str.split('/')) > 1:
        return "/"
    
    return None

def extrair_sigla(texto):
    """
    Extrai siglas de um texto.
    """
    if pd.isna(texto):
        return None
    
    texto = str(texto)
    
    for padrao in PADROES_SIGLAS:
        matches = re.findall(padrao, texto)
        if matches:
            # Retorna a primeira sigla encontrada (mais relevante)
            return matches[0].strip()
    
    return None

def normalizar_executor(valor):
    """
    Normaliza o nome do executor usando o dicionário.
    """
    if pd.isna(valor) or str(valor).strip() == '':
        return valor
    
    valor_str = str(valor).strip()
    valor_original = valor_str
    
    # 1. Verificar correspondência exata
    if valor_str in DICIONARIO_NORMALIZACAO:
        return DICIONARIO_NORMALIZACAO[valor_str]
    
    # 2. Verificar correspondência parcial (começa com)
    for chave, valor_normalizado in DICIONARIO_NORMALIZACAO.items():
        if valor_str.startswith(chave) or chave.startswith(valor_str):
            return valor_normalizado
    
    # 3. Verificar se contém a sigla
    sigla = extrair_sigla(valor_str)
    if sigla and sigla in DICIONARIO_NORMALIZACAO:
        return DICIONARIO_NORMALIZACAO[sigla]
    
    # 4. Verificar palavras-chave
    palavras_chave = {
        'Aerofoto': 'Aerofoto Cruzeiro S.A.',
        'Cruzeiro': 'Aerofoto Cruzeiro S.A.',
        'Geofotos': 'Geofotos Engenharia de Aerolevantamentos S.A.',
        'AGRITEC': 'AGRITEC S.A. Agrimensura e Aerofotogrametria',
        'Aerodata': 'Aerodata S.A. Engenharia de Aerolevantamentos',
        'Esteio': 'Esteio Engenharia e Aerolevantamentos S.A.',
        'SAESP': 'SAESP LTDA',
        'Aerosul': 'Aerosul S.A. Aerofotogrametria Sul do Brasil LTDA.',
        'Engefoto': 'Engefoto Engenharia e Aerolevantamentos S.A.',
        'Aeroimagem': 'Aeroimagem Aerolevantamentos S.A.',
        'Conspar': 'Consórcio Conspar',
        'Saporski': 'Saporski',
        'Polycópias': 'Polycópias',
        'PROENSI': 'Projeto e Engenharia de Sistemas LTDA - PROENSI',
        'GEOMAG': 'GEOMAG LTDA. Engenharia Civil',
        'SENAGRO': 'SENAGRO. Sensoriamento Remoto',
        'ITCF': 'Instituto de Terras, Cartografia e Florestas - ITCF',
        'ITC': 'Instituto de Terras e Cartografia - ITC',
        'FITC': 'Fundação Instituto de Terras e Cartografia - FITC',
        'UFPR': 'Universidade Federal do Paraná - UFPR',
        'MINEROPAR': 'Minerais do Paraná S.A. - MINEROPAR',
        'COMEC': 'Coordenação da Região Metropolitana de Curitiba - COMEC',
        'COPEL': 'Companhia Paranaense de Energia - COPEL',
        'SANEPAR': 'Companhia de Saneamento do Paraná - SANEPAR',
        'IPARDES': 'Instituto Paranaense de Desenvolvimento Econômico e Social - IPARDES',
        'IPPUC': 'Instituto de Pesquisa e Planejamento Urbano de Curitiba - IPPUC',
        'IBGE': 'Instituto Brasileiro de Geografia e Estatística - IBGE',
        'DER': 'Departamento de Estradas de Rodagem - DER',
        'EMBRAPA': 'Empresa Brasileira de Pesquisa Agropecuária - EMBRAPA',
        'CPRM': 'Companhia de Pesquisa de Recursos Minerais - CPRM',
        'URBS': 'Urbanização de Curitiba S.A. - URBS',
    }
    
    for palavra, normalizado in palavras_chave.items():
        if palavra.lower() in valor_str.lower():
            return normalizado
    
    return valor_str

def processar_executores_multivalorados(arquivo_excel, coluna_executor='executor_mapas'):
    """
    Processa a planilha de executores de mapas, desnormalizando valores multivalorados.
    """
    if not os.path.exists(arquivo_excel):
        print(f"❌ Arquivo não encontrado: {arquivo_excel}")
        return None
    
    try:
        df = pd.read_excel(arquivo_excel)
        print(f"✅ Arquivo carregado: {arquivo_excel}")
        print(f"   Total de linhas: {len(df)}")
    except Exception as e:
        print(f"❌ Erro ao carregar arquivo: {e}")
        return None
    
    if coluna_executor not in df.columns:
        print(f"⚠️  A coluna '{coluna_executor}' não foi encontrada.")
        if len(df.columns) == 1:
            coluna_executor = df.columns[0]
            print(f"   Usando a coluna: '{coluna_executor}'")
        else:
            return None
    
    df['id_original'] = df.index + 1
    dados_desnormalizados = []
    
    total_multivalorados = 0
    total_vazios = 0
    total_normalizados = 0
    
    for idx, row in df.iterrows():
        id_original = row['id_original']
        valor = row[coluna_executor]
        linha_excel = idx + 2
        
        # Caso NULL ou vazio
        if pd.isna(valor) or str(valor).strip() == '' or str(valor).strip().upper() == 'NULL':
            total_vazios += 1
            dados_desnormalizados.append({
                'id_original': id_original,
                'linha_excel_original': linha_excel,
                'executor_original': '',
                'executor_normalizado': '',
                'sigla_extraida': None,
                'tipo': 'vazio',
                'separador': None,
                'qtd_executores': 0
            })
            continue
        
        valor_str = str(valor).strip()
        separador = identificar_separador(valor_str)
        
        # Caso não seja multivalorado
        if not separador:
            valor_normalizado = normalizar_executor(valor_str)
            if valor_normalizado != valor_str:
                total_normalizados += 1
            
            sigla = extrair_sigla(valor_normalizado)
            
            dados_desnormalizados.append({
                'id_original': id_original,
                'linha_excel_original': linha_excel,
                'executor_original': valor_str,
                'executor_normalizado': valor_normalizado,
                'sigla_extraida': sigla,
                'tipo': 'único',
                'separador': None,
                'qtd_executores': 1
            })
            continue
        
        # Caso multivalorado
        total_multivalorados += 1
        executores = [e.strip() for e in valor_str.split(separador) if e.strip()]
        
        for i, executor in enumerate(executores, 1):
            executor_normalizado = normalizar_executor(executor)
            if executor_normalizado != executor:
                total_normalizados += 1
            
            sigla = extrair_sigla(executor_normalizado)
            
            dados_desnormalizados.append({
                'id_original': id_original,
                'linha_excel_original': linha_excel,
                'executor_original': valor_str,
                'executor_normalizado': executor_normalizado,
                'sigla_extraida': sigla,
                'tipo': 'múltiplo',
                'separador': separador,
                'qtd_executores': len(executores),
                'posicao_na_lista': i
            })
    
    df_desnormalizado = pd.DataFrame(dados_desnormalizados)
    
    print(f"\n📊 RESUMO DO PROCESSAMENTO:")
    print(f"   • Linhas processadas: {len(df)}")
    print(f"   • Registros vazios/NULL: {total_vazios}")
    print(f"   • Registros multivalorados: {total_multivalorados}")
    print(f"   • Nomes normalizados: {total_normalizados}")
    print(f"   • Total registros após desnormalização: {len(df_desnormalizado)}")
    
    return df_desnormalizado

def analisar_executores(df_desnormalizado):
    """
    Analisa estatísticas dos executores processados.
    """
    print("\n" + "="*80)
    print("📈 ANÁLISE DOS EXECUTORES DE MAPAS")
    print("="*80)
    
    total_linhas_originais = df_desnormalizado['id_original'].nunique()
    total_registros = len(df_desnormalizado)
    
    vazios = len(df_desnormalizado[df_desnormalizado['tipo'] == 'vazio'])
    unicos = len(df_desnormalizado[df_desnormalizado['tipo'] == 'único'])
    multiplos = len(df_desnormalizado[df_desnormalizado['tipo'] == 'múltiplo'])
    
    executores_unicos_df = df_desnormalizado[df_desnormalizado['executor_normalizado'] != '']
    qtd_executores_unicos = executores_unicos_df['executor_normalizado'].nunique()
    
    print(f"\n📊 ESTATÍSTICAS GERAIS:")
    print(f"  • Linhas originais: {total_linhas_originais}")
    print(f"  • Registros após desnormalização: {total_registros}")
    print(f"  • Registros vazios/NULL: {vazios}")
    print(f"  • Executores únicos: {unicos}")
    print(f"  • Executores em listas múltiplas: {multiplos}")
    print(f"  • Executores distintos (catálogo): {qtd_executores_unicos}")
    
    # Top executores mais frequentes
    print(f"\n🏆 TOP 20 EXECUTORES MAIS FREQUENTES:")
    top_20 = executores_unicos_df['executor_normalizado'].value_counts().head(20)
    for i, (executor, count) in enumerate(top_20.items(), 1):
        executor_exib = executor if len(executor) <= 70 else executor[:67] + "..."
        print(f"  {i:2d}. {executor_exib:<70} ({count} ocorrências)")
    
    # Análise de siglas
    print(f"\n🔤 ANÁLISE DE SIGLAS:")
    siglas_df = executores_unicos_df[executores_unicos_df['sigla_extraida'].notna()]
    if not siglas_df.empty:
        print(f"  • Executores com sigla identificada: {len(siglas_df)}")
        top_siglas = siglas_df['sigla_extraida'].value_counts().head(10)
        for sigla, count in top_siglas.items():
            print(f"    - {sigla}: {count} ocorrências")
    
    # Agrupar por tipo de executor
    print(f"\n🏷️  CATEGORIAS DE EXECUTORES:")
    
    def categorizar_executor(executor):
        executor_lower = str(executor).lower()
        if any(emp in executor_lower for emp in ['aerofoto', 'geofotos', 'agritec', 'aerodata', 'esteio', 'saesp', 'aerosul', 'engefoto', 'aeroimagem', 'geomag', 'proensi']):
            return 'Empresa Privada - Aerolevantamento'
        elif any(inst in executor_lower for inst in ['itcf', 'itc', 'fitc', 'ibdf', 'comec', 'copel', 'sanepar', 'mineropar', 'der', 'surehma', 'iap', 'sema', 'sepl', 'urb s']):
            return 'Órgão Público Estadual'
        elif any(fed in executor_lower for fed in ['ibge', 'embrapa', 'cprm', 'geipot']):
            return 'Órgão Público Federal'
        elif 'ufpr' in executor_lower or 'universidade' in executor_lower:
            return 'Instituição de Ensino'
        elif 'consórcio' in executor_lower or '---' in executor:
            return 'Consórcio/Parceria'
        else:
            return 'Outros'
    
    df_temp = executores_unicos_df.copy()
    df_temp['categoria'] = df_temp['executor_normalizado'].apply(categorizar_executor)
    categorias = df_temp['categoria'].value_counts()
    
    for categoria, count in categorias.items():
        print(f"  • {categoria}: {count} ({count/len(df_temp)*100:.1f}%)")
    
    # Listas multivaloradas
    multivalorados = df_desnormalizado[df_desnormalizado['tipo'] == 'múltiplo']
    if not multivalorados.empty:
        print(f"\n🔗 LISTAS MULTIVALORADAS:")
        print(f"  • Total de listas: {multivalorados['id_original'].nunique()}")
        tamanhos = multivalorados.groupby('id_original')['qtd_executores'].first().value_counts().sort_index()
        for tamanho, count in tamanhos.items():
            print(f"    - Listas com {tamanho} executores: {count}")

def exportar_resultados(df_desnormalizado, nome_arquivo='EXECUTORES_MAPAS_PROCESSADOS.xlsx'):
    """
    Exporta os resultados para Excel.
    """
    try:
        with pd.ExcelWriter(nome_arquivo, engine='openpyxl') as writer:
            # Dados completos
            df_desnormalizado.to_excel(writer, sheet_name='Dados Completos', index=False)
            
            # Estatísticas
            percentual_preenchimento = (len(df_desnormalizado[df_desnormalizado['executor_normalizado'] != '']) / len(df_desnormalizado) * 100)
            
            stats = {
                'Métrica': [
                    'Linhas originais',
                    'Registros desnormalizados',
                    'Registros vazios/NULL',
                    'Executores únicos',
                    'Executores em listas múltiplas',
                    'Executores distintos (catálogo)',
                    'Listas multivaloradas',
                    'Percentual de preenchimento'
                ],
                'Valor': [
                    df_desnormalizado['id_original'].nunique(),
                    len(df_desnormalizado),
                    len(df_desnormalizado[df_desnormalizado['tipo'] == 'vazio']),
                    len(df_desnormalizado[df_desnormalizado['tipo'] == 'único']),
                    len(df_desnormalizado[df_desnormalizado['tipo'] == 'múltiplo']),
                    df_desnormalizado[df_desnormalizado['executor_normalizado'] != '']['executor_normalizado'].nunique(),
                    df_desnormalizado[df_desnormalizado['tipo'] == 'múltiplo']['id_original'].nunique(),
                    f"{percentual_preenchimento:.1f}%"
                ]
            }
            
            pd.DataFrame(stats).to_excel(writer, sheet_name='Estatísticas', index=False)
            
            # Catálogo de executores normalizados
            executores_unicos = df_desnormalizado[['executor_normalizado', 'sigla_extraida']].drop_duplicates()
            executores_unicos = executores_unicos[executores_unicos['executor_normalizado'] != '']
            executores_unicos = executores_unicos.sort_values('executor_normalizado')
            executores_unicos.to_excel(writer, sheet_name='Catálogo Executores', index=False)
            
            # Frequência
            frequencia = df_desnormalizado['executor_normalizado'].value_counts().reset_index()
            frequencia.columns = ['Executor', 'Frequência']
            frequencia = frequencia[frequencia['Executor'] != '']
            frequencia.to_excel(writer, sheet_name='Frequência', index=False)
            
            # Listas multivaloradas
            multivalorados = df_desnormalizado[df_desnormalizado['tipo'] == 'múltiplo']
            if not multivalorados.empty:
                listas = multivalorados.groupby('id_original').agg({
                    'linha_excel_original': 'first',
                    'executor_original': 'first',
                    'qtd_executores': 'first',
                    'executor_normalizado': lambda x: ' | '.join(x)
                }).reset_index()
                listas.to_excel(writer, sheet_name='Listas Multivaloradas', index=False)
        
        print(f"\n✅ Arquivo exportado: {nome_arquivo}")
        return True
    except Exception as e:
        print(f"\n❌ Erro ao exportar: {e}")
        return False

def main():
    print("\n" + "="*80)
    print("🗺️  PROCESSADOR INTELIGENTE DE EXECUTORES DE MAPAS")
    print("="*80)
    
    arquivo = 'EXECUTOR_MAPAS.xlsx'
    print(f"\n📂 Processando: {arquivo}")
    
    df = processar_executores_multivalorados(arquivo, coluna_executor='executor_mapas')
    
    if df is not None and not df.empty:
        analisar_executores(df)
        exportar_resultados(df, 'EXECUTORES_MAPAS_ATOMIZADO.xlsx')
        
        print("\n" + "="*80)
        print("✅ PROCESSAMENTO CONCLUÍDO!")
        print("="*80)
        print("\n📁 Arquivo gerado: EXECUTORES_MAPAS_ATOMIZADO.xlsx")
        print("\n🔍 O código identificou e normalizou:")
        print("   • Siglas como ITCF, COMEC, UFPR, etc.")
        print("   • Variações de nomes de empresas")
        print("   • Consórcios e parcerias")
        print("   • Relacionou abreviações com nomes completos")
    else:
        print("\n❌ Nenhum dado processado.")

if __name__ == "__main__":
    main()


🗺️  PROCESSADOR INTELIGENTE DE EXECUTORES DE MAPAS

📂 Processando: EXECUTOR_MAPAS.xlsx
✅ Arquivo carregado: EXECUTOR_MAPAS.xlsx
   Total de linhas: 4470

📊 RESUMO DO PROCESSAMENTO:
   • Linhas processadas: 4470
   • Registros vazios/NULL: 2016
   • Registros multivalorados: 1020
   • Nomes normalizados: 1317
   • Total registros após desnormalização: 5670

📈 ANÁLISE DOS EXECUTORES DE MAPAS

📊 ESTATÍSTICAS GERAIS:
  • Linhas originais: 4470
  • Registros após desnormalização: 5670
  • Registros vazios/NULL: 2016
  • Executores únicos: 1434
  • Executores em listas múltiplas: 2220
  • Executores distintos (catálogo): 38

🏆 TOP 20 EXECUTORES MAIS FREQUENTES:
   1. Geofotos Engenharia de Aerolevantamentos S.A.                          (765 ocorrências)
   2. Aerofoto Cruzeiro S.A.                                                 (667 ocorrências)
   3. Universidade Federal do Paraná - UFPR                                  (478 ocorrências)
   4. AGRITEC S.A. Agrimensura e Aerofotogrametria

# CÓDIGO PARA CORREÇÃO DE CARACTERES CORROMPIDOS EM ASSUNTO_MAPAS

In [5]:
# CORREÇÃO DE CARACTERES - ASSUNTO_LIVROS.xlsx
# Adaptado para corrigir a coluna "assunto_livros"

import pandas as pd
import numpy as np
import re
from pathlib import Path

# ========== 1. CONFIGURAÇÃO ==========
print("🔧 CONFIGURANDO AMBIENTE...")
input_file = "ASSUNTO_MAPAS.xlsx"
output_file = "ASSUNTO_MAPAS_SEMQUEBRAS.xlsx"
coluna_alvo = "assunto_livros"  # COLUNA ESPECÍFICA PARA CORRIGIR

# ========== 2. CARREGAR O ARQUIVO ==========
print(f"\n📂 CARREGANDO ARQUIVO: {input_file}")
try:
    # Tentar carregar com diferentes engines
    try:
        df = pd.read_excel(input_file)
    except:
        df = pd.read_excel(input_file, engine='openpyxl')
    
    print(f"✅ Arquivo carregado com sucesso!")
    print(f"📊 Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas")
    
    # Verificar se a coluna alvo existe
    if coluna_alvo not in df.columns:
        print(f"\n⚠️  ATENÇÃO: Coluna '{coluna_alvo}' não encontrada!")
        print(f"Colunas disponíveis: {list(df.columns)}")
        # Tentar encontrar similar
        for col in df.columns:
            if 'assunto' in col.lower() or 'mapas' in col.lower():
                coluna_alvo = col
                print(f"📌 Usando coluna similar: '{coluna_alvo}'")
                break
    else:
        print(f"✅ Coluna alvo encontrada: '{coluna_alvo}'")
    
    # Verificar estrutura
    print(f"\n🔍 AMOSTRA DA COLUNA '{coluna_alvo}' (5 primeiros valores):")
    for i in range(min(5, len(df))):
        valor = str(df.iloc[i][coluna_alvo]) if not pd.isna(df.iloc[i][coluna_alvo]) else "[VAZIO]"
        print(f"  Linha {i+1}: {valor[:150]}...")
        
except Exception as e:
    print(f"❌ Erro ao carregar: {e}")
    raise

# ========== 3. FUNÇÃO DE CORREÇÃO ESPECÍFICA PARA ASSUNTO_LIVROS ==========
def corrigir_caracteres_assunto_livros(texto):
    """
    Função otimizada para correção dos caracteres corrompidos na coluna assunto_livros
    Baseada nos padrões identificados no conteúdo
    """
    if pd.isna(texto):
        return ""
    
    texto = str(texto)
    
    # PRIMEIRA ETAPA: Correção via encoding (problema de dupla codificação)
    try:
        # O padrão observado é UTF-8 interpretado como Latin-1
        texto_bytes = texto.encode('latin-1', errors='ignore')
        texto = texto_bytes.decode('utf-8', errors='ignore')
    except:
        pass
    
    # SEGUNDA ETAPA: Substituições específicas baseadas no conteúdo
    substituicoes = {
        # Caracteres acentuados básicos (padrão mais comum)
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú', 'Ã±': 'ñ',
        'Ã£': 'ã', 'Ãµ': 'õ', 'Ã§': 'ç', 'Ãª': 'ê', 'Ã´': 'ô', 'Ã¨': 'è',
        'Ã ': 'à', 'Ã\xa0': 'à', 'Ã¢': 'â', 'Ã«': 'ë', 'Ã¯': 'ï', 'Ã¼': 'ü',
        
        # Maiúsculas
        'Ã': 'Á', 'Ã': 'É', 'Ã': 'Í', 'Ã': 'Ó', 'Ã': 'Ú', 'Ã': 'Ã',
        'Ã': 'Ç', 'Ã': 'Õ', 'Ã': 'Ä', 'Ã': 'Ö', 'Ã': 'Ü',
        
        # Padrões específicos do conteúdo de planejamento/urbanismo
        'Ã§Ã£o': 'ção', 'Ãµes': 'ões', 'Ãµ': 'õ', 'Ã£': 'ã',
        'planejamento regionalplanejamento de desenvolvimento': 'planejamento regional/planejamento de desenvolvimento',
        'planejamento regionalprojeto': 'planejamento regional/projeto',
        'planejamento regionaleconomia': 'planejamento regional/economia',
        
        # Termos frequentes no conteúdo
        'demografiapopulaÃ§Ã£opopulaÃ§Ã£o urbanapopulaÃ§Ã£o rural': 'demografia/população/população urbana/população rural',
        'populaÃ§Ã£opopulaÃ§Ã£o urbanapopulaÃ§Ã£o rural': 'população/população urbana/população rural',
        'saneamento rural': 'saneamento rural',
        'sinalizaÃ§Ã£o turÃ­sticaturismo': 'sinalização turística/turismo',
        'histÃ³ria': 'história',
        'agriculturaeconomiaeconomia regionalindÃºstriamigraÃ§Ã£opopulaÃ§Ã£osetor pÃºblico': 'agricultura/economia/economia regional/indústria/migração/população/setor público',
        
        # Termos de planejamento urbano e ambiental
        'desenvolvimento institucional': 'desenvolvimento institucional',
        'indÃºstria': 'indústria',
        'mineraÃ§Ã£o': 'mineração',
        'enchenteinundaÃ§Ã£o': 'enchente/inundação',
        'educaÃ§Ã£o infantilcreche': 'educação infantil/creche',
        'saÃºde pÃºblicapolÃ­tica social': 'saúde pública/política social',
        'educaÃ§Ã£oescola': 'educação/escola',
        'recurso hÃ­dricocadastro': 'recurso hídrico/cadastro',
        'economia regionalvalor adicionado': 'economia regional/valor adicionado',
        'zoneamento urbanouso do soloplanejamento regional': 'zoneamento urbano/uso do solo/planejamento regional',
        
        # Caracteres especiais
        'Â°': '°', 'Âª': 'ª', 'Âº': 'º',
        'â‚¬': '€', 'â€š': '‚', 'â€ž': '„',
        'â€¢': '•', 'â€“': '–', 'â€”': '—',
        'â„¢': '™', 'â€º': '›', 'Â´': '´',
        
        # Termos de meio ambiente
        'meio ambienteimpacto ambiental': 'meio ambiente/impacto ambiental',
        'meio ambientepoluiÃ§Ã£o da Ã¡guabacia hidrogrÃ¡fica': 'meio ambiente/poluição da água/bacia hidrográfica',
        'estatÃ­sticademografiaeconomiaeducaÃ§Ã£osaÃºde pÃºblica': 'estatística/demografia/economia/educação/saúde pública',
        'sÃ­tio arqueolÃ³gico': 'sítio arqueológico',
        'grutadolinamorfologiaconservaÃ§Ã£o': 'gruta/dolina/morfologia/conservação',
        
        # Termos de transporte
        'transporte rodoviÃ¡riotransporte coletivoÃ´nibus': 'transporte rodoviário/transporte coletivo/ônibus',
        'transporte rodoviÃ¡riotransporte coletivosinalizaÃ§Ã£o': 'transporte rodoviário/transporte coletivo/sinalização',
        
        # Substituições específicas para "ã"
        'populaÃ§Ã£o': 'população',
        'urbanizaÃ§Ã£o': 'urbanização',
        'planejamento': 'planejamento',
        'construÃ§Ã£o': 'construção',
        'educaÃ§Ã£o': 'educação',
        'instituiÃ§Ã£o': 'instituição',
        'organizaÃ§Ã£o': 'organização',
        'comunicaÃ§Ã£o': 'comunicação',
        'informaÃ§Ã£o': 'informação',
        'localizaÃ§Ã£o': 'localização',
        'implantação': 'implantação',
        'ampliaÃ§Ã£o': 'ampliação',
        'prevenÃ§Ã£o': 'prevenção',
        
        # Substituições específicas para "ç"
        'aÃ§Ã£o': 'ação',
        'espaÃ§o': 'espaço',
        'processo': 'processo',
        'serviÃ§o': 'serviço',
        'licenciamento': 'licenciamento',
        
        # Substituições específicas para "é"
        'saÃºde': 'saúde',
        'hidrelÃ©trica': 'hidrelétrica',
        'elÃ©trica': 'elétrica',
        'elÃ©trico': 'elétrico',
        'turÃ­stico': 'turístico',
        
        # Substituições específicas para "ô"
        'econÃ´mico': 'econômico',
        'econÃ´mica': 'econômica',
        'zoneamento econÃ´mico': 'zoneamento econômico',
        'ecolÃ³gico': 'ecológico',
        
        # Substituições específicas para "ú"
        'hÃ­drico': 'hídrico',
        'hÃ­drica': 'hídrica',
        'pÃºblico': 'público',
        'pÃºblica': 'pública',
        'municÃ­pio': 'município',
        
        # Cidades e locais
        'ParanÃ¡': 'Paraná',
        'SÃ£o': 'São',
        'JosÃ©': 'José',
        'JoÃ£o': 'João',
        'Paulo': 'Paulo',
        'Curitiba': 'Curitiba',
        
        # Padrões comuns de junção
        'regionalplano': 'regional/plano',
        'regionalprojeto': 'regional/projeto',
        'regionalesporte': 'regional/esporte',
        'regionaluso': 'regional/uso',
        'regionaleconomia': 'regional/economia',
        
        # Separar termos concatenados
        'hidrografiarecurso': 'hidrografia/recurso',
        'economiameio': 'economia/meio',
        'meioambiente': 'meio ambiente',
        'planejamentoregional': 'planejamento regional',
        'urbanizacao': 'urbanização',
        
        # Outros termos frequentes
        'sustentÃ¡vel': 'sustentável',
        'agricultura': 'agricultura',
        'pequena propriedade': 'pequena propriedade',
        'regularizaÃ§Ã£o': 'regularização',
        'fundiÃ¡ria': 'fundiária',
        'assentamento': 'assentamento',
        'reassentamento': 'reassentamento',
        'habitaÃ§Ã£o': 'habitação',
        'loteamento': 'loteamento',
        'infraestrutura': 'infraestrutura',
    }
    
    # Aplicar substituições
    for errado, correto in substituicoes.items():
        texto = texto.replace(errado, correto)
    
    # CORREÇÃO ADICIONAL: Padrões específicos do arquivo
    # Separar termos que estão concatenados (muito comum no arquivo)
    padroes_concat = [
        (r'([a-z])([A-Z])', r'\1/\2'),  # Separar camelCase
        (r'(regional)([a-z])', r'\1/\2'),  # Separar "regional" seguido de palavra
        (r'(planejamento)([a-z])', r'\1/\2'),  # Separar "planejamento" seguido de palavra
        (r'(meio)([a-z])', r'\1 \2'),  # Separar "meio" seguido de palavra
        (r'(uso)([a-z])', r'\1 \2'),  # Separar "uso" seguido de palavra
    ]
    
    for padrao, substituicao in padroes_concat:
        texto = re.sub(padrao, substituicao, texto)
    
    # TERCEIRA ETAPA: Limpeza e normalização
    # Remover caracteres inválidos
    texto = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', ' ', texto)
    texto = re.sub(r'�+', ' ', texto)
    
    # Normalizar espaços e separadores
    texto = re.sub(r'\s+', ' ', texto)
    texto = re.sub(r'/', '/', texto)  # Garantir separador uniforme
    texto = re.sub(r'\/+', '/', texto)  # Remover múltiplas barras
    
    # Ordenar termos quando separados por barras (opcional)
    if '/' in texto:
        termos = texto.split('/')
        termos = [t.strip() for t in termos if t.strip()]
        termos = sorted(set(termos))  # Remover duplicados e ordenar
        texto = '/'.join(termos)
    
    return texto.strip()

# ========== 4. APLICAR CORREÇÃO NA COLUNA ALVO ==========
print(f"\n🔄 APLICANDO CORREÇÃO NA COLUNA '{coluna_alvo}'...")

# Criar cópia do original para comparação
df_original = df.copy()

# Aplicar correção na coluna alvo
antes = df[coluna_alvo].copy()
df[coluna_alvo] = df[coluna_alvo].apply(corrigir_caracteres_assunto_livros)
depois = df[coluna_alvo].copy()

print(f"✅ Correção aplicada na coluna '{coluna_alvo}'")

# ========== 5. ANÁLISE DETALHADA DAS CORREÇÕES ==========
print("\n🔍 ANÁLISE DETALHADA DAS CORREÇÕES:")
print("=" * 100)

# Contar tipos de correções
correcoes_por_tipo = {
    'acentuacao': 0,
    'separacao_termos': 0,
    'termos_especificos': 0,
    'caracteres_invalidos': 0
}

# Analisar amostras
amostras_analise = min(10, len(df))
print(f"\n📋 COMPARAÇÃO DETALHADA (primeiras {amostras_analise} linhas):")
print("=" * 100)

for i in range(amostras_analise):
    original = str(antes.iloc[i]) if not pd.isna(antes.iloc[i]) else ""
    corrigido = str(depois.iloc[i]) if not pd.isna(depois.iloc[i]) else ""
    
    if original != corrigido:
        print(f"\n📄 LINHA {i+1}:")
        print(f"ORIGINAL:  {original[:200]}")
        print(f"CORRIGIDO: {corrigido[:200]}")
        
        # Contar tipos de mudanças
        if 'Ã' in original or 'Â' in original:
            correcoes_por_tipo['acentuacao'] += 1
        if len(corrigido.split('/')) > len(original.split('/')):
            correcoes_por_tipo['separacao_termos'] += 1
        if 'planejamento' in corrigido.lower() or 'regional' in corrigido.lower():
            correcoes_por_tipo['termos_especificos'] += 1
        if '�' in original:
            correcoes_por_tipo['caracteres_invalidos'] += 1
            
        print("-" * 80)

# Estatísticas
print(f"\n📊 ESTATÍSTICAS DE CORREÇÃO:")
for tipo, count in correcoes_por_tipo.items():
    if count > 0:
        print(f"  • {tipo.replace('_', ' ').title()}: {count} ocorrências")

# Contar valores únicos antes e depois
valores_unicos_antes = antes.dropna().unique()
valores_unicos_depois = depois.dropna().unique()
print(f"\n📈 VALORES ÚNICOS:")
print(f"  • Antes da correção: {len(valores_unicos_antes)}")
print(f"  • Depois da correção: {len(valores_unicos_depois)}")

# ========== 6. IDENTIFICAR E CATEGORIZAR TERMOS ==========
print("\n🔬 CATEGORIZAÇÃO DOS TERMAS MAIS FREQUENTES:")

# Concatenar todos os valores corrigidos
todos_termos = ' '.join(depois.dropna().astype(str).tolist())

# Separar por barra e contar frequência
termos_separados = []
for valor in depois.dropna():
    if '/' in str(valor):
        termos_separados.extend([t.strip() for t in str(valor).split('/')])
    else:
        termos_separados.append(str(valor).strip())

# Contar frequência
from collections import Counter
contador_termos = Counter(termos_separados)
termos_mais_comuns = contador_termos.most_common(20)

print(f"\n📋 20 TERMOS MAIS FREQUENTES:")
for termo, freq in termos_mais_comuns:
    print(f"  • {termo:40} → {freq:4d} ocorrências")

# Agrupar por categorias
categorias = {
    'planejamento': ['planejamento', 'regional', 'urbano', 'territorial'],
    'meio_ambiente': ['meio ambiente', 'impacto ambiental', 'proteção ambiental', 'recursos hídricos'],
    'infraestrutura': ['transporte', 'habitação', 'saneamento', 'infraestrutura'],
    'social': ['educação', 'saúde', 'população', 'demografia'],
    'economia': ['economia', 'indústria', 'agricultura', 'desenvolvimento'],
}

print(f"\n🏷️  DISTRIBUIÇÃO POR CATEGORIAS:")
for categoria, termos in categorias.items():
    contagem = sum(sum(1 for t in termos_separados if termo in t.lower()) for termo in termos)
    print(f"  • {categoria.replace('_', ' ').title():20} → {contagem:4d} menções")

# ========== 7. DETECTAR PROBLEMAS RESIDUAIS ==========
print("\n🔍 VERIFICANDO PROBLEMAS RESIDUAIS NA COLUNA CORRIGIDA...")

caracteres_problematicos = ['Ã', 'Â', 'â', '€', '�', '˜', '™']
problemas_encontrados = []

for idx, valor in enumerate(depois):
    if pd.isna(valor):
        continue
    texto = str(valor)
    for char in caracteres_problematicos:
        if char in texto:
            problemas_encontrados.append((idx, char, texto[:100]))
            break
    if len(problemas_encontrados) >= 5:
        break

if problemas_encontrados:
    print(f"⚠️  Foram encontrados {len(problemas_encontrados)} problemas residuais:")
    for idx, char, texto in problemas_encontrados[:3]:
        print(f"  Linha {idx+1}: Caractere '{char}' em '{texto}...'")
    
    # Aplicar correção final
    print("\n🔄 APLICANDO LIMPEZA FINAL...")
    def limpeza_final(texto):
        if pd.isna(texto):
            return texto
        texto = str(texto)
        for char in caracteres_problematicos:
            texto = texto.replace(char, '')
        return texto.strip()
    
    df[coluna_alvo] = df[coluna_alvo].apply(limpeza_final)
    print("✅ Limpeza final aplicada!")
else:
    print("🎉 Nenhum problema residual encontrado!")

# ========== 8. SALVAR ARQUIVO CORRIGIDO ==========
print(f"\n💾 SALVANDO ARQUIVO CORRIGIDO: {output_file}")

try:
    # Salvar mantendo outras colunas
    df.to_excel(output_file, index=False)
    print(f"✅ Arquivo salvo com sucesso!")
    
    # Verificar se salvou corretamente
    df_salvo = pd.read_excel(output_file)
    valores_corrigidos = df_salvo[coluna_alvo].dropna().unique()[:5]
    
    print(f"\n📋 AMOSTRA DOS DADOS SALVOS (coluna '{coluna_alvo}'):")
    for i, valor in enumerate(valores_corrigidos[:5]):
        print(f"  {i+1}. {str(valor)[:150]}...")
        
except Exception as e:
    print(f"❌ Erro ao salvar: {e}")
    # Tentar alternativa
    try:
        df.to_excel(output_file, index=False, engine='openpyxl')
        print("✅ Salvo com engine 'openpyxl'")
    except Exception as e2:
        print(f"❌ Erro alternativo: {e2}")
        # Salvar apenas a coluna corrigida como backup
        df_backup = pd.DataFrame({coluna_alvo: df[coluna_alvo]})
        csv_file = "ASSUNTO_MAPAS_SEMQUEBRAS.csv"
        df_backup.to_csv(csv_file, index=False, encoding='utf-8-sig')
        print(f"✅ Coluna salva como CSV: {csv_file}")

# ========== 9. RESUMO FINAL ==========
print("\n" + "="*80)
print("🎯 PROCESSO DE CORREÇÃO CONCLUÍDO!")
print("="*80)
print(f"📁 Arquivo original: {input_file}")
print(f"📁 Arquivo corrigido: {output_file}")
print(f"🎯 Coluna corrigida: '{coluna_alvo}'")
print(f"📊 Total de registros: {len(df)}")
print("="*80)

# ========== 10. EXPORTAR ANÁLISE DETALHADA ==========
print("\n📊 EXPORTANDO ANÁLISE DETALHADA...")

# Criar DataFrame com análise
df_analise = pd.DataFrame({
    'linha': range(1, len(df) + 1),
    f'{coluna_alvo}_original': antes,
    f'{coluna_alvo}_corrigido': df[coluna_alvo],
    'alterado': antes != df[coluna_alvo]
})

# Salvar análise
arquivo_analise = "ANALISE_CORRECAO_ASSUNTO_MAPAS.xlsx"
df_analise.to_excel(arquivo_analise, index=False)
print(f"✅ Análise detalhada salva em: {arquivo_analise}")

# Exportar termos únicos
df_termos_unicos = pd.DataFrame({
    'termo': list(contador_termos.keys()),
    'frequencia': list(contador_termos.values())
}).sort_values('frequencia', ascending=False)

arquivo_termos = "TERMOS_UNICOS_ASSUNTO_MAPAS.xlsx"
df_termos_unicos.to_excel(arquivo_termos, index=False)
print(f"✅ Lista de termos únicos salva em: {arquivo_termos}")

print("\n" + "="*80)
print("📈 RESUMO DA CORREÇÃO:")
print("="*80)
print(f"• Total de linhas processadas: {len(df)}")
print(f"• Linhas com alterações: {df_analise['alterado'].sum()} ({df_analise['alterado'].sum()/len(df)*100:.1f}%)")
print(f"• Termos únicos identificados: {len(df_termos_unicos)}")
print(f"• Principais categorias: Planejamento, Meio Ambiente, Infraestrutura")
print("="*80)

🔧 CONFIGURANDO AMBIENTE...

📂 CARREGANDO ARQUIVO: ASSUNTO_MAPAS.xlsx
✅ Arquivo carregado com sucesso!
📊 Dimensões: 4470 linhas × 1 colunas

⚠️  ATENÇÃO: Coluna 'assunto_livros' não encontrada!
Colunas disponíveis: ['assunto_mapas']
📌 Usando coluna similar: 'assunto_mapas'

🔍 AMOSTRA DA COLUNA 'assunto_mapas' (5 primeiros valores):
  Linha 1: planimetriaaltimetriaplanialtimetria ...
  Linha 2: planimetriaaltimetriaplanialtimetria ...
  Linha 3: planimetriaaltimetriaplanialtimetria ...
  Linha 4: planimetriaaltimetriaplanialtimetria ...
  Linha 5: planimetriaaltimetriaplanialtimetria ...

🔄 APLICANDO CORREÇÃO NA COLUNA 'assunto_mapas'...
✅ Correção aplicada na coluna 'assunto_mapas'

🔍 ANÁLISE DETALHADA DAS CORREÇÕES:

📋 COMPARAÇÃO DETALHADA (primeiras 10 linhas):

📄 LINHA 1:
ORIGINAL:  planimetriaaltimetriaplanialtimetria 
CORRIGIDO: planimetriaaltimetriaplanialtimetria
--------------------------------------------------------------------------------

📄 LINHA 2:
ORIGINAL:  planimetriaalt

# CÓDIGO ETL PARA INSERÇÃO DOS DADOS DA PLANILHA NO BANCO

In [6]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch
from datetime import datetime
import re
import logging

class MapasETL:
    def __init__(self, dataframes, db_params):
        """
        Inicializa o ETL para exportação de dados de mapas.
        
        Args:
            dataframes: Dicionário com os DataFrames carregados.
            db_params: Dicionário com parâmetros de conexão ao banco.
        """
        self.dataframes = dataframes
        self.df_principal = dataframes.get('TbMapas')
        self.df_area = dataframes.get('area_geografica_mapas_processada')
        self.df_aut_corp = dataframes.get('autores_corporativos_mapas_processados')
        self.df_exec = dataframes.get('executores_mapas_atomizados')
        
        # Mapa de tabelas de apoio
        self.local_map = {}        # nome_local -> id_local
        self.autor_map = {}         # (nome_autor, tipo_autor) -> id_autor
        self.executor_map = {}      # nome_executor -> id_executor
        self.area_map = {}           # codigo_area -> {'nome': ..., 'id': ...}
        
        # Configurar logging
        logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
        self.logger = logging.getLogger(__name__)
        
        self.conn = None
        self.cursor = None
        self.db_params = db_params
        
    def connect_db(self):
        """Conecta ao banco de dados PostgreSQL."""
        try:
            self.conn = psycopg2.connect(**self.db_params)
            self.cursor = self.conn.cursor()
            self.logger.info("Conexão com o banco de dados estabelecida.")
        except Exception as e:
            self.logger.error(f"Erro ao conectar: {e}")
            raise
    
    def close_db(self):
        """Fecha a conexão."""
        if self.cursor:
            self.cursor.close()
        if self.conn:
            self.conn.close()
        self.logger.info("Conexão fechada.")
    
    def _adjust_varchar_columns(self):
        """
        Verifica se os dados excedem o tamanho definido nas colunas VARCHAR da tabela MAPA
        e, se necessário, altera a tabela para aumentar o limite.
        """
        self.logger.info("Verificando tamanho das colunas VARCHAR...")
        # Lista de colunas que podem precisar de ajuste (nome no banco, nome no df, tamanho atual)
        colunas_para_verificar = [
            ('titulo_mapa', 'titulo_mapas', 500),
            ('conteudo_mapa', 'conteudo_mapas', 1000),
            ('nota_geral_mapa', 'nota_geral_mapas', 1000),
            ('aquisicao_mapa', 'aquisicao_mapas', 100),
            ('elaboracao_mapa', 'metodo_elaboracao_mapas', 300),
            ('assunto_mapa', 'assunto_mapas', 10000),
            ('fonte_mapa', 'fonte_mapas', 200),
            ('setor_mapa', 'setor_mapas', 7),
        ]
        for coluna_banco, coluna_df, tamanho_atual in colunas_para_verificar:
            if coluna_df in self.df_principal.columns:
                max_len = self.df_principal[coluna_df].dropna().astype(str).map(len).max()
                if pd.notna(max_len) and max_len > tamanho_atual:
                    novo_tamanho = max_len + 10  # margem de segurança
                    self.logger.info(f"Coluna {coluna_banco} precisa de aumento: atual {tamanho_atual}, necessário {max_len}. Alterando para {novo_tamanho}.")
                    try:
                        self.cursor.execute(f"ALTER TABLE mapa ALTER COLUMN {coluna_banco} TYPE VARCHAR({novo_tamanho});")
                        self.conn.commit()
                    except Exception as e:
                        self.logger.warning(f"Não foi possível alterar a coluna {coluna_banco}: {e}")
                        self.conn.rollback()
    
    def prepare_dataframes(self):
        """Prepara e limpa todos os DataFrames."""
        self.logger.info("Preparando DataFrames...")
        
        # 1. DataFrame principal
        if self.df_principal is not None:
            # Padronizar nomes das colunas (remover espaços, minúsculas)
            self.df_principal.columns = self.df_principal.columns.str.strip().str.lower()
            
            # Converter data_mapas para datetime
            if 'data_mapas' in self.df_principal.columns:
                self.df_principal['data_mapas'] = pd.to_datetime(
                    self.df_principal['data_mapas'],
                    errors='coerce',
                    dayfirst=True
                )
            
            # Substituir valores nulos/vazios por None
            self.df_principal = self.df_principal.replace(['NULL', 'null', '', 'nan', 'NaN'], None)
            
            # Limpar strings
            text_columns = self.df_principal.select_dtypes(include=['object']).columns
            for col in text_columns:
                self.df_principal[col] = self.df_principal[col].apply(
                    lambda x: str(x).strip() if pd.notna(x) else None
                )
            
            # Garantir que idmapas seja int
            if 'idmapas' in self.df_principal.columns:
                self.df_principal['idmapas'] = pd.to_numeric(self.df_principal['idmapas'], errors='coerce')
            
            self.logger.info(f"DataFrame principal: {len(self.df_principal)} registros.")
        
        # 2. Área geográfica processada
        if self.df_area is not None:
            self.df_area.columns = self.df_area.columns.str.strip().str.lower()
            # Converter linha_original para inteiro e criar id_mapa (linha_original - 1)
            if 'linha_original' in self.df_area.columns:
                self.df_area['linha_original'] = pd.to_numeric(self.df_area['linha_original'], errors='coerce')
                self.df_area['id_mapa'] = self.df_area['linha_original'] - 1
                self.df_area = self.df_area.dropna(subset=['id_mapa', 'codigo', 'municipio'])
                self.df_area['id_mapa'] = self.df_area['id_mapa'].astype(int)
                self.df_area['codigo'] = pd.to_numeric(self.df_area['codigo'], errors='coerce').astype('Int64')
            self.logger.info(f"DataFrame área: {len(self.df_area)} registros.")
        
        # 3. Autores corporativos
        if self.df_aut_corp is not None:
            self.df_aut_corp.columns = self.df_aut_corp.columns.str.strip().str.lower()
            if 'id_original' in self.df_aut_corp.columns:
                self.df_aut_corp['id_original'] = pd.to_numeric(self.df_aut_corp['id_original'], errors='coerce')
            if 'autor_desnormalizado' in self.df_aut_corp.columns:
                self.df_aut_corp['autor_desnormalizado'] = self.df_aut_corp['autor_desnormalizado'].apply(
                    lambda x: str(x).strip() if pd.notna(x) else None
                )
                self.df_aut_corp = self.df_aut_corp.dropna(subset=['autor_desnormalizado', 'id_original'])
            self.logger.info(f"DataFrame autores corporativos: {len(self.df_aut_corp)} registros.")
        
        # 4. Executores
        if self.df_exec is not None:
            self.df_exec.columns = self.df_exec.columns.str.strip().str.lower()
            if 'id_original' in self.df_exec.columns:
                self.df_exec['id_original'] = pd.to_numeric(self.df_exec['id_original'], errors='coerce')
            if 'executor_normalizado' in self.df_exec.columns:
                self.df_exec['executor_normalizado'] = self.df_exec['executor_normalizado'].apply(
                    lambda x: str(x).strip() if pd.notna(x) else None
                )
                self.df_exec = self.df_exec.dropna(subset=['executor_normalizado', 'id_original'])
            self.logger.info(f"DataFrame executores: {len(self.df_exec)} registros.")
        
        return True
    
    def extract_unique_values(self):
        """Extrai valores únicos para as tabelas de apoio."""
        self.logger.info("Extraindo valores únicos...")
        
        # 1. LOCAL
        if 'local_mapas' in self.df_principal.columns:
            locais = self.df_principal['local_mapas'].dropna().unique()
            for local in locais:
                if local and str(local).strip():
                    self.local_map[str(local).strip()] = None
            self.logger.info(f"Locais encontrados: {len(self.local_map)}")
        
        # 2. AUTORES PESSOAIS (tipo 'pessoal')
        if 'autor_pessoal_mapas' in self.df_principal.columns:
            autores_pess = self.df_principal['autor_pessoal_mapas'].dropna().unique()
            for autor in autores_pess:
                if autor:
                    chave = (autor, 'pessoal')
                    self.autor_map[chave] = None
            self.logger.info(f"Autores pessoais encontrados: {len(autores_pess)}")
        
        # 3. AUTORES CORPORATIVOS
        if self.df_aut_corp is not None and 'autor_desnormalizado' in self.df_aut_corp.columns:
            autores_corp = self.df_aut_corp['autor_desnormalizado'].dropna().unique()
            for autor in autores_corp:
                if autor:
                    chave = (autor, 'corporativo')
                    self.autor_map[chave] = None
            self.logger.info(f"Autores corporativos encontrados: {len(autores_corp)}")
        
        # 4. EXECUTORES
        if self.df_exec is not None and 'executor_normalizado' in self.df_exec.columns:
            executores = self.df_exec['executor_normalizado'].dropna().unique()
            for executor in executores:
                if executor:
                    self.executor_map[executor] = None
            self.logger.info(f"Executores encontrados: {len(self.executor_map)}")
        
        # 5. ÁREAS GEOGRÁFICAS (código como ID)
        if self.df_area is not None and 'codigo' in self.df_area.columns and 'municipio' in self.df_area.columns:
            areas_unicas = self.df_area[['codigo', 'municipio']].drop_duplicates()
            for _, row in areas_unicas.iterrows():
                codigo = row['codigo']
                municipio = row['municipio']
                if pd.notna(codigo) and pd.notna(municipio):
                    try:
                        codigo_int = int(codigo)
                        self.area_map[codigo_int] = {
                            'nome': str(municipio).strip(),
                            'codigo': codigo_int,
                            'id': None
                        }
                    except (ValueError, TypeError):
                        continue
            self.logger.info(f"Áreas geográficas encontradas: {len(self.area_map)}")
        
        # Resumo
        self.logger.info("Resumo da extração:")
        self.logger.info(f"  Locais: {len(self.local_map)}")
        self.logger.info(f"  Autores (total): {len(self.autor_map)}")
        self.logger.info(f"  Executores: {len(self.executor_map)}")
        self.logger.info(f"  Áreas geográficas: {len(self.area_map)}")
        
        return True
    
    def insert_support_tables(self):
        """Insere as tabelas de suporte (LOCAL, AUTOR, EXECUTOR, AREA_GEOGRAFICA)."""
        try:
            self.logger.info("Inserindo tabelas de suporte...")
            self.conn.autocommit = False
            
            # 1. LOCAL (SEM ON CONFLICT)
            inserted_locais = 0
            for nome_local in list(self.local_map.keys()):
                # Primeiro verificar se já existe
                self.cursor.execute(
                    "SELECT id_local FROM local WHERE nome_local = %s",
                    (nome_local,)
                )
                resultado = self.cursor.fetchone()
                
                if resultado:
                    # Já existe, usar o ID existente
                    self.local_map[nome_local] = resultado[0]
                else:
                    # Não existe, inserir
                    self.cursor.execute(
                        "INSERT INTO local (nome_local) VALUES (%s) RETURNING id_local",
                        (nome_local,)
                    )
                    self.local_map[nome_local] = self.cursor.fetchone()[0]
                    inserted_locais += 1
            
            self.logger.info(f"  Locais: {inserted_locais} novos de {len(self.local_map)} totais.")
            
            # 2. AUTOR (com tipo) - SEM ON CONFLICT
            inserted_autores = 0
            for (nome_autor, tipo_autor) in list(self.autor_map.keys()):
                # Verificar se já existe
                self.cursor.execute(
                    "SELECT id_autor FROM autor WHERE nome_autor = %s AND tipo_autor = %s",
                    (nome_autor, tipo_autor)
                )
                resultado = self.cursor.fetchone()
                
                if resultado:
                    # Já existe
                    self.autor_map[(nome_autor, tipo_autor)] = resultado[0]
                else:
                    # Inserir novo
                    self.cursor.execute(
                        "INSERT INTO autor (nome_autor, tipo_autor) VALUES (%s, %s) RETURNING id_autor",
                        (nome_autor, tipo_autor)
                    )
                    self.autor_map[(nome_autor, tipo_autor)] = self.cursor.fetchone()[0]
                    inserted_autores += 1
            
            self.logger.info(f"  Autores: {inserted_autores} novos de {len(self.autor_map)} totais.")
            
            # 3. EXECUTOR - SEM ON CONFLICT
            inserted_executores = 0
            for nome_executor in list(self.executor_map.keys()):
                # Verificar se já existe
                self.cursor.execute(
                    "SELECT id_executor FROM executor WHERE nome_executor = %s",
                    (nome_executor,)
                )
                resultado = self.cursor.fetchone()
                
                if resultado:
                    self.executor_map[nome_executor] = resultado[0]
                else:
                    self.cursor.execute(
                        "INSERT INTO executor (nome_executor) VALUES (%s) RETURNING id_executor",
                        (nome_executor,)
                    )
                    self.executor_map[nome_executor] = self.cursor.fetchone()[0]
                    inserted_executores += 1
            
            self.logger.info(f"  Executores: {inserted_executores} novos de {len(self.executor_map)} totais.")
            
            # 4. AREA_GEOGRAFICA (código como ID) - JÁ TEM PK, então pode usar ON CONFLICT
            inserted_areas = 0
            for codigo, info in self.area_map.items():
                nome_area = info['nome']
                try:
                    self.cursor.execute(
                        """INSERT INTO area_geografica (id_area_geografica, nome_area_geografica) 
                           VALUES (%s, %s) 
                           ON CONFLICT (id_area_geografica) DO UPDATE SET nome_area_geografica = EXCLUDED.nome_area_geografica
                           RETURNING id_area_geografica""",
                        (codigo, nome_area)
                    )
                    info['id'] = self.cursor.fetchone()[0]
                    inserted_areas += 1
                except Exception as e:
                    self.logger.warning(f"Erro ao inserir área {nome_area} ({codigo}): {e}")
                    self.cursor.execute(
                        "SELECT id_area_geografica FROM area_geografica WHERE id_area_geografica = %s",
                        (codigo,)
                    )
                    res = self.cursor.fetchone()
                    if res:
                        info['id'] = res[0]
            
            self.logger.info(f"  Áreas geográficas: {inserted_areas} inseridas/atualizadas de {len(self.area_map)} totais.")
            
            self.conn.commit()
            self.logger.info("Tabelas de suporte inseridas com sucesso!")
            return True
        
        except Exception as e:
            self.conn.rollback()
            self.logger.error(f"Erro ao inserir tabelas de suporte: {e}")
            import traceback
            traceback.print_exc()
            return False

    def insert_mapas_and_relationships(self):
        """
        Insere os mapas e seus relacionamentos.
        """
        try:
            self.logger.info("Inserindo mapas e relacionamentos...")
            self.conn.autocommit = False
            
            # ===== PARTE 1: INSERIR OS MAPAS =====
            self.logger.info("Inserindo mapas...")
            
            # Ajustar tamanho das colunas se necessário
            self._adjust_varchar_columns()
            
            # Verificar a estrutura da tabela mapa
            self.cursor.execute("""
                SELECT column_name 
                FROM information_schema.columns 
                WHERE table_name = 'mapa'
                ORDER BY ordinal_position;
            """)
            colunas = self.cursor.fetchall()
            self.logger.info("Colunas da tabela mapa:")
            for col in colunas:
                self.logger.info(f"  - {col[0]}")
            
            # NOME CORRETO DA COLUNA DE LOCAL
            nome_coluna_local = 'local_id'
            
            # Contadores
            mapas_count = 0
            mapas_com_titulo_padrao = 0
            
            for index, row in self.df_principal.iterrows():
                id_mapa = row.get('idmapas')
                if pd.isna(id_mapa):
                    self.logger.warning(f"Linha {index} com ID nulo, pulando...")
                    continue
                
                try:
                    id_mapa = int(id_mapa)
                except:
                    self.logger.warning(f"ID inválido na linha {index}: {id_mapa}")
                    continue
                
                # VERIFICAR E TRATAR TÍTULO NULO
                titulo = row.get('titulo_mapas')
                if pd.isna(titulo) or titulo is None or str(titulo).strip() == '':
                    titulo = "*** MAPA SEM TÍTULO ***"
                    mapas_com_titulo_padrao += 1
                
                # Obter ID do local (FK)
                local_id = None
                if pd.notna(row.get('local_mapas')):
                    local_nome = str(row['local_mapas']).strip()
                    local_id = self.local_map.get(local_nome)
                
                # Preparar valores para inserção
                valores = [
                    id_mapa,
                    row.get('n_chamada_mapas'),
                    titulo,
                    row.get('escala_mapas'),
                    row.get('articulacao_mapas'),
                    row.get('projecao_mapas'),
                    row.get('latitude_mapas'),
                    row.get('longitude_mapas'),
                    local_id,
                    row.get('data_mapas') if pd.notna(row.get('data_mapas')) else None,
                    row.get('colacao_mapas'),
                    row.get('conteudo_mapas'),
                    row.get('nota_geral_mapas'),
                    row.get('aquisicao_mapas'),
                    row.get('metodo_elaboracao_mapas'),
                    row.get('assunto_mapas'),
                    row.get('fonte_mapas'),
                    row.get('setor_mapas')
                ]
                
                # Construir SQL
                sql = f"""
                    INSERT INTO mapa (
                        id_mapa, n_chamada_mapa, titulo_mapa, escala_mapa, articulacao_mapa,
                        projecao_mapa, latitude_mapa, longitude_mapa, {nome_coluna_local}, data_mapa,
                        colacao_mapa, conteudo_mapa, nota_geral_mapa, aquisicao_mapa,
                        elaboracao_mapa, assunto_mapa, fonte_mapa, setor_mapa
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (id_mapa) DO UPDATE SET
                        n_chamada_mapa = EXCLUDED.n_chamada_mapa,
                        titulo_mapa = EXCLUDED.titulo_mapa,
                        escala_mapa = EXCLUDED.escala_mapa,
                        articulacao_mapa = EXCLUDED.articulacao_mapa,
                        projecao_mapa = EXCLUDED.projecao_mapa,
                        latitude_mapa = EXCLUDED.latitude_mapa,
                        longitude_mapa = EXCLUDED.longitude_mapa,
                        {nome_coluna_local} = EXCLUDED.{nome_coluna_local},
                        data_mapa = EXCLUDED.data_mapa,
                        colacao_mapa = EXCLUDED.colacao_mapa,
                        conteudo_mapa = EXCLUDED.conteudo_mapa,
                        nota_geral_mapa = EXCLUDED.nota_geral_mapa,
                        aquisicao_mapa = EXCLUDED.aquisicao_mapa,
                        elaboracao_mapa = EXCLUDED.elaboracao_mapa,
                        assunto_mapa = EXCLUDED.assunto_mapa,
                        fonte_mapa = EXCLUDED.fonte_mapa,
                        setor_mapa = EXCLUDED.setor_mapa
                """
                
                try:
                    self.cursor.execute(sql, valores)
                    mapas_count += 1
                    if mapas_count % 100 == 0:
                        self.logger.info(f"  {mapas_count} mapas inseridos/atualizados...")
                except Exception as e:
                    self.logger.error(f"Erro ao inserir mapa ID {id_mapa}: {e}")
                    self.logger.error(f"Valores: {valores}")
                    raise
            
            self.logger.info(f"  Total de mapas inseridos/atualizados: {mapas_count}")
            self.logger.info(f"  Mapas que receberam título padrão: {mapas_com_titulo_padrao}")
            
            # Commit parcial antes das relações
            self.conn.commit()
            self.logger.info("Mapas salvos. Inserindo relações...")
            
            # ===== PARTE 2: INSERIR AS RELAÇÕES (O CÓDIGO QUE VOCÊ MOSTROU) =====
            
            # ANTES de começar a inserir relações, criar um conjunto de mapas válidos
            ids_mapas_validos = set(self.df_principal['idmapas'].dropna().astype(int).unique())
            self.logger.info(f"📊 Mapas válidos para relações: {len(ids_mapas_validos)}")
            
            # ANTES de começar a inserir relações, criar um conjunto de mapas válidos
            ids_mapas_validos = set(self.df_principal['idmapas'].dropna().astype(int).unique())
            self.logger.info(f"📊 Mapas válidos para relações: {len(ids_mapas_validos)}")
            
            # 1. AUTORES PESSOAIS
            if 'autor_pessoal_mapas' in self.df_principal.columns:
                self.logger.info("Inserindo relações com autores pessoais...")
                rel_autor_count = 0
                rel_autor_ignorados = 0
                
                for _, row in self.df_principal.iterrows():
                    id_mapa = row.get('idmapas')
                    autor_nome = row.get('autor_pessoal_mapas')
                    
                    if pd.notna(id_mapa) and pd.notna(autor_nome):
                        id_mapa_int = int(id_mapa)
                        
                        # VERIFICAR SE O MAPA EXISTE
                        if id_mapa_int not in ids_mapas_validos:
                            rel_autor_ignorados += 1
                            continue
                        
                        autor_nome = str(autor_nome).strip()
                        autor_id = self.autor_map.get((autor_nome, 'pessoal'))
                        
                        if autor_id:
                            try:
                                self.cursor.execute(
                                    """
                                    INSERT INTO mapa_autor (mapa_id, autor_id) 
                                    VALUES (%s, %s)
                                    ON CONFLICT (mapa_id, autor_id) DO NOTHING
                                    """,
                                    (id_mapa_int, autor_id)
                                )
                                if self.cursor.rowcount > 0:
                                    rel_autor_count += 1
                            except Exception as e:
                                self.logger.error(f"Erro inesperado em autor pessoal (mapa {id_mapa}): {e}")
                                raise  # Se der erro inesperado, para tudo
                
                self.logger.info(f"  Autores pessoais inseridos: {rel_autor_count}")
                if rel_autor_ignorados > 0:
                    self.logger.warning(f"  Autores ignorados (mapa inexistente): {rel_autor_ignorados}")
            
            # 2. AUTORES CORPORATIVOS
            if self.df_aut_corp is not None and 'id_original' in self.df_aut_corp.columns:
                self.logger.info("Inserindo relações com autores corporativos...")
                rel_autor_corp_count = 0
                rel_autor_corp_ignorados = 0
                rel_autor_corp_duplicados = 0
                
                for _, row in self.df_aut_corp.iterrows():
                    id_mapa = row.get('id_original')
                    autor_nome = row.get('autor_desnormalizado')
                    
                    if pd.notna(id_mapa) and autor_nome:
                        id_mapa_int = int(id_mapa)
                        
                        # VERIFICAR SE O MAPA EXISTE
                        if id_mapa_int not in ids_mapas_validos:
                            rel_autor_corp_ignorados += 1
                            continue
                        
                        autor_nome = str(autor_nome).strip()
                        autor_id = self.autor_map.get((autor_nome, 'corporativo'))
                        
                        if autor_id:
                            try:
                                self.cursor.execute(
                                    """
                                    INSERT INTO mapa_autor (mapa_id, autor_id) 
                                    VALUES (%s, %s)
                                    ON CONFLICT (mapa_id, autor_id) DO NOTHING
                                    """,
                                    (id_mapa_int, autor_id)
                                )
                                if self.cursor.rowcount > 0:
                                    rel_autor_corp_count += 1
                                else:
                                    rel_autor_corp_duplicados += 1
                            except Exception as e:
                                self.logger.error(f"Erro inesperado em autor corporativo (mapa {id_mapa}): {e}")
                                raise
                
                self.logger.info(f"  Autores corporativos inseridos: {rel_autor_corp_count}")
                self.logger.info(f"  Autores corporativos duplicados: {rel_autor_corp_duplicados}")
                if rel_autor_corp_ignorados > 0:
                    self.logger.warning(f"  Autores corporativos ignorados (mapa inexistente): {rel_autor_corp_ignorados}")
            
            # 3. EXECUTORES
            if self.df_exec is not None and 'id_original' in self.df_exec.columns:
                self.logger.info("Inserindo relações com executores...")
                rel_executor_count = 0
                rel_executor_ignorados = 0
                rel_executor_duplicados = 0
                
                for _, row in self.df_exec.iterrows():
                    id_mapa = row.get('id_original')
                    executor_nome = row.get('executor_normalizado')
                    
                    if pd.notna(id_mapa) and executor_nome:
                        id_mapa_int = int(id_mapa)
                        
                        # VERIFICAR SE O MAPA EXISTE
                        if id_mapa_int not in ids_mapas_validos:
                            rel_executor_ignorados += 1
                            continue
                        
                        executor_nome = str(executor_nome).strip()
                        executor_id = self.executor_map.get(executor_nome)
                        
                        if executor_id:
                            try:
                                self.cursor.execute(
                                    """
                                    INSERT INTO mapa_executor (mapa_id, executor_id) 
                                    VALUES (%s, %s)
                                    ON CONFLICT (mapa_id, executor_id) DO NOTHING
                                    """,
                                    (id_mapa_int, executor_id)
                                )
                                if self.cursor.rowcount > 0:
                                    rel_executor_count += 1
                                else:
                                    rel_executor_duplicados += 1
                            except Exception as e:
                                self.logger.error(f"Erro inesperado em executor (mapa {id_mapa}): {e}")
                                raise
                
                self.logger.info(f"  Executores inseridos: {rel_executor_count}")
                self.logger.info(f"  Executores duplicados: {rel_executor_duplicados}")
                if rel_executor_ignorados > 0:
                    self.logger.warning(f"  Executores ignorados (mapa inexistente): {rel_executor_ignorados}")
            
            # 4. ÁREAS GEOGRÁFICAS (já temos essa parte)
            if self.df_area is not None and 'id_mapa' in self.df_area.columns and 'codigo' in self.df_area.columns:
                self.logger.info("Inserindo relações com áreas geográficas...")
                rel_area_count = 0
                rel_area_ignorados = 0
                rel_area_duplicados = 0
                
                for _, row in self.df_area.iterrows():
                    id_mapa = row.get('id_mapa')
                    codigo = row.get('codigo')
                    
                    if pd.notna(id_mapa) and pd.notna(codigo):
                        id_mapa_int = int(id_mapa)
                        
                        # VERIFICAR SE O MAPA EXISTE
                        if id_mapa_int not in ids_mapas_validos:
                            rel_area_ignorados += 1
                            continue
                        
                        try:
                            codigo_int = int(codigo)
                            area_info = self.area_map.get(codigo_int)
                            
                            if area_info and area_info.get('id'):
                                self.cursor.execute(
                                    """
                                    INSERT INTO mapa_area_geografica (mapa_id, area_geografica_id) 
                                    VALUES (%s, %s)
                                    ON CONFLICT (mapa_id, area_geografica_id) DO NOTHING
                                    """,
                                    (id_mapa_int, area_info['id'])
                                )
                                if self.cursor.rowcount > 0:
                                    rel_area_count += 1
                                else:
                                    rel_area_duplicados += 1
                                    
                        except (ValueError, TypeError):
                            continue
                
                self.logger.info(f"  Áreas geográficas inseridas: {rel_area_count}")
                self.logger.info(f"  Áreas geográficas duplicadas: {rel_area_duplicados}")
                if rel_area_ignorados > 0:
                    self.logger.warning(f"  Áreas geográficas ignoradas (mapa inexistente): {rel_area_ignorados}")
            
            self.conn.commit()
            self.logger.info("✅ Todas as relações inseridas com sucesso!")
    
            return True
            
        except Exception as e:
            self.conn.rollback()
            self.logger.error(f"Erro ao inserir mapas e relacionamentos: {e}")
            import traceback
            traceback.print_exc()
            return False
    
    def verify_data(self):
        """Verifica se os dados foram inseridos corretamente."""
        try:
            self.logger.info("Verificando dados inseridos...")
            
            queries = [
                ("Mapas", "SELECT COUNT(*) FROM mapa"),
                ("Locais", "SELECT COUNT(*) FROM local"),
                ("Autores", "SELECT COUNT(*) FROM autor"),
                ("Autores por tipo", "SELECT tipo_autor, COUNT(*) FROM autor GROUP BY tipo_autor"),
                ("Executores", "SELECT COUNT(*) FROM executor"),
                ("Áreas Geográficas", "SELECT COUNT(*) FROM area_geografica"),
                ("Relações Autor", "SELECT COUNT(*) FROM mapa_autor"),
                ("Relações Executor", "SELECT COUNT(*) FROM mapa_executor"),
                ("Relações Área", "SELECT COUNT(*) FROM mapa_area_geografica")
            ]
            
            for nome, query in queries:
                try:
                    self.cursor.execute(query)
                    if "GROUP BY" in query:
                        results = self.cursor.fetchall()
                        self.logger.info(f"  {nome}:")
                        for tipo, count in results:
                            self.logger.info(f"    {tipo}: {count}")
                    else:
                        count = self.cursor.fetchone()[0]
                        self.logger.info(f"  {nome}: {count}")
                except Exception as e:
                    self.logger.warning(f"  {nome}: Erro na query - {e}")
            
            return True
        except Exception as e:
            self.logger.error(f"Erro ao verificar dados: {e}")
            return False
    
    def run_etl(self):
        """Executa todo o processo ETL."""
        self.logger.info("=== INICIANDO PROCESSO ETL MAPAS ===")
        
        try:
            self.connect_db()
            if not self.prepare_dataframes():
                return False
            self.extract_unique_values()
            if not self.insert_support_tables():
                return False
            if not self.insert_mapas_and_relationships():
                return False
            self.verify_data()
            self.logger.info("=== PROCESSO ETL CONCLUÍDO COM SUCESSO! ===")
            return True
        except Exception as e:
            self.logger.error(f"Erro durante o processo ETL: {e}")
            import traceback
            traceback.print_exc()
            return False
        finally:
            self.close_db()


def carregar_todos_arquivos():
    """
    Carrega todos os arquivos Excel com busca case insensitive e diagnóstico.
    """
    print("=" * 60)
    print("📂 CARREGANDO ARQUIVOS")
    print("=" * 60)
    
    import os
    from pathlib import Path
    
    # Mostrar diretório atual
    print(f"📁 Diretório atual: {os.getcwd()}")
    
    # Listar todos os arquivos .xlsx no diretório
    arquivos_encontrados = []
    for arquivo in os.listdir('.'):
        if arquivo.lower().endswith('.xlsx'):
            arquivos_encontrados.append(arquivo)
    
    print(f"\n📋 Arquivos .xlsx encontrados ({len(arquivos_encontrados)}):")
    for arquivo in sorted(arquivos_encontrados):
        tamanho = os.path.getsize(arquivo)
        print(f"   - {arquivo} ({tamanho} bytes)")
    
    # Mapeamento de chaves para nomes de arquivo (case insensitive)
    arquivos_desejados = {
        'TbMapas': 'TbMapas.xlsx',
        'area_geografica_mapas_processada': 'AREA_GEOGRAFICA_MAPAS_PROCESSADA.xlsx',
        'autores_corporativos_mapas_processados': 'AUTORES_CORPORATIVOS_MAPAS_PROCESSADOS.xlsx',
        'executores_mapas_atomizados': 'EXECUTORES_MAPAS_ATOMIZADO.xlsx'
    }
    
    # Criar índice case insensitive
    indice_arquivos = {f.lower(): f for f in arquivos_encontrados}
    
    dataframes = {}
    
    print("\n📖 TENTANDO CARREGAR:")
    for chave, nome_esperado in arquivos_desejados.items():
        nome_lower = nome_esperado.lower()
        
        if nome_lower in indice_arquivos:
            nome_real = indice_arquivos[nome_lower]
            print(f"\n  ✅ Encontrado: {nome_real} (esperava: {nome_esperado})")
            
            try:
                df = pd.read_excel(nome_real)
                dataframes[chave] = df
                print(f"     📊 Linhas: {len(df)}")
                print(f"     📊 Colunas: {list(df.columns)}")
            except Exception as e:
                print(f"     ❌ Erro ao ler: {e}")
                dataframes[chave] = None
        else:
            print(f"\n  ❌ Não encontrado: {nome_esperado}")
            dataframes[chave] = None
    
    # Resumo final
    print("\n" + "=" * 60)
    print("📊 RESUMO DO CARREGAMENTO:")
    print("=" * 60)
    for chave, df in dataframes.items():
        if df is not None:
            print(f"  ✅ {chave}: {len(df)} linhas")
        else:
            print(f"  ❌ {chave}: não carregado")
    
    return dataframes


def executar_etl_completo():
    """Função principal que orquestra todo o processo."""
    print("=" * 60)
    print("🚀 INICIANDO ETL COMPLETO DE MAPAS")
    print("=" * 60)
    
    dataframes = carregar_todos_arquivos()
    
    # Parâmetros de conexão - ajuste conforme seu ambiente
    DB_PARAMS = {
        'host': 'localhost',
        'database': 'MAPAS',
        'user': 'xxxxxxxxx',
        'password': 'xxxxx',
        'port': 00000
    }
    
    etl = MapasETL(dataframes, DB_PARAMS)
    sucesso = etl.run_etl()
    
    if sucesso:
        print("\n✅ ETL COMPLETO FINALIZADO COM SUCESSO!")
    else:
        print("\n❌ ETL FINALIZADO COM ERROS!")


if __name__ == "__main__":
    executar_etl_completo()

🚀 INICIANDO ETL COMPLETO DE MAPAS
📂 Carregando arquivos...
  - Carregando TbMapas.xlsx...
    4469 linhas.
  - Carregando AREA_GEOGRAFICA_MAPAS_PROCESSADA.xlsx...


2026-03-16 15:36:04,214 - INFO - === INICIANDO PROCESSO ETL MAPAS ===
2026-03-16 15:36:04,240 - INFO - Conexão com o banco de dados estabelecida.
2026-03-16 15:36:04,241 - INFO - Preparando DataFrames...
2026-03-16 15:36:04,286 - INFO - DataFrame principal: 4469 registros.
C:\Users\felipewada\AppData\Local\Temp\ipykernel_6912\3127083123.py:126: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.df_area['id_mapa'] = self.df_area['id_mapa'].astype(int)
C:\Users\felipewada\AppData\Local\Temp\ipykernel_6912\3127083123.py:127: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/sta

    12858 linhas.
  - Carregando AUTORES_CORPORATIVOS_MAPAS_PROCESSADOS.xlsx...
    Arquivo AUTORES_CORPORATIVOS_MAPAS_PROCESSADOS.xlsx não encontrado. Pulando.
  - Carregando EXECUTORES_MAPAS_ATOMIZADOS.xlsx...
    Arquivo EXECUTORES_MAPAS_ATOMIZADOS.xlsx não encontrado. Pulando.


2026-03-16 15:36:04,430 - INFO -   300 mapas inseridos/atualizados...
2026-03-16 15:36:04,456 - INFO -   400 mapas inseridos/atualizados...
2026-03-16 15:36:04,482 - INFO -   500 mapas inseridos/atualizados...
2026-03-16 15:36:04,508 - INFO -   600 mapas inseridos/atualizados...
2026-03-16 15:36:04,521 - INFO -   700 mapas inseridos/atualizados...
2026-03-16 15:36:04,556 - INFO -   800 mapas inseridos/atualizados...
2026-03-16 15:36:04,578 - INFO -   900 mapas inseridos/atualizados...
2026-03-16 15:36:04,598 - INFO -   1000 mapas inseridos/atualizados...
2026-03-16 15:36:04,620 - INFO -   1100 mapas inseridos/atualizados...
2026-03-16 15:36:04,644 - INFO -   1200 mapas inseridos/atualizados...
2026-03-16 15:36:04,673 - INFO -   1300 mapas inseridos/atualizados...
2026-03-16 15:36:04,707 - INFO -   1400 mapas inseridos/atualizados...
2026-03-16 15:36:04,734 - INFO -   1500 mapas inseridos/atualizados...
2026-03-16 15:36:04,768 - INFO -   1600 mapas inseridos/atualizados...
2026-03-16 15


✅ ETL COMPLETO FINALIZADO COM SUCESSO!


# CÓDIGO PARA JUNÇÃO DE ASSUNTOS

In [8]:
import pandas as pd
import re

# Carregar o arquivo Excel
caminho_arquivo = 'ASSUNTO_MAPAS.xlsx'
df = pd.read_excel(caminho_arquivo)

# Nome da coluna que contém os assuntos
nome_coluna = 'assunto_mapas'

# Função para limpar cada assunto
def limpar_assunto(texto):
    if isinstance(texto, str):
        # 1. Substituir barras e espaços por um espaço único (para garantir a separação)
        #    e depois remover espaços extras no início/fim.
        texto_limpo = re.sub(r'[/\s]+', ' ', texto).strip()
        # 2. Remover todos os espaços restantes, juntando tudo.
        texto_limpo = texto_limpo.replace(' ', '')
        return texto_limpo
    else:
        # Se não for string (ex: valor nulo), retorna o próprio valor ou uma string vazia
        return '' if pd.isna(texto) else texto

# Aplicar a função à coluna desejada
df[nome_coluna] = df[nome_coluna].apply(limpar_assunto)

# Salvar o resultado em um novo arquivo Excel (opcional)
nome_arquivo_saida = 'ASSUNTO_MAPAS_UNIFICADO.xlsx'
df.to_excel(nome_arquivo_saida, index=False)

print("Arquivo processado e salvo como:", nome_arquivo_saida)

# Exibir as primeiras linhas para conferência
print("\nPrimeiras linhas do arquivo processado:")
print(df.head(20).to_string())

Arquivo processado e salvo como: ASSUNTO_MAPAS_UNIFICADO.xlsx

Primeiras linhas do arquivo processado:
                           assunto_mapas
0   planimetriaaltimetriaplanialtimetria
1   planimetriaaltimetriaplanialtimetria
2   planimetriaaltimetriaplanialtimetria
3   planimetriaaltimetriaplanialtimetria
4   planimetriaaltimetriaplanialtimetria
5   planimetriaaltimetriaplanialtimetria
6   planimetriaaltimetriaplanialtimetria
7   planimetriaaltimetriaplanialtimetria
8   planimetriaaltimetriaplanialtimetria
9                        planialtimetria
10  planimetriaaltimetriaplanialtimetria
11  planimetriaaltimetriaplanialtimetria
12  planimetriaaltimetriaplanialtimetria
13  planimetriaaltimetriaplanialtimetria
14  planimetriaaltimetriaplanialtimetria
15  planimetriaaltimetriaplanialtimetria
16  planimetriaaltimetriaplanialtimetria
17  planimetriaaltimetriaplanialtimetria
18  planimetriaaltimetriaplanialtimetria
19                       planialtimetria
